In [ ]:
rm(list = ls(all.names = TRUE))
# Clear the workspace and trigger garbage collection
rm(list = ls())  # Remove all objects
gc()             # Trigger garbage collection

# Function to monitor memory and system health
monitor_health <- function() {
  cat("------ Memory Usage Report ------\n")
  print(gc())
  
  # Check swap usage (Linux/Unix only)
  if (.Platform$OS.type == "unix") {
    swap_usage <- system("free -h | grep Swap", intern = TRUE)
    cat("Swap Usage:\n", swap_usage, "\n")
    
    # Disk usage for /tmp directory (if relevant for temporary files)
    disk_usage <- system("df -h /tmp", intern = TRUE)
    cat("Disk Usage (/tmp):\n", disk_usage, "\n")
  } else {
    cat("Swap and disk usage monitoring not supported on this OS.\n")
  }
  
  # Display total memory available (useful for Linux)
  if (.Platform$OS.type == "unix") {
    total_mem <- system("free -h | grep Mem", intern = TRUE)
    cat("Total Memory:\n", total_mem, "\n")
  }
}

# Trigger an initial health check
cat("Initial Memory Usage:\n")
print(gc())
monitor_health()

# Set frequent manual garbage collection
cat("Triggering additional garbage collection.\n")
gc()


In [ ]:
#!/usr/bin/env Rscript
library(parallel)
library(dlm)
library(exdqlm)
library(mvtnorm)
library(jmuOutlier)
library(sn)
library(Matrix)
library(future)
library(future.apply)
library(numDeriv)
library(foreach)
library(doParallel)
library(dataRetrieval)
library(dplyr)
library(zoo)
library(tseries)
library(tidyverse)
library(patchwork)
library(rvest)
library(expint)
library(nimble)
library(nloptr)
library(expm)
library(numDeriv)
library(Rcpp)
library(RcppArmadillo)
library(RcppEigen)
library(ks)
library(MASS)
library(FNN)

n.samp <- 5000 
cut <- 1
m <- 1

harmonics = c(1, 2, 1/6.8333333)   

# Set environment variables for Boost, Eigen, LAPACK, and BLAS
Sys.setenv("PKG_CXXFLAGS"="-I/data/muscat_data/jaguir26/libs/eigen -I/data/muscat_data/jaguir26/libs/boost/include -DEIGEN_DONT_VECTORIZE")
Sys.setenv("PKG_LIBS"="-L/data/muscat_data/jaguir26/slibs/lib64 -L/data/muscat_data/jaguir26/libs/boost/lib -llapack -lblas -lboost_random -lboost_system -fopenmp")

# Update the LD_LIBRARY_PATH inside the R environment
Sys.setenv(LD_LIBRARY_PATH="/data/muscat_data/jaguir26/libs/lib64:/data/muscat_data/jaguir26/libs/boost/lib:/lib64")

Rcpp::sourceCpp("/data/muscat_data/jaguir26/project1_ucsc_phd/sampling_exal.cpp")
Rcpp::sourceCpp("/data/muscat_data/jaguir26/project1_ucsc_phd/sampling_truncnorm.cpp")
Rcpp::sourceCpp('/data/muscat_data/jaguir26/project1_ucsc_phd/kalman_NDLM.cpp')


In [ ]:
SIMS <- TRUE
use_covariates <- TRUE
df_t       <- 0.9995
df_s       <- 0.995
df_s67     <- 0.9995
df.discrep <- 0.999999999 
df_trans   <- 0.999999999 
df_covs    <- 0.999999999
lambda     <- 0.9

In [ ]:

# Function to check if a matrix is positive definite
is.positive.definite <- function(x) {
  eigenvalues <- eigen(x)$values
  return(all(eigenvalues > 0))
}

# Function to compute inverse or square root of inverse using Cholesky Decomposition
compute_cholesky <- function(q, compute_sqrt_inverse = FALSE) {
  if (!is.positive.definite(q)) {
    stop("The matrix is not positive definite.")
  }
  
  # Compute Cholesky decomposition
  chol_decomp <- chol(as.matrix(q))
  
  # Convert to Matrix class to use with chol2inv
  U <- Matrix(chol_decomp, sparse = TRUE)
  
  # Compute inverse using Cholesky decomposition
  inv_q <- chol2inv(U)
  
  if (!compute_sqrt_inverse) {
    return(list(inverse = inv_q))
  } else {
    # Compute square root of the inverse
    # The square root of the inverse in this context is the inverse of the upper triangular matrix U
    sqrt_inv_q <- solve(U)
    
    # Check if the square root of the inverse times itself results in the inverse
    sqrt_inv_q_product <- sqrt_inv_q %*% t(sqrt_inv_q)
    is_correct <- all.equal(sqrt_inv_q_product, inv_q, tolerance = 1e-10)
    
    return(list(inverse = inv_q, sqrt_inverse = sqrt_inv_q, check = is_correct))
  }
}
#
log.g<-function(gam){	log(2)+stats::pnorm(-abs(gam),log=T)+0.5*gam^2 }
L.fn<-function(p0){ stats::uniroot(function(gam) exp(log.g(gam))-(1-p0), c(-1000,0))$root }
U.fn<-function(p0){ stats::uniroot(function(gam) exp(log.g(gam))-p0, c(0,1000))$root }
p.fn<-function(p0,gam){ (p0-as.numeric(gam<0))/exp(log.g(gam))+as.numeric(gam<0)}
A.fn<-function(p0,gam){ temp.p = p.fn(p0,gam); return((1-2*temp.p)/(temp.p*(1-temp.p))) }
B.fn<-function(p0,gam){ temp.p = p.fn(p0,gam); return((2)/(temp.p*(1-temp.p))) }
C.fn<-function(p0,gam){ temp.p = p.fn(p0,gam); return((as.numeric(gam>0)-temp.p)^(-1)) }
#
CheckLossFn = function(p0,diff){diff*p0 - diff*as.numeric(diff<0)}
#
dlm_df = function(y, model, df, dim.df, s.priors = list(l0=1,S0=10), just.lik=FALSE){
  ### Gets the Time Series Length / Replicate number
  y = check_ts(y)
  TT = nrow(y)
  ### Gets the State Parameter dimension and Prior Distribution Parameters
  m0 = model$m0
  C0 = model$C0
  l0 = s.priors$l0
  S0 = s.priors$S0
  n = length(m0)
  ### Constructs F and G
  FF = model$FF
  GG = model$GG
  ### Variable Saving
  ### Posterior Distribution
  m = matrix(0,TT,n)
  C = array(0,c(TT,n,n))
  ### Predictive State Distribution
  a = matrix(0,TT,n)
  R = array(0,dim = c(TT,n,n))
  P = array(0,dim = c(TT,n,n))
  W = array(0,dim = c(TT,n,n))
  ### One-Step Ahead Forecast
  f = matrix(0,TT,1)
  Q = array(0,c(TT,1,1))
  inv.Q = array(0,c(TT,1,1))
  ### Regression Variables
  e = matrix(0,TT,1)
  A = array(0,c(TT,n,1))
  ### Sample Variance
  S = vector("numeric",TT)
  l = vector("numeric",TT)

  # Prior Dim Check
  m0 = matrix(m0,n,1)
  C0 = matrix(C0,n,n)
  ### Discount Factor Blocking
  df.mat = make_df_mat(df,dim.df,n)

  ### First Update
  ### One-step state forecast
  a[1,]  = GG[,,1] %*% m0
  P[1,,] = GG[,,1] %*% C0 %*% t(GG[,,1])
  W[1,,] = df.mat * P[1,,]
  R[1,,] = P[1,,] + W[1,,]
  ### One-step ahead forecast
  f[1,] = t(FF[,1]) %*% a[1,]
  Q[1,,] = as.matrix(1 + t(FF[,1]) %*% R[1,,] %*% FF[,1],1,1)
  inv.Q[1,,] = chol2inv(chol(Q[1,,]))
  ### Auxilary Variables
  e[1,]  = as.matrix(y[1,] - f[1,],1,1)
  A[1,,] = R[1,,] %*% FF[,1] %*% inv.Q[1,,]
  ### Variance update
  l[1] = l0 + 1
  S[1] = l0 * S0 / l[1] + (t(e[1,]) %*% inv.Q[1,,] %*% e[1,] / l[1])
  ### Posterior Distribution
  m[1,]  = a[1,] + as.matrix(A[1,,],n,1) %*% e[1,]
  C[1,,] = R[1,,] - as.matrix(A[1,,],n,1) %*% Q[1,,] %*% t(A[1,,])
  C[1,,] = (C[1,,] + t(C[1,,]))/2

  for(i in 2:TT){
    ### One-step state forecast
    a[i,]  = GG[,,i] %*% m[i-1,]
    P[i,,] = GG[,,i] %*% C[i-1,,] %*% t(GG[,,i])
    W[i,,] = df.mat * P[i,,]
    R[i,,] = P[i,,] + W[i,,]
    ### One-step ahead forecast
    f[i,] = t(FF[,i]) %*% a[i,]
    Q[i,,] = matrix(1 + t(FF[,i])%*% R[i,,]%*% FF[,i],1,1)
    inv.Q[i,,] = chol2inv(chol(Q[i,,]))
    ### Auxilary Variables
    e[i,]  = as.matrix(y[i,] - f[i,],1,1)
    A[i,,] = as.matrix(R[i,,] %*% FF[,i] %*% inv.Q[i,,],n,1)
    ### Variance update
    l[i] = l[i-1] + 1
    S[i] = l[i-1] * S[i-1] / l[i] + (t(e[i,]) %*% inv.Q[i,,] %*% e[i,] / l[i])
    ### Posterior Distribution
    m[i,]  = a[i,] + as.matrix(A[i,,],n,1) %*% e[i,]
    C[i,,] = R[i,,] - as.matrix(A[i,,],n,1) %*% Q[i,,] %*% t(as.matrix(A[i,,],n,1))
    C[i,,] = (C[i,,] + t(C[i,,]))/2
  }

  ### Adjust By Variance
  R[1,,] = S0 * R[1,,]
  Q[1,,]   = S0 * Q[1,,]
  C[1,,]   = S[1] * C[1,,]
  for(i in 2:TT){
    R[i,,] = S[i-1] * R[i,,]
    Q[i,,]   = S[i-1] * Q[i,,]
    C[i,,]   = S[i] * C[i,,]
  }

  # Calculate Log-Likelihood
  det.Q = log(abs(Q[1,,])) ; llik = lgamma((l0+1)/2)-lgamma(l0/2)-log(pi*l0)/2-det.Q/2-(l0+1)*log(1+t(e[1,])%*%inv.Q[1,,]%*%e[1,]/l0)/2
  for(t in 2:TT){
    det.Q = log(abs(Q[t,,]))
    llik = llik + lgamma((l[t-1]+1)/2)-lgamma(l[t-1]/2)-log(pi*l[t-1])/2-det.Q/2-(l[t-1]+1)*log(1+t(e[t,])%*%inv.Q[t,,]%*%e[t,]/l[t-1])/2
  }
  if(just.lik){
    return(list(llik = llik))
  }

  ## SMOOTHING
  ### Initializes recursive relations
  sa = matrix(0,TT,n)
  sR = array(0, dim = c(TT,n,n))
  ### Runs the recursive equations
  sa[TT,]  = m[TT,]
  sR[TT,,] = C[TT,,]
  for(k in 1:(TT-1)){
  ### Computes the Auxilary recursion Variable B
    B = C[TT-k,,] %*% t(GG[,,i]) %*% solve(R[TT-k+1,,])
    sa[TT-k,] = m[TT-k,] + B %*% (sa[TT-k+1,] - a[TT-k+1,])
    sR[TT-k,,] = C[TT-k,,] + B %*% (sR[TT-k+1,,] - R[TT-k+1,,]) %*% t(B)
  }
  ### Adjusts the variance update
  for(k in 1:TT){
    sR[TT-k,,] = S[TT] * sR[TT-k,,] / S[TT-k]
  }
  return(list(fm = m, fC = C, m = sa, C = sR,model = model, s = S, n = l))
}
#
make_df_mat = function(df,dim.df,n){
  if(sum(dim.df)!=n){ stop("sum of component dimensions given in dim.df does not match m0") }
  if(length(df)!=length(dim.df)){ stop("length of component discount factors does not match length of component dimensions") }
  dfs = rep(df,dim.df)
  n.dfs = length(dim.df)
  ind.dfs = c(0,sapply(1:length(dim.df),function(x){sum(dim.df[1:x])}),n)
  df.mat = matrix(0,n,n)
  for(j in 1:n.dfs){
    df.mat[(ind.dfs[j]+1):ind.dfs[(j+1)],(ind.dfs[j]+1):ind.dfs[(j+1)]] = (1-dfs[ind.dfs[(j+1)]])/dfs[ind.dfs[(j+1)]]
  }
  return(df.mat)
}
#
check_mod = function(model){
  if(dlm::is.dlm(model)){
    model = dlmMod(model)
  }
  if(!is.vector(model$m0)){
    if(ncol(model$m0) != 1){
      stop("m0 must be a vector or a matrix with 1 column")
      }
    }
  p = length(model$m0)
  model$C0 = as.matrix(model$C0)
  if(p != dim(model$C0)[1] & p != dim(model$C0)[2]){
    stop("C0 must be a square matrix matching the dimension of m0")
    }
  if(!all.equal(model$C0, t(model$C0)) | !all(eigen(model$C0)$values >= 0)){
    stop("C0 must be a covariance matrix")
  }
  if(!is.vector(model$FF)){
    if(nrow(model$FF) != p){
      stop("FF must be a vector of length matching the dimension of m0, or a matrix with number of rows matching the dimension of m0")
    }
  }else{
    if(length(model$FF) != p){
      stop("FF must be a vector of length matching the dimension of m0, or a matrix with number of rows matching the dimension of m0")
    }
  }
  if(is.null(dim(model$GG)[3])){
    model$GG = as.matrix(model$GG)
  }else{
    if(is.na(dim(model$GG)[3])){
      model$GG = as.matrix(model$GG)
    }else{
      model$GG = as.array(model$GG)
    }
  }
  if(p != dim(model$GG)[1] & p != dim(model$GG)[2]){
    stop("GG must be a square matrix matching the dimension of m0, or an array with first two dimensions matching the dimension of m0")
  }
  model$m0 = as.matrix(model$m0)
  model$FF = as.matrix(model$FF)
  return(model)
}
#
check_logics = function(gam.init,sig.init,fix.gamma,fix.sigma,dqlm.ind){
  retval <- NULL
  retval$gam.init = gam.init
  retval$fix.gamma = fix.gamma
  retval$dqlm.ind = dqlm.ind
  if(dqlm.ind){
    if(gam.init!=0 | !fix.gamma){
      retval$gam.init <- gam.init <- 0
      retval$fix.gamma <- fix.gamma <- TRUE
    }
  }else{
    if(gam.init==0 && fix.gamma==TRUE){
      retval$dqlm.ind = TRUE
    }
  }
  if(fix.gamma & is.na(gam.init)){ stop("when fix.gamma = TRUE, gam.init must be specified") }
  if(fix.sigma & is.na(sig.init)){ stop("when fix.sigma = TRUE, sig.init must be specified") }
  return(retval)
}
#
check_ts = function(dat){
  dat = as.matrix(dat)
  if(all(dim(dat)>1)){
    stop("data must be univariate time-series")
  }
  if(dim(dat)[1]<dim(dat)[2]){
    dat = t(dat)
  }
  return(invisible(dat))
}
#
is.exdqlm = function(m){ return(inherits(m,"exdqlm")) }

parameters_path <- "/data/muscat_data/jaguir26/projects/Project/Input/exAL/parameters/parameters.txt"

# Check if the file exists
if (!file.exists(parameters_path)) {
  stop("The parameters file does not exist at the specified path: ", parameters_path)
}

lines <- readLines(parameters_path)

# Check if the lines variable is empty or not as expected
if (length(lines) == 0) {
  stop("No content found in the parameters file: ", parameters_path)
}

# Process each line and assign variables
for (line in lines) {
  # Remove leading and trailing whitespaces
  line <- trimws(line)
  
  # Skip empty lines and comments
  if (nchar(line) == 0 || grepl("^#", line)) next
  
  # Evaluate and assign
  eval(parse(text = line))
}
#
dlm_df = function(y, model, df, dim.df, s.priors = list(l0=1,S0=10), just.lik=FALSE){
  
  ### Gets the Time Series Length / Replicate number
  TT = length(y)
  ### Gets the State Parameter dimension and Prior Distribution Parameters
  m0 = model$m0
  C0 = model$C0
  l0 = s.priors$l0
  S0 = s.priors$S0
  n = length(m0)
  ### Constructs F and G
  FF = model$FF
  GG = model$GG
  ### Variable Saving
  ### Posterior Distribution
  m = matrix(0,TT,n)
  C = array(0,c(TT,n,n))
  ### Predictive State Distribution
  a = matrix(0,TT,n)
  R = array(0,dim = c(TT,n,n))
  P = array(0,dim = c(TT,n,n))
  W = array(0,dim = c(TT,n,n))
  ### One-Step Ahead Forecast
  f = matrix(0,TT,1)
  Q = array(0,c(TT,1,1))
  inv.Q = array(0,c(TT,1,1))
  ### Regression Variables
  e = matrix(0,TT,1)
  A = array(0,c(TT,n,1))
  ### Sample Variance
  S = vector("numeric",TT)
  l = vector("numeric",TT)
  
  # Prior Dim Check
  m0 = matrix(m0,n,1)
  C0 = matrix(C0,n,n)
  ### Discount Factor Blocking
  df.mat = make_df_mat(df,dim.df,n)
  
  ### First Update
  ### One-step state forecast
  a[1,]  = GG[,,1] %*% m0
  P[1,,] = GG[,,1] %*% C0 %*% t(GG[,,1])
  W[1,,] = df.mat * P[1,,]
  R[1,,] = P[1,,] + W[1,,]
  ### One-step ahead forecast
  f[1,] = t(FF[,,1]) %*% a[1,]
  Q[1,,] = as.matrix(1 + t(FF[,,1]) %*% R[1,,] %*% FF[,,1],1,1)
  inv.Q[1,,] = chol2inv(chol(Q[1,,]))
  ### Auxilary Variables
  e[1,]  = as.matrix(y[1] - f[1,],1,1)
  A[1,,] = R[1,,] %*% FF[,,1] %*% inv.Q[1,,]
  ### Variance update
  l[1] = l0 + 1
  S[1] = l0 * S0 / l[1] + (t(e[1,]) %*% inv.Q[1,,] %*% e[1,] / l[1])
  ### Posterior Distribution
  m[1,]  = a[1,] + as.matrix(A[1,,],n,1) %*% e[1,]
  C[1,,] = R[1,,] - as.matrix(A[1,,],n,1) %*% Q[1,,] %*% t(A[1,,])
  C[1,,] = (C[1,,] + t(C[1,,]))/2
  
  for(i in 2:TT){
    ### One-step state forecast
    a[i,]  = GG[,,i] %*% m[i-1,]
    P[i,,] = GG[,,i] %*% C[i-1,,] %*% t(GG[,,i])
    W[i,,] = df.mat * P[i,,]
    R[i,,] = P[i,,] + W[i,,]
    ### One-step ahead forecast
    f[i,] = t(FF[,,i]) %*% a[i,]
    Q[i,,] = matrix(1 + t(FF[,,i])%*% R[i,,]%*% FF[,,i],1,1)
    inv.Q[i,,] = chol2inv(chol(Q[i,,]))
    ### Auxilary Variables
    e[i,]  = as.matrix(y[i] - f[i,],1,1)
    A[i,,] = as.matrix(R[i,,] %*% FF[,,i] %*% inv.Q[i,,],n,1)
    ### Variance update
    l[i] = l[i-1] + 1
    S[i] = l[i-1] * S[i-1] / l[i] + (t(e[i,]) %*% inv.Q[i,,] %*% e[i,] / l[i])
    ### Posterior Distribution
    m[i,]  = a[i,] + as.matrix(A[i,,],n,1) %*% e[i,]
    C[i,,] = R[i,,] - as.matrix(A[i,,],n,1) %*% Q[i,,] %*% t(as.matrix(A[i,,],n,1))
    C[i,,] = (C[i,,] + t(C[i,,]))/2
  }
  
  ### Adjust By Variance
  R[1,,] = S0 * R[1,,]
  Q[1,,]   = S0 * Q[1,,]
  C[1,,]   = S[1] * C[1,,]
  for(i in 2:TT){
    R[i,,] = S[i-1] * R[i,,]
    Q[i,,]   = S[i-1] * Q[i,,]
    C[i,,]   = S[i] * C[i,,]
  }
  
  # Calculate Log-Likelihood
  det.Q = log(abs(Q[1,,])) ; llik = lgamma((l0+1)/2)-lgamma(l0/2)-log(pi*l0)/2-det.Q/2-(l0+1)*log(1+t(e[1,])%*%inv.Q[1,,]%*%e[1,]/l0)/2
  for(t in 2:TT){
    det.Q = log(abs(Q[t,,]))
    llik = llik + lgamma((l[t-1]+1)/2)-lgamma(l[t-1]/2)-log(pi*l[t-1])/2-det.Q/2-(l[t-1]+1)*log(1+t(e[t,])%*%inv.Q[t,,]%*%e[t,]/l[t-1])/2
  }
  if(just.lik){
    return(list(llik = llik))
  }
  
  ## SMOOTHING
  ### Initializes recursive relations
  sa = matrix(0,TT,n)
  sR = array(0, dim = c(TT,n,n))
  ### Runs the recursive equations
  sa[TT,]  = m[TT,]
  sR[TT,,] = C[TT,,]
  for(k in 1:(TT-1)){
    ### Computes the Auxilary recursion Variable B
    B = C[TT-k,,] %*% t(GG[,,i]) %*% solve(R[TT-k+1,,])
    sa[TT-k,] = m[TT-k,] + B %*% (sa[TT-k+1,] - a[TT-k+1,])
    sR[TT-k,,] = C[TT-k,,] + B %*% (sR[TT-k+1,,] - R[TT-k+1,,]) %*% t(B)
  }
  ### Adjusts the variance update
  for(k in 1:TT){
    sR[TT-k,,] = S[TT] * sR[TT-k,,] / S[TT-k]
  }
  return(list(fm = m, fC = C, m = sa, C = sR,model = model, s = S, n = l))
}
#
make_df_mat = function(df,dim.df,n){
  if(sum(dim.df)!=n){ stop("sum of component dimensions given in dim.df does not match m0") }
  if(length(df)!=length(dim.df)){ stop("length of component discount factors does not match length of component dimensions") }
  dfs = rep(df,dim.df)
  n.dfs = length(dim.df)
  ind.dfs = c(0,sapply(1:length(dim.df),function(x){sum(dim.df[1:x])}),n)
  df.mat = matrix(0,n,n)
  for(j in 1:n.dfs){
    df.mat[(ind.dfs[j]+1):ind.dfs[(j+1)],(ind.dfs[j]+1):ind.dfs[(j+1)]] = (1-dfs[ind.dfs[(j+1)]])/dfs[ind.dfs[(j+1)]]
  }
  return(df.mat)
}
#
make_df_mat_k = function(df,dim.df,n,k){
  if(sum(dim.df)!=n){ stop("sum of component dimensions given in dim.df does not match m0") }
  if(length(df)!=length(dim.df)){ stop("length of component discount factors does not match length of component dimensions") }
  dfs = rep(df,dim.df)
  n.dfs = length(dim.df)
  ind.dfs = c(0,sapply(1:length(dim.df),function(x){sum(dim.df[1:x])}),n)
  df.mat = matrix(0,n,n)
  for(j in 1:n.dfs){
    df.mat[(ind.dfs[j]+1):ind.dfs[(j+1)],(ind.dfs[j]+1):ind.dfs[(j+1)]] = (1-dfs[ind.dfs[(j+1)]]^k)/dfs[ind.dfs[(j+1)]]^k
  }
  return(df.mat)
}
#
H_t_k_r <- function(GG, t, k, r){
  n <- dim(GG)[1]
  I <- diag(n)
  for (s in (t+k-r):(t+k)) {
    I <- GG[,,s] %*% I   
  }
  return(I)
}
#
# Function to estimate log density using KDE for univariate data
estimate_log_density_kde_univariate <- function(data, points) {
  kde_result <- kde(data)
  density_estimates <- predict(kde_result, x = points)
  log_density <- log(density_estimates + .Machine$double.eps*100)  # Add small value to avoid log(0)
  return(log_density)
}
#
# Function to estimate the expectation term for univariate data
estimate_expectation_term_univariate <- function(sample_from_p, sample_size) {
  # Generate a sample from the standard normal distribution
  sample_from_normal <- rnorm(sample_size)
  
  # Estimate log density of p at points sampled from the standard normal distribution
  log_density_estimates <- estimate_log_density_kde_univariate(sample_from_p, sample_from_normal)
  
  # Compute the Monte Carlo estimate of the expectation
  expectation_estimate <- mean(log_density_estimates)
  
  return(expectation_estimate)
}
#
# Function to estimate the KL divergence D_KL(N(0, 1) || p) for univariate data
estimate_kl_divergence_univariate_normal_to_p <- function(sample_from_p, sample_size) {
  # Estimate the expectation term
  expectation_term <- estimate_expectation_term_univariate(sample_from_p, sample_size)
  
  # Compute the KL divergence
  kl_divergence <- -0.5 * log(2 * pi) - 0.5 - expectation_term
  
  return(kl_divergence)
}
#
# Function to estimate KL divergence using k-NN with entropy package for multivariate data
estimate_kl_divergence_knn_entropy <- function(sample_from_p, sample_size, k = 10) {
  # Generate a sample from the multivariate standard normal distribution
  sample_from_normal <- matrix(rnorm(sample_size * ncol(sample_from_p)), ncol = ncol(sample_from_p))
  
  # Estimate KL divergence using entropy package's KL.div function
  kl_divergence <- KL.divergence(sample_from_p, sample_from_normal, k = k)
  
  # Return only the final estimate
  return(tail(kl_divergence, n = 1))
}


In [ ]:

#
# Unified function to estimate KL divergence based on the input sample
estimate_kl_divergence <- function(sample, sample_size = 1000) {
  # Check if the sample is univariate or multivariate
  if (is.vector(sample) || ncol(sample) == 1) {
    # Univariate case
    if (is.vector(sample)) {
      sample_from_p <- sample
    } else {
      sample_from_p <- sample[, 1]
    }
    
    # Estimate the KL divergence using the KDE-based method
    estimated_kl_divergence <- estimate_kl_divergence_univariate_normal_to_p(sample_from_p, sample_size)
    
  } else {
    # Multivariate case
    sample_from_p <- sample
    
    # Estimate the KL divergence using the k-NN based method with entropy package
    estimated_kl_divergence <- estimate_kl_divergence_knn_entropy(sample_from_p, sample_size, k = 10)
  }
  
  # Return the estimate
  return(estimated_kl_divergence)
}




In [ ]:

#
# Function to estimate differential entropy using KDE for univariate data
estimate_differential_entropy_kde_univariate <- function(data) {
  kde_result <- kde(data)
  estimates <- kde_result$estimate
  estimates[estimates <= 0] <- .Machine$double.eps*100 # Prevent log(0) issues
  log_estimates <- log(estimates)
  log_estimates[!is.finite(log_estimates)] <- 0 # Handle non-finite values
  entropy_estimate <- -sum(estimates * log_estimates) * diff(kde_result$eval.points)[1]
  return(entropy_estimate)
}
#
# Function to estimate differential entropy using KDE for multivariate data
estimate_differential_entropy_kde_multivariate <- function(data) {
  kde_result <- kde(data)
  estimates <- kde_result$estimate
  estimates[estimates <= 0] <- .Machine$double.eps*100 # Prevent log(0) issues
  log_estimates <- log(estimates)
  log_estimates[!is.finite(log_estimates)] <- 0 # Handle non-finite values
  entropy_estimate <- -sum(estimates * log_estimates) * prod(diff(kde_result$eval.points[[1]]))
  return(entropy_estimate)
}
#
# Function to estimate the KL divergence D_KL(p || N(0, I)) for univariate data
estimate_kl_divergence_univariate <- function(data) {
  # Estimate the differential entropy H(p)
  H_p <- estimate_differential_entropy_kde_univariate(data)
  
  # Compute the expected value of the squared norm of the vectors
  E_p_x2 <- mean(data^2)
  
  # Dimensionality is 1 for univariate data
  k <- 1
  
  # Compute the KL divergence
  kl_divergence <- -H_p + (k / 2) * log(2 * pi) + (1 / 2) * E_p_x2
  
  return(kl_divergence)
}
#
# Function to estimate the KL divergence D_KL(p || N(0, I)) for multivariate data
estimate_kl_divergence_multivariate <- function(data) {
  # Estimate the differential entropy H(p)
  H_p <- estimate_differential_entropy_kde_multivariate(data)
  
  # Dimensionality of the vectors
  k <- ncol(data)
  
  # Compute the expected value of the squared norm of the vectors
  E_p_xTx <- mean(rowSums(data^2))
  
  # Compute the KL divergence
  kl_divergence <- -H_p + (k / 2) * log(2 * pi) + (1 / 2) * E_p_xTx
  
  return(kl_divergence)
}
#
# Wrapper function for any sample
compute_kl_divergence <- function(sample) {
  # Ensure the input sample is a matrix
  sample <- as.matrix(sample)
  
  # Determine if the sample is univariate or multivariate
  if (ncol(sample) == 1) {
    kl_divergence <- estimate_kl_divergence_univariate(sample)
  } else {
    kl_divergence <- estimate_kl_divergence_multivariate(sample)
  }
  
  return(kl_divergence)
}
#
concatenate_matrix_columns <- function(matrix_input) {
  # Concatenate the columns of the matrix
  concatenated_vector <- c(matrix_input)
  return(concatenated_vector)
}
#
preallocate_matrix_list <- function(column_counts, num_rows) {
  # Initialize an empty list
  matrix_list <- vector("list", length(column_counts))
  
  # Loop through the column counts and create matrices
  for (i in seq_along(column_counts)) {
    num_cols <- column_counts[i]
    matrix_list[[i]] <- matrix(NA, nrow = num_rows, ncol = num_cols)
  }
  
  return(matrix_list)
}

In [ ]:
# Read and process ELI_lon data
ELI_lon <- read.csv("/data/muscat_data/jaguir26/projects/Project/Input/exAL/covariates/cov_1_ELI.csv")
merged_sst_data <- read.csv("/data/muscat_data/jaguir26/projects/Project/Input/exAL/covariates/cov_2_ONI.csv")
ELI_lon$time <- as.Date(ELI_lon$time)
adjustment_years <- 170
ELI_lon$time <- ELI_lon$time - years(adjustment_years)
#
CFSToCMS_CONVERSION_FACTOR = 0.0283168466
# Read and process USGS data
data_usgs_r <- readNWISdv(siteNumbers = site_code[1], parameterCd = "00060", statCd = "00003")
San_Lorenzo_Daily_USGS_R <- data_usgs_r %>%
  mutate(timestamp = as.Date(Date),
         data0 = log(X_00060_00003*CFSToCMS_CONVERSION_FACTOR + 1)) %>%
  filter(timestamp > as.Date("1979-01-01"))
San_Lorenzo_Daily_USGS_R$time <- San_Lorenzo_Daily_USGS_R$timestamp
#
# SOIL
csv_file_path <- "/data/muscat_data/jaguir26/project1_ucsc_phd/climate_indices/soil_moisture_daily_avg.csv"
soil_moisture_data <- read.csv(csv_file_path)
soil_moisture_data$time <- as.Date(soil_moisture_data$time)
colnames(soil_moisture_data) <- c('time','soil')
#
# Merge datasets based on 'time'
merged_data <- merge(ELI_lon, merged_sst_data, by = "time")
merged_data <- merge(merged_data, San_Lorenzo_Daily_USGS_R, by = "time")
merged_data <- merged_data[, c(1:6, 10)]
colnames(merged_data) <- c("time", "eli", "nino12", "nino3", "nino34", "nino4", "flow")
merged_data$eli_smooth <- rollmean(merged_data$eli, k = KK, align = "right", fill = NA)
merged_data$oni <- rollmean(merged_data$nino34, k = KK, align = "right", fill = NA)
merged_data$eli_smooth[1:(KK-1)] <- merged_data$eli[1:(KK-1)]#
merged_data$oni[1:(KK-1)] <- merged_data$nino34[1:(KK-1)]
merged_data$flow_log <- log(merged_data$flow + 1)
#
# Adding soil
merged_data <- merge(merged_data, soil_moisture_data, by = "time")

# Standardize specified columns
standardize <- function(x) {
  (x - mean(x, na.rm = TRUE)) / sd(x, na.rm = TRUE)
}
columns_to_standardize <- c("eli_smooth", "oni", "flow_log", "soil")
merged_data[columns_to_standardize] <- lapply(merged_data[columns_to_standardize], standardize)
#
# Read streamflow data and merge with covariates
data_path <- "/data/muscat_data/jaguir26/project1_ucsc_phd/retros_2022-12-25.csv"
streamflow_data <- read_csv(data_path, show_col_types = FALSE)
timestamps <- as.Date(streamflow_data$Date)
time_series_matrix <- as.matrix(streamflow_data[, c('USGS', 'GloFAS', 'NWS3.0')])
Y_usgs <- data.frame(time = timestamps, time_series_matrix)
#
plot_data <- merge(merged_data, Y_usgs, by = "time")
ppt_data <- read.csv("/data/muscat_data/jaguir26/project1_ucsc_phd/PPT.csv")
ppt_data$time <- as.Date(ppt_data$time)
plot_data <- merge(plot_data, ppt_data, by = "time")
########################################################
# INDECES
file_path <- "/data/muscat_data/jaguir26/project1_ucsc_phd/climate_indices/combined_indices_daily_standardized.csv"
combined_indices <- read_csv(file_path, show_col_types = FALSE)
combined_indices['time']  <- as.Date(combined_indices$Date )
plot_data <- merge(plot_data, combined_indices, by = "time")
#
plot_data <- plot_data[cut:nrow(plot_data),]

covs <- c('ppt', 
          'soil')

indices <- c('Solar Flux',
              'ONI',
              'WHWP',
              'GMT',
              'AMO',
              'TSA',
              'TNA',
              'SOI')

# components_file_path <- "/data/muscat_data/jaguir26/project1_ucsc_phd/standardized_principal_components.csv"
# principal_components_df <- read_csv(components_file_path)
# principal_components_df$Date <- as.Date(principal_components_df$Date)
# plot_data$time <- as.Date(plot_data$time)
# merged_data <- merge(plot_data, principal_components_df, by.x = "time", by.y = "Date", all.x = TRUE)

components_file_path <- "/data/muscat_data/jaguir26/project1_ucsc_phd/dynamic_components_with_timestamps.csv"
principal_components_df <- read_csv(components_file_path, show_col_types = FALSE)
principal_components_df$Date <- as.Date(principal_components_df$Date)
plot_data$time <- as.Date(plot_data$time)
merged_data <- merge(plot_data, principal_components_df, by.x = "time", by.y = "Date", all.x = TRUE)


covariates <- plot_data[, covs]
covariates <- apply(covariates, 2, standardize)
for(i in 1:dim(covariates)[2] ){
    covariates[,i] <- covariates[,i]-min(covariates[,i])+1
}
#LOG###########################################
covariates <- log(log(covariates+1))

X <- cbind(covariates, merged_data[,'Component_1']) 
# X <- cbind(covariates, merged_data[,'Static_PCA']) 
X <- apply(X, 2, standardize)

#LOG#######################################################
# Set up Y and X matrices
# Y <- t(as.matrix(plot_data[, c('USGS', 'GloFAS', 'NWS3.0')]))
Y <- t(as.matrix(plot_data[, c('USGS')]))
Y <- log(Y)
TT <- dim(Y)[2]
J <- dim(Y)[1] - 1
#
timestamps <- plot_data[, 'time']


In [ ]:
# Model setup without covariates
s_yy <- sd(Y, na.rm = TRUE)  
m_yy <- mean(Y, na.rm = TRUE) + s_yy*qnorm(0.5)
kk <- 0.1 * s_yy
trend.comp <- polytrendMod(1, m0 = m_yy, C0 = kk)
harm <- harmonics
seas.comp <- seasMod(p = 363.5854, h = harm, C0 = 0.08 * kk * diag(2 * length(harm)))
model <- combineMods(trend.comp, seas.comp)
p <- length(model$m0)
#
idx <- 1:TT
y <- Y[,idx]
TT_sub <- length(idx)
#
if (is.null(nrow(y))) {
  JJJ <- 1
  y <- array(y, c(JJJ, length(y)))
} else {
  JJJ <- nrow(Y)
  y <- array(y, c(JJJ, ncol(y)))
}
#

In [ ]:
n.samp <- 5000
gam.init <- array(rep(0, JJJ), c(JJJ, 1))
sig.init <- array(rep(1, JJJ), c(JJJ, 1))
PriorSigma <- array(NA_real_, c(JJJ, 2))
PriorGamma <- array(NA_real_, c(JJJ, 3))
verbose <- TRUE
#
m0 <- c(model$m0, rep(0, J))
C0 <- bdiag(model$C0, 0.1 * kk * diag(J))

##########################################
##########################################
#
df_discrep <- rep(df.discrep, J)
#
df = c(df_t, df_s, df_s67); 
dim.df = c(1, 2*length(harm)-2, 2); 
k <- 10
##########################################
##########################################

df.mat <- make_df_mat(df, dim.df, p)
df.mat.k <- make_df_mat_k(df, dim.df, p, k)
if (J <= 0) {
  ex.df.mat <- df.mat
  ex.df.mat.k <- df.mat.k
} else {
#   extra_df.mat <- make_df_mat(df_discrep, rep(1, J), J)
  extra_df.mat <- make_df_mat(c(df.discrep), c(J), J)
#   extra_df.mat.k <- make_df_mat_k(df_discrep, rep(1, J), J, k)
  extra_df.mat.k<- make_df_mat_k(c(df.discrep), c(J), J, k)
  ex.df.mat <- bdiag(df.mat, extra_df.mat)
  ex.df.mat.k <- bdiag(df.mat.k, extra_df.mat.k)
}
#
model_simp <- model
df_simp <- df
dim.df_simp <- dim.df
model_simp$GG <- array(model_simp$GG, c(p, p, TT))
model_simp$FF <- array(model_simp$FF, c(p, 1, TT))
model$m0 <- m0
model$C0 <- C0
if (use_covariates) {
  # Adding covariates
  px <- dim(X)[2]
  ppx <- px + 1

  F1 <- matrix(model$FF, p, J + 1)
  F2 <- cbind(rep(0, J), diag(J))
  Fx <- rbind(rep(1, J + 1), matrix(0, nrow = px, ncol = J + 1))
  FF <- array(rbind(F1, F2, Fx), c(p + J + ppx, 1 + J, TT))

  Gx <- as.matrix(bdiag(lambda, diag(px)))
  Gx <- array(rep(Gx, TT), dim = c(ppx, ppx, TT))
  Gx[1, 2:ppx, ] <- as.matrix(t(X))

  GG <- array(bdiag(model$GG, diag(J)), c(p + J, p + J, TT))
  model$GG <- GG
  GG_dim <- dim(model$GG)[1]
  new_dim <- GG_dim + ppx
  GGG <- array(0, dim = c(new_dim, new_dim, TT))
  GGG[1:GG_dim, 1:GG_dim, ] <- model$GG
  GGG[(GG_dim + 1):new_dim, (GG_dim + 1):new_dim, ] <- Gx

  model$FF <- FF
  model$GG <- GGG

  # df.covs <- rep(df_covs, ppx)
  # extra_df.mat <- make_df_mat(df.covs, rep(1, ppx), ppx)
  # extra_df.mat.k <- make_df_mat_k(df.covs, rep(1, ppx), ppx, k)

  extra_df.mat <- make_df_mat(c(df_trans,df_covs), c(1,px), ppx)
  extra_df.mat.k <- make_df_mat_k(c(df_trans,df_covs), c(1,px), ppx, k)

  ex.df.mat <- bdiag(ex.df.mat, extra_df.mat)
  ex.df.mat.k <- bdiag(ex.df.mat.k, extra_df.mat.k)

  model$m0 <- c(model$m0, rep(0, ppx))
  model$C0 <- bdiag(model$C0, 0.03 * kk * diag(ppx))

} else {
  # Without covariates
  GG <- array(bdiag(model$GG, diag(J)), c(p + J, p + J, TT))
  model$GG <- GG
  F1 <- matrix(model$FF, p, J + 1)
  F2 <- cbind(rep(0, J), diag(J))
  FF <- array(rbind(F1, F2), c(p + J, 1 + J, TT))
  model$FF <- FF
  ppx <- 0
}
#
FF <- model$FF
GG <- model$GG


In [ ]:
######################## Init VB
exps <- Y
vars <- exps^2
new.sigma2_M <- array(1, c(1+J))
new.theta.out = list(exps = exps, 
                      exps2 = (exps)^2,
                      vars = 2*(exps)^2,
                      elbo.part = 0)
########################
# Prior for variances
a_s <- 1e-6
b_s <- 1e-6
a_update <- rep(TT/2 + a_s, 1+J)
b_update <- 0.5*rowSums((Y-exps)^2+vars)+b_s
sigma2 <- b_update/(a_update-1) 
seq.sigma2 <-  sigma2
D <- diag(sigma2)
########################
m0 <-  model$m0
C0 <-  model$C0 

C0 <- as.matrix(C0)
m0 <- model$m0
ex.df.mat <- as.matrix(ex.df.mat)
ex.df.mat.k <- as.matrix(ex.df.mat.k)


In [ ]:
a_s <- 1e-6
b_s <- 1e-6

a_update <- rep(NA_real_, 1+J)
b_update <- rep(NA_real_, 1+J)

vars <- new.theta.out$exps2-new.theta.out$exps^2
for(j in 0:J){
    if(j==0){
        a_update[j+1] <- (TT)/2 + a_s
        b_update[j+1] <- 0.5*sum((Y[j+1,]-new.theta.out$exps[j+1,1:TT])^2)+0.5*(sum(vars[j,1:TT]))+b_s
    }else{
        a_update[j+1] <- (TT+ranges[j]*num_mem[J-j+1])/2 + a_s
        b <- 0.5*sum((Y[j+1,]-new.theta.out$exps[j+1,1:TT])^2)+0.5*(sum(vars[j,1:TT]))+b_s
        b <- b + 0.5*sum((ensembles[[j]]-matrix(rep(new.theta.out$exps[j+1,(TT+1):(TT+ranges[j])],num_mem[J-j+1]), ncol = num_mem[J-j+1]))^2)
        b_update[j+1] <- b + 0.5*num_mem[J-j+1]*sum(vars[j+1,(TT+1):(TT+ranges[j])])
    }
}

sigma2 <- b_update/(a_update-1) 
seq.sigma2 <-  sigma2

if(j==0){
    D <- array(sigma2, c(1,1))
}else {
   D <- diag(sigma2)
}

In [ ]:
# Prior values for nu0 and S0
nu0 <- rep(5, J + 1)
S0 <- rep(10, J + 1)

In [ ]:
########################
dM <- 1 #Fix to one?
Ones <- matrix(1, dim(model$GG)[1], dim(model$GG)[1])
crit_ELBO <- 0
ELBO <- 0
seq.elbo = ELBO
iter = 0
FLAG = TRUE
tol1 <- 1e-1
tol2 <- 1e-1
seq.ndlm_elbo <- 0


In [ ]:
# new.theta.out_aprox$exps[1,idx]-new.theta.out$exps[1,idx]

In [ ]:

# while (FLAG) {  

#     #################################################################################################### 
#     # if (crit < tol1) {
#     update.theta <- update_theta_cpp_ndlm(GG, m0, C0, D, FF, y, ex.df.mat, ex.df.mat.k, Ones, p, J, ppx, TT, k, dM)
   
#     FF_t <- aperm(FF, c(2, 1, 3))
#     multiply_matrices <- function(slice_index) {
#       FF_t[,,slice_index] %*% update.theta$sm[,slice_index]
#     }
#     result_list <- lapply(1:ncol(update.theta$sm), multiply_matrices)
#     result_array <- array(unlist(result_list), dim = c(J+1, 1, ncol(update.theta$sm)))
#     result_array <- aperm(result_array, c(1, 3, 2))[,,1]
#     exps <- result_array

#     compute_product_1 <- function(t) {
#       FF_t_slice <- FF_t[,,t]
#       sC_slice <- update.theta$sC[,,t]
#       FF_slice <- FF[,,t]
#       result_slice <- t(FF_slice)%*%sC_slice%*%(FF_slice )
#       return(result_slice)
#     }
#     result_list_1 <- lapply(1:dim(FF)[3], compute_product_1)
#     vars_1 <- simplify2array(result_list_1)

#    if(J>0){
#     vars <- (apply(vars_1, 3, function(x) diag(x)))
#     exps2 = exps^2 + vars
#     }else{
#     exps2 = exps^2 + vars_1
#     vars_1 <- array( vars_1, c(1,TT) )  
#     exps2 <- array( exps2, c(1,TT) )  
#     exps <- array( exps, c(1,TT) )    
#     }
    
#       new.theta.out  <- update.theta 
#       new.theta.out$exps  <- exps
#       new.theta.out$exps2  <- exps2
#       new.theta.out$vars  <- vars
#       exps_aprox  <- exps
#     #################################################################################################### 
#     elbo <- 0
#     elbo <- elbo + new.theta.out$elbo.part
#     #################################################################################################### 
#     a_update <- rep(TT/2 + a_s, 1+J)
#     b_update <- 0.5*rowSums((Y-new.theta.out$exps)^2+new.theta.out$vars)+b_s
#     sigma2 <- b_update/(a_update-1) 
#     if(j==0){
#         D <- array(sigma2, c(1,1))
#     }else {
#       D <- diag(sigma2)
#     }
#     seq.sigma2 <-  c(seq.sigma2,b_update/(a_update-1)) 
#     ####################################################################################################  
#     elbo <- elbo + sum(-(TT/2+a_update)*log(b_update)+lgamma(a_update)+TT/2*digamma(a_update))  
#     elbo <- elbo/TT/(J+1)
#     ####################################################################################################  
#     crit_ELBO <- abs(elbo-ELBO)
#     ELBO <- elbo


#   seq.ndlm_elbo <- c(seq.ndlm_elbo, ELBO)  
#   print(c(crit_ELBO, ELBO))
#   flush.console()
#   iter <- iter+1
# }
# if (verbose) {
#   cat(sprintf("VB converged: %s iterations, %s seconds", 
#               iter, round(run.time$toc - run.time$tic, 3)), "\n")
# }



In [ ]:
# idx <- (TT-500):TT
# plot.ts(exp(new.theta.out$exps[1,idx])-1)
# points(exp(Y[1,idx])-1, lwd = 1.50, col = 'red')
# lines(exp(new.theta.out$exps[1,idx])-1)

# plot.ts((Y[1,idx])-(new.theta.out$exps[1,idx]))

In [ ]:
# plot.ts((new.theta.out$sm[1,]))
# plot.ts((new.theta.out$sm[2,]))
# plot.ts((new.theta.out$sm[4,]))
# plot.ts((new.theta.out$sm[6,]))
# plot.ts((new.theta.out$sm[8,]))
# plot.ts((new.theta.out$sm[9,]))
# plot.ts((new.theta.out$sm[10,]))
# plot.ts((new.theta.out$sm[11,]))

In [ ]:
# nu0 <- rep(100, 1)  # Degrees of freedom (vague but > 2 for well-defined mean)
# S0 <- rep(1000, 1) # Scale (high values imply weak prior knowledge)

# update.theta_exact <- update_theta_cpp_ndlm_exactV(GG, m0, C0, FF, y, ex.df.mat, ex.df.mat.k, Ones, p, J, ppx, TT, k, dM, nu0, S0)

# FF_t <- aperm(FF, c(2, 1, 3))
# multiply_matrices <- function(slice_index) {
#     FF_t[,,slice_index] %*% update.theta_exact$sm[,slice_index]
# }
# result_list <- lapply(1:ncol(update.theta_exact$sm), multiply_matrices)
# result_array <- array(unlist(result_list), dim = c(J+1, 1, ncol(update.theta_exact$sm)))
# result_array <- aperm(result_array, c(1, 3, 2))[,,1]
# exps <- result_array

# compute_product_1 <- function(t) {
#     FF_t_slice <- FF_t[,,t]
#     sC_slice <- update.theta_exact$sC[,,t]
#     FF_slice <- FF[,,t]
#     result_slice <- t(FF_slice)%*%sC_slice%*%(FF_slice )
#     return(result_slice)
# }
# result_list_1 <- lapply(1:dim(FF)[3], compute_product_1)
# vars_1 <- simplify2array(result_list_1)

# if(J>0){
# vars <- (apply(vars_1, 3, function(x) diag(x)))
# exps2 = exps^2 + vars
# }else{
# exps2 = exps^2 + vars_1
# vars_1 <- array( vars_1, c(1,TT) )  
# exps2 <- array( exps2, c(1,TT) )  
# exps <- array( exps, c(1,TT) )    
# }

# new.theta.out  <- update.theta_exact 
# new.theta.out$exps  <- exps
# new.theta.out$exps2  <- exps2
# new.theta.out$vars  <- vars
# exps_exact  <- exps


In [ ]:
# ############################
# idx <- (TT-100):(TT)
# plot.ts(Y[1,idx])
# lines(exps_exact[1,idx], col = 'blue') 
# lines(exps_aprox[1,idx], col = 'red') 
# ############################

In [ ]:
# tictoc::tic("run time")
# ########################
# sig.samp <- t(matrix(rinvgamma(n.samp*length(sigma2) , shape = a_update, rate = 1/b_update),nrow=length(sigma2)))
# samp_theta_t = function(t) {
#   LL <- t(chol(as.matrix(new.theta.out$sC[, , t])))
#   return(new.theta.out$sm[, t] + LL %*% matrix(stats::rnorm(n.samp *  (p+J), 0, 1), p+J, n.samp))
# }
# # 
# #######################  
# samp.theta = array(NA, c(p+J, TT, n.samp))
# for (t in 1:TT) {
#   samp.theta[, t, ] = samp_theta_t(t)
# }
# save_variables <- function(var_names, filename, dir_path) {
#   file_path <- file.path(dir_path, filename)
#   save_cmd <- paste("save(", paste(var_names, collapse = ", "), ", file = file_path)")
#   eval(parse(text = save_cmd))
#   cat("Variables saved to:", file_path, "\n")
# }
# new.theta.out_M <- new.theta.out
# samp.sigma_M = sig.samp
# samp.theta_M = samp.theta
# ndlm.standard.forecast.errors = new.theta.out$standard_forecast_errors
# vars_to_save <- c("new.theta.out_M", 
#                   "samp.sigma_M", 
#                   "samp.theta_M",
#                   "seq.ndlm_elbo",
#                   "ndlm.standard.forecast.errors")
# save_variables(vars_to_save, "variables_M.RData", "/home/jaguir26/projects/notebooks")


# errors <- matrix(new.theta.out$standard_forecast_errors[1,], ncol = 1)
# s <- 0.5 * compute_kl_divergence(errors)
# s <- s + 0.5 *  estimate_kl_divergence(new.theta.out$standard_forecast_errors[1,])
# ######################
# ######################
# ######################
# print(c(s, elbo))
# flush.console()
# ######################
# ######################
# ######################




In [ ]:
# FF_t <- aperm(FF, c(2, 1, 3))
# multiply_matrices <- function(slice_index) {
#     FF_t[,,slice_index] %*% update.theta_exact$sm[,slice_index]
# }

In [ ]:
# idx <- (TT-500):(TT)
# plot.ts(Y[1,idx])
# fff <- exps
# qqq <- sqrt(vars_1)
# q1 <- fff+qqq*qnorm(0.975)
# q3 <- fff+qqq*qnorm(0.025)
# lines(q1[1,idx], col='red')
# lines(fff[1,idx], col='blue')
# lines(q3[1,idx], col='red')


1. DONE! EXACT SOLUTION
2. DONE! SOLVE THE SAMPLING
3. Simulate a DLM!

### Synth Simulation

In [ ]:
# List of required packages
required_packages <- c("Matrix", "ggplot2", "dplyr")

# Function to install and load packages
install_and_load <- function(packages) {
  for (pkg in packages) {
    if (!require(pkg, character.only = TRUE)) {
      install.packages(pkg, dependencies = TRUE)
      library(pkg, character.only = TRUE)
    } else {
      library(pkg, character.only = TRUE)
    }
  }
}

# Install and load the required packages
install_and_load(required_packages)

# Confirm the loaded libraries
print("All required packages are installed and loaded!")


In [ ]:
set_matrices <- function(trend_degree = NULL, 
                         harmonics = NULL, 
                         period = NULL, 
                         covariates = NULL, 
                         covariate_mode = "regressor",  # Options: "regressor", "transfer"
                         transfer_rate = 1,
                         T = 100) {
  # Trend component based on the specified degree
  construct_trend_matrices <- function(degree) {
    state_dim <- degree + 1
    G <- diag(1, state_dim)
    for (i in 1:(state_dim - 1)) {
      G[i, i + 1] <- 1
    }
    F <- c(1, rep(0, state_dim - 1))
    list(G = G, F = F)
  }
  
  if (!is.null(trend_degree)) {
    trend_matrices <- construct_trend_matrices(trend_degree)
    G_trend <- trend_matrices$G
    F_trend <- trend_matrices$F
  } else {
    G_trend <- NULL
    F_trend <- NULL
  }
  
  # Seasonal component based on harmonics
  construct_seasonal_matrices <- function(harmonics, period) {
    if (is.null(harmonics) || is.null(period)) {
      return(list(G = NULL, F = NULL))
    }
    total_dim <- sum(ifelse(period / harmonics == 2, 1, 2))
    G <- matrix(0, nrow = total_dim, ncol = total_dim)
    F <- numeric(total_dim)
    
    idx <- 1
    for (r in harmonics) {
      if (period / r == 2) {  # Nyquist harmonic
        G[idx, idx] <- -1
        F[idx] <- 1
        idx <- idx + 1
      } else {  # Standard harmonic
        freq <- 2 * pi * r / period
        G[idx:(idx + 1), idx:(idx + 1)] <- matrix(c(cos(freq), sin(freq), 
                                                    -sin(freq), cos(freq)), nrow = 2)
        F[idx] <- 1
        F[idx + 1] <- 0
        idx <- idx + 2
      }
    }
    return(list(G = G, F = F))
  }
  
  seasonal_matrices <- construct_seasonal_matrices(harmonics, period)
  G_seasonal <- seasonal_matrices$G
  F_seasonal <- seasonal_matrices$F
  
  # Covariates component
  if (!is.null(covariates)) {
    px <- ncol(covariates)
    G_covariates <- array(0, dim = c(px + (covariate_mode == "transfer"), 
                                     px + (covariate_mode == "transfer"), T))
    F_covariates <- array(0, dim = c(px + (covariate_mode == "transfer"), 1, T))
    
    for (t in 1:T) {
      if (covariate_mode == "regressor") {
        G_covariates[, , t] <- diag(px)
        F_covariates[, 1, t] <- covariates[t, ]
      } else if (covariate_mode == "transfer") {
        Gx <- diag(px + 1)
        Gx[1, 1] <- transfer_rate
        Gx[1, 2:(px + 1)] <- covariates[t, ]
        G_covariates[, , t] <- Gx
        F_covariates[1, 1, t] <- 1
      } else {
        stop("Invalid covariate_mode. Use 'regressor' or 'transfer'.")
      }
    }
  } else {
    G_covariates <- NULL
    F_covariates <- NULL
  }
  
  # Combine components into G and F
  max_dim <- sum(
    sapply(
      list(
        if (!is.null(G_trend)) ncol(G_trend) else 0,
        if (!is.null(G_seasonal)) ncol(G_seasonal) else 0,
        if (!is.null(G_covariates)) dim(G_covariates)[1] else 0
      ), 
      identity
    )
  )
  G <- array(0, dim = c(max_dim, max_dim, T))
  F <- array(0, dim = c(max_dim, 1, T))
  
  for (t in 1:T) {
    idx <- 1
    if (!is.null(G_trend)) {
      trend_dim <- ncol(G_trend)
      G[idx:(idx + trend_dim - 1), idx:(idx + trend_dim - 1), t] <- G_trend
      F[idx:(idx + trend_dim - 1), 1, t] <- F_trend
      idx <- idx + trend_dim
    }
    if (!is.null(G_seasonal)) {
      seasonal_dim <- ncol(G_seasonal)
      G[idx:(idx + seasonal_dim - 1), idx:(idx + seasonal_dim - 1), t] <- G_seasonal
      F[idx:(idx + seasonal_dim - 1), 1, t] <- F_seasonal
      idx <- idx + seasonal_dim
    }
    if (!is.null(G_covariates)) {
      cov_dim <- ncol(G_covariates[, , t])
      G[idx:(idx + cov_dim - 1), idx:(idx + cov_dim - 1), t] <- G_covariates[, , t]
      F[idx:(idx + cov_dim - 1), 1, t] <- F_covariates[, 1, t]
    }
  }
  
  return(list(G = G, F = F))
}

simulate_dlm <- function(G, F, T, initial_state = NULL, observation_sd = 1, evolution_sd = 1) {
  # Extract dimensions
  state_dim <- dim(G)[1]
  
  # Initialize state and observation matrices
  theta <- matrix(0, nrow = T, ncol = state_dim)
  y <- numeric(T)
  
  # Initialize the state
  if (is.null(initial_state)) {
    theta[1, ] <- rnorm(state_dim, mean = 0, sd = evolution_sd)
  } else {
    theta[1, ] <- initial_state
  }
  
  # Simulate observations and state evolution
  for (t in 1:T) {
    # Compute observation
    y[t] <- sum(F[, , t] * theta[t, ]) + rnorm(1, mean = 0, sd = observation_sd)
    
    # Update state if not the last timestamp
    if (t < T) {
      theta[t + 1, ] <- G[, , t] %*% theta[t, ] + rnorm(state_dim, mean = 0, sd = evolution_sd)
    }
  }
  
  return(list(y = y, theta = theta))
}


In [ ]:
# library(ggplot2)

# seed <- 666

# # Define parameters
# TT <- 4000
# B  <- 5000
# T  <- TT + B - 1
# trend_degree <- 1
# harmonics    <- c(1)
# period       <- 365
# transfer_rate  <- 0.999
# observation_sd <- 35000
# evolution_sd   <- 1.0e-70

# # Set seed for reproducibility
# set.seed(seed)

# # Generate covariates
# covariates <- exp(matrix(rnorm(T * 1, sd = 4), ncol = 1))  # 1 random covariate

# # Generate matrices
# dlm_matrices <- set_matrices(
#   trend_degree = trend_degree,
#   harmonics = harmonics, 
#   period = period,
#   covariates = covariates,
#   covariate_mode = "transfer",
#   transfer_rate = transfer_rate,
#   T = T
# )

# # Initial state
# initial_state <- c(-1000, 300, 100000, 0, 5, 2)

# # Simulate the model
# simulation_results <- simulate_dlm(
#   G = dlm_matrices$G,
#   F = dlm_matrices$F,
#   T = T,
#   initial_state = initial_state,
#   observation_sd = observation_sd,
#   evolution_sd = evolution_sd
# )

# # Extract results
# y <- simulation_results$y[B:T]
# theta <- simulation_results$theta[B:T, ]
# theta0 <- simulation_results$theta[B-1, ]

# # Save observation plot
# # png(filename = paste0("simulated_observations_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
# plot(y, type = "l", col = "blue", lwd = 1.5, xlab = "Time", ylab = "Observation", 
#       main = paste("Simulated Observations (Seed:", seed, ")"))
# # dev.off()

# # Save state components plot
# # png(filename = paste0("simulated_states_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
# matplot(theta, type = "l", lty = 1, lwd = 1.5, xlab = "Time", ylab = "State Components", 
#         main = paste("Simulated State Components (Seed:", seed, ")"))
# legend("topright", legend = paste0("State ", 1:ncol(theta)), col = 1:ncol(theta), lty = 1, cex = 0.8)
# # dev.off()

# cat("Simulation and plots completed for seed:", seed, "\n")


# p = 4
# ppx = 2
# J = 0

# nu0 <- rep(4, 1)  # Degrees of freedom (vague but > 2 for well-defined mean)
# S0 <- nu0*rep(observation_sd, 1) # Scale (high values imply weak prior knowledge)
# m0 <- theta0
# C0 <- diag(rep(12000/(p+ppx), ncol(theta)))  # Large variance for vague prior
# ones <- matrix(1, ncol=(ncol(theta)),nrow=(ncol(theta)))


# df = 1-1e-70
# dim.df = c(6); 
# k <- 2
# ex_df_mat <- make_df_mat(df, dim.df, p+ppx+J)
# ex_df_mat_k <- make_df_mat_k(df, dim.df, p+ppx+J, k)

# Rcpp::sourceCpp('/data/muscat_data/jaguir26/project1_ucsc_phd/kalman_NDLM.cpp')

# fit_results <- update_theta_cpp_ndlm_exactV(
#       GG = dlm_matrices$G[,,B:T,drop = FALSE],
#       m0 = m0,
#       C0 = C0,
#       FF = dlm_matrices$F[,,B:T,drop = FALSE],
#       y = matrix(y, nrow=1),  # Reshape observations into a column vector
#       ex_df_mat = ex_df_mat,
#       ex_df_mat_k = ex_df_mat_k,
#       Ones = ones,  # Identity matrix for compatibility
#       p = 4,                     # Number of trend components (adjust as necessary)
#       J = 0,       # Number of observed components
#       ppx = 1+1,                   # No explanatory variables
#       TT = TT,                    # Number of time points
#       k = k,                     # 1-step ahead forecast
#       dM = 0,                    # No additional parameters
#       nu0 = nu0,
#       S0 = S0
# )

# update.theta_exact <- fit_results

# FF_t <- aperm(dlm_matrices$F[,,B:T,drop = FALSE], c(2, 1, 3))
# multiply_matrices <- function(slice_index) {
#     FF_t[,,slice_index] %*% update.theta_exact$sm[,slice_index]
# }
# result_list <- lapply(1:ncol(update.theta_exact$sm), multiply_matrices)
# result_array <- array(unlist(result_list), dim = c(J+1, 1, ncol(update.theta_exact$sm)))
# result_array <- aperm(result_array, c(1, 3, 2))[,,1]
# exps <- result_array

# compute_product_1 <- function(t) {
#     FF_t_slice <- FF_t[,,t]
#     sC_slice <- update.theta_exact$sC[,,t]
#     result_slice <- ((FF_t_slice)%*%sC_slice)%*%(FF_t_slice )
#     return(result_slice)
# }
# result_list_1 <- lapply(1:ncol(update.theta_exact$sm), compute_product_1)
# vars_1 <- simplify2array(result_list_1)

# S_TT <- c(update.theta_exact$S_t)
# n_TT <- c(update.theta_exact$nu_t)

# plot.ts(y, lwd = 0.1)
# points(y, pch = 19, lwd = 0.1)
# abline(h = sum(FF_t[,,1]*theta0))
# lines(exps, col = 'darkorange', lty = 1, lwd = 1.5)
# lines(exps+sqrt(S_TT/n_TT)*qnorm(0.975), col = 'lightblue', lty = 2, lwd = 1.5)
# lines(exps+sqrt(S_TT/n_TT)*qnorm(0.025), col = 'lightblue', lty = 2, lwd = 1.5)

# # Function to plot state estimates with filled credible intervals
# plot_state <- function(index, theta, update.theta_exact) {
#   # Extract filtered state estimates and confidence bands
#   mean_estimate <- update.theta_exact$sm[index,]
#   ci_upper <- mean_estimate + sqrt(update.theta_exact$sC[index, index,]) * qnorm(0.975)
#   ci_lower <- mean_estimate + sqrt(update.theta_exact$sC[index, index,]) * qnorm(0.025)
  
#   # Time points
#   time_points <- seq_along(mean_estimate)
  
#   # Plot time series
#   plot(time_points, mean_estimate, type = "l", col = 'darkblue', ylim = range(theta[, index]), lwd = 1.5,
#        main = paste("State Variable", index), ylab = "Estimate", xlab = "Time")
  
#   # Add filled credible interval using polygon()
#   polygon(c(time_points, rev(time_points)), c(ci_upper, rev(ci_lower)), 
#           col = adjustcolor("lightblue", alpha.f = 0.5), border = NA)
  
#   # Replot mean estimate to keep it on top
#   lines(time_points, mean_estimate, col = 'darkblue', lwd = 0.1)
  
#   # Overlay true state values
#   points(time_points, theta[, index], col = 'red', pch = 19, cex = 0.1)  # Smaller points for clarity
#   lines(time_points, theta[, index], col = 'red', lwd = 0.01)
# }

# # List of state indices to plot
# state_indices <- c(1, 2, 3, 5, 6)

# # Apply function over state indices
# par(mfrow = c(1,1))  # Arrange plots in grid (3 rows, 2 columns)
# lapply(state_indices, plot_state, theta = theta, update.theta_exact = update.theta_exact)



In [ ]:
# Load necessary library
library(ggplot2)

# Function to run and plot simulations
run_simulation <- function(seed) {
  
  # Define parameters
  TT <- 1200
  B  <- 5000
  T  <- TT + B - 1
  trend_degree <- 1
  harmonics    <- c(1)
  period       <- 12
  transfer_rate  <- 0.995
  observation_sd <- 0.5
  evolution_sd   <- 1.0e-70
  
  # Set seed for reproducibility
  set.seed(seed)
  
  # Generate covariates
  covariates <- exp(matrix(rnorm(T * 1, sd = 4), ncol = 1))  # 1 random covariate
  
  # Generate matrices
  dlm_matrices <- set_matrices(
    trend_degree = trend_degree,
    harmonics = harmonics, 
    period = period,
    covariates = covariates,
    covariate_mode = "transfer",
    transfer_rate = transfer_rate,
    T = T
  )
  
  # Initial state
  initial_state <- c(-1000, 500, 100000, 0, 5, 2)/100000*observation_sd
  
  # Simulate the model
  simulation_results <- simulate_dlm(
    G = dlm_matrices$G,
    F = dlm_matrices$F,
    T = T,
    initial_state = initial_state,
    observation_sd = observation_sd,
    evolution_sd = evolution_sd
  )
  
  # Extract results
  y <- simulation_results$y[B:T]
  theta <- simulation_results$theta[B:T, ]
  theta0 <- simulation_results$theta[B-1, ]
  
  # Save observation plot
  # png(filename = paste0("simulated_observations_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
  plot(y, type = "l", col = "black", lwd = 0.5, xlab = "Time", ylab = "Observation", 
       main = paste("Simulated Observations (Seed:", seed, ")"))
  points(y, pch = 19, col = "black", cex = 0.5)
  # dev.off()
  
  # Save state components plot
  # png(filename = paste0("simulated_states_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
  matplot(theta, type = "l", lty = 1, lwd = 1.5, xlab = "Time", ylab = "State Components", 
          main = paste("Simulated State Components (Seed:", seed, ")"))
  legend("topright", legend = paste0("State ", 1:ncol(theta)), col = 1:ncol(theta), lty = 1, cex = 0.8)
  # dev.off()
  
  # Set up model parameters
  p = 4
  ppx = 2
  J = 0
  
  nu0 <- rep(4, 1)  # Degrees of freedom (>2 for well-defined mean)
  S0 <- nu0 * rep(observation_sd, 1)  # Scale (high values imply weak prior knowledge)
  m0 <- theta0
  C0 <- diag(rep(120 / (p + ppx), ncol(theta)))  # Large variance for vague prior
  ones <- matrix(1, ncol = (ncol(theta)), nrow = (ncol(theta)))
  
  # Create degree of freedom matrices
  df = 1 - 1e-6
  dim.df = c(6)
  k <- 2
  ex_df_mat <- make_df_mat(df, dim.df, p + ppx + J)
  ex_df_mat_k <- make_df_mat_k(df, dim.df, p + ppx + J, k)
  
  # Load Kalman filter function
  Rcpp::sourceCpp('/data/muscat_data/jaguir26/project1_ucsc_phd/kalman_NDLM.cpp')

  # Fit model
  fit_results <- update_theta_cpp_ndlm_exactV(
    GG = dlm_matrices$G[,,B:T, drop = FALSE],
    m0 = m0,
    C0 = C0,
    FF = dlm_matrices$F[,,B:T, drop = FALSE],
    y = matrix(y, nrow = 1),
    ex_df_mat = ex_df_mat,
    ex_df_mat_k = ex_df_mat_k,
    Ones = ones,
    p = 4,
    J = 0,
    ppx = 1 + 1,
    TT = TT,
    k = k,
    dM = 0,
    nu0 = nu0,
    S0 = S0
  )
  
  update.theta_exact <- fit_results
  
  # Compute expectations
  FF_t <- aperm(dlm_matrices$F[,,B:T, drop = FALSE], c(2, 1, 3))
  
  compute_exp <- function(slice_index) {
    FF_t[,,slice_index] %*% update.theta_exact$sm[,slice_index]
  }
  
  result_list <- lapply(1:ncol(update.theta_exact$sm), compute_exp)
  result_array <- array(unlist(result_list), dim = c(J + 1, 1, ncol(update.theta_exact$sm)))
  result_array <- aperm(result_array, c(1, 3, 2))[,,1]
  exps <- result_array
  
  # Compute variances
  compute_var <- function(t) {
    FF_t_slice <- FF_t[,,t]
    sC_slice <- update.theta_exact$sC[,,t]
    ((FF_t_slice) %*% sC_slice) %*% (FF_t_slice)
  }
  
  result_list_1 <- lapply(1:ncol(update.theta_exact$sm), compute_var)
  vars_1 <- simplify2array(result_list_1)
  
  S_TT <- c(update.theta_exact$S_t)
  n_TT <- c(update.theta_exact$nu_t)
  
  # Save prediction plot with filled credible intervals
  # png(filename = paste0("prediction_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
  plot(y, type = "l", col = "black", lwd = 0.5, xlab = "Time", ylab = "Observation", 
       main = paste("Prediction (Seed:", seed, ")"))
  points(y, pch = 19, col = "black", cex = 0.5)
  
  ci_upper <- exps + sqrt(S_TT / n_TT) * qnorm(0.975)
  ci_lower <- exps + sqrt(S_TT / n_TT) * qnorm(0.025)
  
  polygon(c(seq_along(y), rev(seq_along(y))), c(ci_upper, rev(ci_lower)), 
          col = adjustcolor("lightblue", alpha.f = 0.5), border = NA)
  
  lines(exps, col = 'darkorange', lwd = 0.8)
  # dev.off()

  # Save state estimates plot
  plot_state <- function(index) {
    mean_estimate <- update.theta_exact$sm[index,]
    ci_upper <- mean_estimate + sqrt(update.theta_exact$sC[index, index,]) * qnorm(0.975)
    ci_lower <- mean_estimate + sqrt(update.theta_exact$sC[index, index,]) * qnorm(0.025)
    
    time_points <- seq_along(mean_estimate)
    
    # png(filename = paste0("state_", index, "_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
    plot(time_points, mean_estimate, type = "l", col = 'darkblue', ylim = range(theta[, index]), lwd = 1.5,
         main = paste("State Variable", index, "(Seed:", seed, ")"), ylab = "Estimate", xlab = "Time")
    polygon(c(time_points, rev(time_points)), c(ci_upper, rev(ci_lower)), 
            col = adjustcolor("lightblue", alpha.f = 0.5), border = NA)
    lines(time_points, mean_estimate, col = 'darkblue', lwd = 1.5)
    points(time_points, theta[, index], col = 'red', pch = 19, cex = 0.5)
    lines(time_points, theta[, index], col = 'red', lwd = 0.5)
    # dev.off()
  }

  lapply(c(1, 2, 3, 5, 6), plot_state)
  
  cat("Simulation and plots completed for seed:", seed, "\n")
}

# Run simulations for different seeds
# seeds <- c(111, 300, 666, 888)
seeds <- c(111, 300, 666, 3000916, 888891678)
lapply(seeds, run_simulation)


In [ ]:
# make_df_mat = function(df,dim.df,n){
#   if(sum(dim.df)!=n){ stop("sum of component dimensions given in dim.df does not match m0") }
#   if(length(df)!=length(dim.df)){ stop("length of component discount factors does not match length of component dimensions") }
#   dfs = rep(df,dim.df)
#   n.dfs = length(dim.df)
#   ind.dfs = c(0,sapply(1:length(dim.df),function(x){sum(dim.df[1:x])}),n)
#   df.mat = matrix(0,n,n)
#   for(j in 1:n.dfs){
#     df.mat[(ind.dfs[j]+1):ind.dfs[(j+1)],(ind.dfs[j]+1):ind.dfs[(j+1)]] = (1-dfs[ind.dfs[(j+1)]])/dfs[ind.dfs[(j+1)]]
#   }
#   return(df.mat)
# }
# make_df_mat_k = function(df,dim.df,n,k){
#   if(sum(dim.df)!=n){ stop("sum of component dimensions given in dim.df does not match m0") }
#   if(length(df)!=length(dim.df)){ stop("length of component discount factors does not match length of component dimensions") }
#   dfs = rep(df,dim.df)
#   n.dfs = length(dim.df)
#   ind.dfs = c(0,sapply(1:length(dim.df),function(x){sum(dim.df[1:x])}),n)
#   df.mat = matrix(0,n,n)
#   for(j in 1:n.dfs){
#     df.mat[(ind.dfs[j]+1):ind.dfs[(j+1)],(ind.dfs[j]+1):ind.dfs[(j+1)]] = (1-dfs[ind.dfs[(j+1)]]^k)/dfs[ind.dfs[(j+1)]]^k
#   }
#   return(df.mat)
# }

In [ ]:
# update.theta_exact <- fit_results

# FF_t <- aperm(FF, c(2, 1, 3))
# multiply_matrices <- function(slice_index) {
#     FF_t[,,slice_index] %*% update.theta_exact$sm[,slice_index]
# }
# result_list <- lapply(1:ncol(update.theta_exact$sm), multiply_matrices)
# result_array <- array(unlist(result_list), dim = c(J+1, 1, ncol(update.theta_exact$sm)))
# result_array <- aperm(result_array, c(1, 3, 2))[,,1]
# exps <- result_array

# compute_product_1 <- function(t) {
#     FF_t_slice <- FF_t[,,t]
#     sC_slice <- update.theta_exact$sC[,,t]
#     FF_slice <- FF[,,t]
#     result_slice <- t(FF_slice)%*%sC_slice%*%(FF_slice )
#     return(result_slice)
# }
# result_list_1 <- lapply(1:dim(FF)[3], compute_product_1)
# vars_1 <- simplify2array(result_list_1)

# if(J>0){
# vars <- (apply(vars_1, 3, function(x) diag(x)))
# exps2 = exps^2 + vars
# }else{
# exps2 = exps^2 + vars_1
# vars_1 <- array( vars_1, c(1,TT) )  
# exps2 <- array( exps2, c(1,TT) )  
# exps <- array( exps, c(1,TT) )    
# }

# new.theta.out  <- update.theta_exact 
# new.theta.out$exps  <- exps
# new.theta.out$exps2  <- exps2
# new.theta.out$vars  <- vars
# exps_exact  <- exps


In [ ]:
# S_TT <- c(update.theta_exact$S_t)
# n_TT <- c(update.theta_exact$nu_t)
# S_TT
# n_TT
# observation_sd
# S_TT/n_TT

In [ ]:
# plot.ts(y, lwd = 0.1)
# points(y, pch = 19, lwd = 0.1)
# abline(h = sum(FF_t[,,1]*theta0))
# lines(exps, col = 'darkorange', lty = 1, lwd = 1.5)
# lines(exps+sqrt(S_TT/n_TT)*qnorm(0.975), col = 'lightblue', lty = 2, lwd = 1.5)
# lines(exps+sqrt(S_TT/n_TT)*qnorm(0.025), col = 'lightblue', lty = 2, lwd = 1.5)

In [ ]:
# plot.ts(t(update.theta_exact$sm[,]), col='red', lty = 2)
# plot.ts(t(update.theta_exact$fm[,]), col='purple', lty = 2)
# plot.ts(theta[,], col='blue')

In [ ]:
# length(theta[,1])
# length(update.theta_exact$sm[1,])

In [ ]:
# # Function to plot state estimates with filled credible intervals
# plot_state <- function(index, theta, update.theta_exact) {
#   # Extract filtered state estimates and confidence bands
#   mean_estimate <- update.theta_exact$sm[index,]
#   ci_upper <- mean_estimate + sqrt(update.theta_exact$sC[index, index,]) * qnorm(0.975)
#   ci_lower <- mean_estimate + sqrt(update.theta_exact$sC[index, index,]) * qnorm(0.025)
  
#   # Time points
#   time_points <- seq_along(mean_estimate)
  
#   # Plot time series
#   plot(time_points, mean_estimate, type = "l", col = 'darkblue', ylim = range(theta[, index]), lwd = 1.5,
#        main = paste("State Variable", index), ylab = "Estimate", xlab = "Time")
  
#   # Add filled credible interval using polygon()
#   polygon(c(time_points, rev(time_points)), c(ci_upper, rev(ci_lower)), 
#           col = adjustcolor("lightblue", alpha.f = 0.5), border = NA)
  
#   # Replot mean estimate to keep it on top
#   lines(time_points, mean_estimate, col = 'darkblue', lwd = 0.1)
  
#   # Overlay true state values
#   points(time_points, theta[, index], col = 'red', pch = 19, cex = 0.1)  # Smaller points for clarity
#   lines(time_points, theta[, index], col = 'red', lwd = 0.01)
# }

# # List of state indices to plot
# state_indices <- c(1, 2, 3, 5, 6)

# # Apply function over state indices
# par(mfrow = c(1,1))  # Arrange plots in grid (3 rows, 2 columns)
# lapply(state_indices, plot_state, theta = theta, update.theta_exact = update.theta_exact)


In [ ]:
# errors <- matrix(fit_results$standard_forecast_errors_k, ncol = 1)
# plot.ts(errors)

# errors <- matrix(fit_results$standard_forecast_errors, ncol = 1)
# plot.ts(errors)

In [ ]:
# df = 1-1e-6
# dim.df = c(6); 

# ex_df_mat <- make_df_mat(df, dim.df, p+ppx+J)
# ex_df_mat_k <- make_df_mat_k(df, dim.df, p+ppx+J, k)

# fit_results <- update_theta_cpp_ndlm_exactV(
#       GG = dlm_matrices$G[,,B:T,drop = FALSE],
#       m0 = m0,
#       C0 = C0,
#       FF = dlm_matrices$F[,,B:T,drop = FALSE],
#       y = matrix(y, nrow=1),  # Reshape observations into a column vector
#       ex_df_mat = ex_df_mat,
#       ex_df_mat_k = ex_df_mat_k,
#       Ones = ones,  # Identity matrix for compatibility
#       p = 4,                     # Number of trend components (adjust as necessary)
#       J = 0,       # Number of observed components
#       ppx = 1+1,                   # No explanatory variables
#       TT = TT,                    # Number of time points
#       k = k,                     # 1-step ahead forecast
#       dM = 0,                    # No additional parameters
#       nu0 = nu0,
#       S0 = S0
# )

# compute_jsd <- function(p_sample, gridsize = c(100)) {
#   # Step 2: Perform KDE on the sample to estimate the density of p
#   kde_p <- kde(p_sample, gridsize = gridsize)
  
#   # Extract density estimate and grid points
#   pdf_p <- kde_p$estimate  # Estimated density
#   eval_points <- kde_p$eval.points  # Grid points
  
#   if (is.null(pdf_p) || is.null(eval_points)) {
#     stop("KDE estimation failed. Check input dimensions or grid size.")
#   }
  
#   # Step 4: Define the distribution q (standard univariate normal)
#   mean_q <- 0  # Mean of standard normal
#   sd_q <- 1    # Standard deviation of standard normal
  
#   # Evaluate the standard normal PDF on the same grid as kde_p
#   pdf_q <- dnorm(eval_points, mean = mean_q, sd = sd_q)
  
#   # Step 6: Normalize the densities
#   pdf_p <- pdf_p / sum(pdf_p)
#   pdf_q <- pdf_q / sum(pdf_q)
  
#   # Step 7: Function to compute the KL divergence
#   KL.divergence <- function(p, q) {
#     epsilon <- 1e-10  # Small value to prevent division by zero or log of zero
#     p <- p + epsilon
#     q <- q + epsilon
#     return(sum(p * log(p / q)))
#   }
  
#   # Step 8: Function to compute the Jensen-Shannon divergence
#   JSD <- function(p, q) {
#     m <- 0.5 * (p + q)
#     return(0.5 * KL.divergence(p, m) + 0.5 * KL.divergence(q, m))
#   }
  
#   # Step 9: Compute the Jensen-Shannon divergence
#   js_divergence <- JSD(pdf_p, pdf_q)
#   return(js_divergence)
# }

#   compute_crps <- function(x) {
#     cdf <- pnorm(x)  # Standard normal CDF
#     pdf <- dnorm(x)  # Standard normal PDF
#     crps <- x * (2 * cdf - 1) + 2  * pdf - 1/sqrt(pi)
#     return(crps)
#   }

#   compute_mean_crps <- function(errors,q) {
#     crps_values <- sapply(errors, compute_crps)  # Apply CRPS formula
#     mean_crps <- mean(q*crps_values)  # Take the mean of all CRPS values
#     return(mean_crps)
#   }

#   scores <- rep(0,4)
#   for(t in 1:TT){
#       ft   <-  FF_t[,,t]%*%t(mvtnorm::rmvnorm(n.samp, mean = update.theta_exact$sm[,t], sigma = update.theta_exact$sC[,,t]))
#       qt   <-  FF_t[,,t]%*%update.theta_exact$sC[,,t]%*%FF_t[,,t] + S_TT
#       e_t  <- c(1/sqrt(c(qt))*(y[t] - ft))

#       scores <- scores + c(compute_kl_divergence(e_t)
#                           ,estimate_kl_divergence(e_t)
#                           ,compute_jsd(e_t, gridsize = c(100))
#                           ,compute_mean_crps(e_t, c(qt)))

#   }
#   scores <- scores/TT
  
#   return(scores[3])

# }

In [ ]:
# library(parallel)
# library(mvtnorm)

# n_samp <- 300


# # df = 1-1e-6
# # dim.df = c(6); 

# f_obj <- function(x){

# df = x[1]
# dim.df = c(6); 

# transfer_rate  <- x[2]

# # Generate matrices
# dlm_matrices_ <- set_matrices(
#   trend_degree = trend_degree,
#   harmonics = harmonics, 
#   period = period,
#   covariates = covariates,
#   covariate_mode = "transfer",
#   transfer_rate = transfer_rate,
#   T = T
# )

# ex_df_mat <- make_df_mat(df, dim.df, p+ppx+J)
# ex_df_mat_k <- make_df_mat_k(df, dim.df, p+ppx+J, k)

# fit_results <- update_theta_cpp_ndlm_exactV(
#       GG = dlm_matrices_$G[,,B:T,drop = FALSE],
#       m0 = m0,
#       C0 = C0,
#       FF = dlm_matrices_$F[,,B:T,drop = FALSE],
#       y = matrix(y, nrow=1),  # Reshape observations into a column vector
#       ex_df_mat = ex_df_mat,
#       ex_df_mat_k = ex_df_mat_k,
#       Ones = ones,  # Identity matrix for compatibility
#       p = 4,                     # Number of trend components (adjust as necessary)
#       J = 0,       # Number of observed components
#       ppx = 1+1,                   # No explanatory variables
#       TT = TT,                    # Number of time points
#       k = k,                     # 1-step ahead forecast
#       dM = 0,                    # No additional parameters
#       nu0 = nu0,
#       S0 = S0
# )

# # Function to pre-generate samples for all time steps
# generate_samples <- function(fit_results, TT, n_samp) {
#   samples <- lapply(1:TT, function(t) {
#     mvtnorm::rmvnorm(n_samp, 
#                      mean = fit_results$sm[, t], 
#                      sigma = fit_results$sC[,, t])
#   })
#   return(samples)
# }

# # Step 1: Pre-generate samples
# samples <- generate_samples(fit_results, TT, n_samp)


# compute_scores_parallel <- function(TT, samples, FF_t, S_TT, y, fit_results) {
#   # Parallel computation for scores
#   results <- mclapply(1:TT, function(t) {
#     # Debugging: Print current time step
#     cat("Time step:", t, "\n")
    
#     # Compute forecast samples
#     ft <- FF_t[,,t] %*% t(samples[[t]])
    
#     # Compute forecast variance
#     qt <- c(FF_t[,,t] %*% fit_results$sC[,,t] %*% FF_t[,,t] + S_TT)
    
#     # Standardized forecast errors
#     e_t <- c(1 / sqrt(qt) * (y[t] - ft))
    
#     # Return both results as a list
#     list(e_t = e_t, qt = qt)
#   }, mc.cores = detectCores() - 1)
  
#   # Extract e_t and qt from the results list
#   e_t_list <- lapply(results, function(x) x$e_t)  # Extract e_t
#   qt_list <- lapply(results, function(x) x$qt)   # Extract qt

#   # Convert lists to matrices
#   e_t_matrix <- do.call(rbind, e_t_list)  # Matrix of e_t (T x S)
#   qt_matrix <- do.call(rbind, qt_list)   # Matrix of qt (T x 1)

#   # Return matrices
#   return(list(e_t_matrix = e_t_matrix, qt_matrix = qt_matrix))
# }

# # Parallelized execution
# results <- compute_scores_parallel(
#   TT = TT, 
#   samples = samples, 
#   FF_t = FF_t, 
#   S_TT = S_TT, 
#   y = y, 
#   fit_results = fit_results
# )

# # Extract matrices
# e_t_matrix <- results$e_t_matrix
# qt_matrix <- results$qt_matrix

# compute_crps <- function(x) {
#   cdf <- pnorm(x)  # Standard normal CDF
#   pdf <- dnorm(x)  # Standard normal PDF
#   crps <- x * (2 * cdf - 1) + 2  * pdf - 1/sqrt(pi)
#   return(crps)
# }

# compute_jsd_200 <- function(x) {
#   return(compute_jsd(x, gridsize = c(200)))
# }

# compute_jsd <- function(p_sample, gridsize = c(100)) {
#   # Step 2: Perform KDE on the sample to estimate the density of p
#   kde_p <- kde(p_sample, gridsize = gridsize)
  
#   # Extract density estimate and grid points
#   pdf_p <- kde_p$estimate  # Estimated density
#   eval_points <- kde_p$eval.points  # Grid points
  
#   if (is.null(pdf_p) || is.null(eval_points)) {
#     stop("KDE estimation failed. Check input dimensions or grid size.")
#   }
  
#   # Step 4: Define the distribution q (standard univariate normal)
#   mean_q <- 0  # Mean of standard normal
#   sd_q <- 1    # Standard deviation of standard normal
  
#   # Evaluate the standard normal PDF on the same grid as kde_p
#   pdf_q <- dnorm(eval_points, mean = mean_q, sd = sd_q)
  
#   # Step 6: Normalize the densities
#   pdf_p <- pdf_p / sum(pdf_p)
#   pdf_q <- pdf_q / sum(pdf_q)
  
#   # Step 7: Function to compute the KL divergence
#   KL.divergence <- function(p, q) {
#     epsilon <- 1e-10  # Small value to prevent division by zero or log of zero
#     p <- p + epsilon
#     q <- q + epsilon
#     return(sum(p * log(p / q)))
#   }
  
#   # Step 8: Function to compute the Jensen-Shannon divergence
#   JSD <- function(p, q) {
#     m <- 0.5 * (p + q)
#     return(0.5 * KL.divergence(p, m) + 0.5 * KL.divergence(q, m))
#   }
  
#   # Step 9: Compute the Jensen-Shannon divergence
#   js_divergence <- JSD(pdf_p, pdf_q)
#   return(js_divergence)
# }

# compute_crps <- function(x) {
#   cdf <- pnorm(x)  # Standard normal CDF
#   pdf <- dnorm(x)  # Standard normal PDF
#   crps <- x * (2 * cdf - 1) + 2  * pdf - 1/sqrt(pi)
#   return(crps)
# }

# # Compute mean CRPS for a vector of errors
# compute_mean_crps <- function(errors,q) {
#   crps_values <- sapply(errors, compute_crps)  # Apply CRPS formula
#   mean_crps <- mean(q*crps_values)  # Take the mean of all CRPS values
#   return(mean_crps)
# }

#   compute_crps <- function(x) {
#     cdf <- pnorm(x)  # Standard normal CDF
#     pdf <- dnorm(x)  # Standard normal PDF
#     crps <- x * (2 * cdf - 1) + 2  * pdf - 1/sqrt(pi)
#     return(crps)
#   }

#   compute_mean_crps <- function(errors,q) {
#     crps_values <- sapply(errors, compute_crps)  # Apply CRPS formula
#     mean_crps <- mean(q*crps_values)  # Take the mean of all CRPS values
#     return(mean_crps)
#   }

#   # for(t in 1:TT){
#   #     ft   <-  FF_t[,,t]%*%t(mvtnorm::rmvnorm(n.samp, mean = update.theta_exact$sm[,t], sigma = update.theta_exact$sC[,,t]))
#   #     qt   <-  FF_t[,,t]%*%update.theta_exact$sC[,,t]%*%FF_t[,,t] + S_TT
#   #     e_t  <- c(1/sqrt(c(qt))*(y[t] - ft))

#   #     scores <- scores + c(compute_kl_divergence(e_t)
#   #                         ,estimate_kl_divergence(e_t)
#   #                         ,compute_jsd(e_t, gridsize = c(100))
#   #                         ,compute_mean_crps(e_t, c(qt)))

#   # }
#   # scores <- scores/TT

#   scores <- rep(0,4)
#   scores[1] <- mean(apply(e_t_matrix, 2, compute_kl_divergence))
#   scores[2] <- mean(apply(e_t_matrix, 2, estimate_kl_divergence))
#   scores[3] <- mean(qt_matrix*colSums(apply(e_t_matrix, 2, compute_crps))/TT)
#   scores[4] <- mean(apply(e_t_matrix, 2, compute_jsd_200))
  
#   return(scores[4])
  
# }



In [ ]:
# f_obj(c(1-1e-6,0.999))
# f_obj(c(0.998, 0.9998))

In [ ]:
# observation_sd 
# evolution_sd   
# S_TT/n_

## Optimization

In [ ]:
# library(ggplot2)
# library(parallel)
# library(mvtnorm)
# library(GA)
# library(parallel)

# run_simulation <- function(seed) {
#   TT <- 1200
#   B  <- 5000
#   T  <- TT + B - 1
#   trend_degree <- 1
#   harmonics    <- c(1)
#   period       <- 12
#   transfer_rate  <- 0.995
#   observation_sd <- 1
#   evolution_sd   <- 1.0e-70
  
#   set.seed(seed)
  
#   covariates <- exp(matrix(rnorm(T * 1, sd = 4), ncol = 1))  # 1 random covariate
  
#   # Generate matrices
#   dlm_matrices <- set_matrices(
#     trend_degree = trend_degree,
#     harmonics = harmonics, 
#     period = period,
#     covariates = covariates,
#     covariate_mode = "transfer",
#     transfer_rate = transfer_rate,
#     T = T
#   )
  
#   # Initial state
#   initial_state <- c(-1000, 500, 100000, 0, 5, 2)/100000*observation_sd
  
#   # Simulate the model
#   simulation_results <- simulate_dlm(
#     G = dlm_matrices$G,
#     F = dlm_matrices$F,
#     T = T,
#     initial_state = initial_state,
#     observation_sd = observation_sd,
#     evolution_sd = evolution_sd
#   )
  
#   # Extract results
#   y <- simulation_results$y[B:T]
#   theta <- simulation_results$theta[B:T, ]
#   theta0 <- simulation_results$theta[B-1, ]
  
#   # Save observation plot
#   png(filename = paste0("simulated_observations_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
#   plot(y, type = "l", col = "blue", lwd = 1.5, xlab = "Time", ylab = "Observation", 
#        main = paste("Simulated Observations (Seed:", seed, ")"))
#   dev.off()
  
#   # Save state components plot
#   png(filename = paste0("simulated_states_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
#   matplot(theta, type = "l", lty = 1, lwd = 1.5, xlab = "Time", ylab = "State Components", 
#           main = paste("Simulated State Components (Seed:", seed, ")"))
#   legend("topright", legend = paste0("State ", 1:ncol(theta)), col = 1:ncol(theta), lty = 1, cex = 0.8)
#   dev.off()
  
#   # Set up model parameters
#   p = 4
#   ppx = 2
#   J = 0
  
#   nu0 <- rep(4, 1)  # Degrees of freedom (>2 for well-defined mean)
#   S0 <- nu0 * rep(observation_sd, 1)  # Scale (high values imply weak prior knowledge)
#   m0 <- theta0
#   C0 <- diag(rep(120 / (p + ppx), ncol(theta)))  # Large variance for vague prior
#   ones <- matrix(1, ncol = (ncol(theta)), nrow = (ncol(theta)))
  
#   # Create degree of freedom matrices
#   df = 1 - 1e-10
#   dim.df = c(6)
#   k <- 2
#   ex_df_mat <- make_df_mat(df, dim.df, p + ppx + J)
#   ex_df_mat_k <- make_df_mat_k(df, dim.df, p + ppx + J, k)
  
#   # Load Kalman filter function
#   Rcpp::sourceCpp('/data/muscat_data/jaguir26/project1_ucsc_phd/kalman_NDLM.cpp')

#   # Fit model
#   fit_results <- update_theta_cpp_ndlm_exactV(
#     GG = dlm_matrices$G[,,B:T, drop = FALSE],
#     m0 = m0,
#     C0 = C0,
#     FF = dlm_matrices$F[,,B:T, drop = FALSE],
#     y = matrix(y, nrow = 1),
#     ex_df_mat = ex_df_mat,
#     ex_df_mat_k = ex_df_mat_k,
#     Ones = ones,
#     p = 4,
#     J = 0,
#     ppx = 1 + 1,
#     TT = TT,
#     k = k,
#     dM = 0,
#     nu0 = nu0,
#     S0 = S0
#   )
  
#   update.theta_exact <- fit_results
  
#   # Compute expectations
#   FF_t <- aperm(dlm_matrices$F[,,B:T, drop = FALSE], c(2, 1, 3))
  
#   compute_exp <- function(slice_index) {
#     FF_t[,,slice_index] %*% update.theta_exact$sm[,slice_index]
#   }
  
#   result_list <- lapply(1:ncol(update.theta_exact$sm), compute_exp)
#   result_array <- array(unlist(result_list), dim = c(J + 1, 1, ncol(update.theta_exact$sm)))
#   result_array <- aperm(result_array, c(1, 3, 2))[,,1]
#   exps <- result_array
  
#   # Compute variances
#   compute_var <- function(t) {
#     FF_t_slice <- FF_t[,,t]
#     sC_slice <- update.theta_exact$sC[,,t]
#     ((FF_t_slice) %*% sC_slice) %*% (FF_t_slice)
#   }
  
#   result_list_1 <- lapply(1:ncol(update.theta_exact$sm), compute_var)
#   vars_1 <- simplify2array(result_list_1)
  
#   S_TT <- c(update.theta_exact$S_t)
#   n_TT <- c(update.theta_exact$nu_t)
  
#   # Save prediction plot with filled credible intervals
#   png(filename = paste0("prediction_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
#   plot(y, type = "l", col = "black", lwd = 0.5, xlab = "Time", ylab = "Observation", 
#        main = paste("Prediction (Seed:", seed, ")"))
#   points(y, pch = 19, col = "black", cex = 0.5)
  
#   ci_upper <- exps + sqrt(S_TT / n_TT) * qnorm(0.975)
#   ci_lower <- exps + sqrt(S_TT / n_TT) * qnorm(0.025)
  
#   polygon(c(seq_along(y), rev(seq_along(y))), c(ci_upper, rev(ci_lower)), 
#           col = adjustcolor("lightblue", alpha.f = 0.5), border = NA)
  
#   lines(exps, col = 'darkorange', lwd = 0.8)
#   dev.off()

#   # Save state estimates plot
#   plot_state <- function(index) {
#     mean_estimate <- update.theta_exact$sm[index,]
#     ci_upper <- mean_estimate + sqrt(update.theta_exact$sC[index, index,]) * qnorm(0.975)
#     ci_lower <- mean_estimate + sqrt(update.theta_exact$sC[index, index,]) * qnorm(0.025)
    
#     time_points <- seq_along(mean_estimate)
    
#     png(filename = paste0("state_", index, "_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
#     plot(time_points, mean_estimate, type = "l", col = 'darkblue', ylim = range(theta[, index]), lwd = 1.5,
#          main = paste("State Variable", index, "(Seed:", seed, ")"), ylab = "Estimate", xlab = "Time")
#     polygon(c(time_points, rev(time_points)), c(ci_upper, rev(ci_lower)), 
#             col = adjustcolor("lightblue", alpha.f = 0.5), border = NA)
#     lines(time_points, mean_estimate, col = 'darkblue', lwd = 1.5)
#     points(time_points, theta[, index], col = 'red', pch = 19, cex = 0.5)
#     lines(time_points, theta[, index], col = 'red', lwd = 0.5)
#     dev.off()
#   }

#   lapply(c(1, 2, 3, 5, 6), plot_state)
  
#   cat("Simulation and plots completed for seed:", seed, "\n")

#   n_samp <- 400

#   f_obj <- function(x, iii){

#   # df = x[1]
#   df = 1 - 1e-10
#   dim.df = c(6); 

#   # transfer_rate  <- x[2]
#   transfer_rate  <- x

#   # Generate matrices
#   dlm_matrices_ <- set_matrices(
#     trend_degree = trend_degree,
#     harmonics = harmonics, 
#     period = period,
#     covariates = covariates,
#     covariate_mode = "transfer",
#     transfer_rate = transfer_rate,
#     T = T
#   )

#   ex_df_mat <- make_df_mat(df, dim.df, p+ppx+J)
#   ex_df_mat_k <- make_df_mat_k(df, dim.df, p+ppx+J, k)

#   fit_results <- update_theta_cpp_ndlm_exactV(
#         GG = dlm_matrices_$G[,,B:T,drop = FALSE],
#         m0 = m0,
#         C0 = C0,
#         FF = dlm_matrices_$F[,,B:T,drop = FALSE],
#         y = matrix(y, nrow=1),  # Reshape observations into a column vector
#         ex_df_mat = ex_df_mat,
#         ex_df_mat_k = ex_df_mat_k,
#         Ones = ones,  # Identity matrix for compatibility
#         p = 4,                     # Number of trend components (adjust as necessary)
#         J = 0,       # Number of observed components
#         ppx = 1+1,                   # No explanatory variables
#         TT = TT,                    # Number of time points
#         k = k,                     # 1-step ahead forecast
#         dM = 0,                    # No additional parameters
#         nu0 = nu0,
#         S0 = S0
#   )

#   # Function to pre-generate samples for all time steps
#   generate_samples <- function(fit_results, TT, n_samp) {
#     samples <- lapply(1:TT, function(t) {
#       mvtnorm::rmvnorm(n_samp, 
#                       mean = fit_results$sm[, t], 
#                       sigma = fit_results$sC[,, t])
#     })
#     return(samples)
#   }

#   # Step 1: Pre-generate samples
#   samples <- generate_samples(fit_results, TT, n_samp)

#   compute_scores_parallel <- function(TT, samples, FF_t, S_TT, y, fit_results) {
#     # Parallel computation for scores
#     results <- mclapply(1:TT, function(t) {
#       # Debugging: Print current time step
#       cat("Time step:", t, "\n")
      
#       # Compute forecast samples
#       ft <- FF_t[,,t] %*% t(samples[[t]])
      
#       # Compute forecast variance
#       qt <- c(FF_t[,,t] %*% fit_results$sC[,,t] %*% FF_t[,,t] + S_TT)
      
#       # Standardized forecast errors
#       e_t <- c(1 / sqrt(qt) * (y[t] - ft))
      
#       # Return both results as a list
#       list(e_t = e_t, qt = qt)
#     }, mc.cores = detectCores() - 1)
    
#     # Extract e_t and qt from the results list
#     e_t_list <- lapply(results, function(x) x$e_t)  # Extract e_t
#     qt_list <- lapply(results, function(x) x$qt)   # Extract qt

#     # Convert lists to matrices
#     e_t_matrix <- do.call(rbind, e_t_list)  # Matrix of e_t (T x S)
#     qt_matrix <- do.call(rbind, qt_list)   # Matrix of qt (T x 1)

#     # Return matrices
#     return(list(e_t_matrix = e_t_matrix, qt_matrix = qt_matrix))
#   }

#   # Parallelized execution
#   results <- compute_scores_parallel(
#     TT = TT, 
#     samples = samples, 
#     FF_t = FF_t, 
#     S_TT = S_TT, 
#     y = y, 
#     fit_results = fit_results
#   )

#   # Extract matrices
#   e_t_matrix <- results$e_t_matrix
#   qt_matrix <- results$qt_matrix

#   compute_crps <- function(x) {
#     cdf <- pnorm(x)  # Standard normal CDF
#     pdf <- dnorm(x)  # Standard normal PDF
#     crps <- x * (2 * cdf - 1) + 2  * pdf - 1/sqrt(pi)
#     return(crps)
#   }

#   compute_jsd_200 <- function(x) {
#     return(compute_jsd(x, gridsize = c(200)))
#   }

#   compute_jsd <- function(p_sample, gridsize = c(100)) {
#     # Step 2: Perform KDE on the sample to estimate the density of p
#     kde_p <- kde(p_sample, gridsize = gridsize)
    
#     # Extract density estimate and grid points
#     pdf_p <- kde_p$estimate  # Estimated density
#     eval_points <- kde_p$eval.points  # Grid points
    
#     if (is.null(pdf_p) || is.null(eval_points)) {
#       stop("KDE estimation failed. Check input dimensions or grid size.")
#     }
    
#     # Step 4: Define the distribution q (standard univariate normal)
#     mean_q <- 0  # Mean of standard normal
#     sd_q <- 1    # Standard deviation of standard normal
    
#     # Evaluate the standard normal PDF on the same grid as kde_p
#     pdf_q <- dnorm(eval_points, mean = mean_q, sd = sd_q)
    
#     # Step 6: Normalize the densities
#     pdf_p <- pdf_p / sum(pdf_p)
#     pdf_q <- pdf_q / sum(pdf_q)
    
#     # Step 7: Function to compute the KL divergence
#     KL.divergence <- function(p, q) {
#       epsilon <- 1e-16  # Small value to prevent division by zero or log of zero
#       p <- p + epsilon
#       q <- q + epsilon
#       return(sum(p * log(p / q)))
#     }
    
#     # Step 8: Function to compute the Jensen-Shannon divergence
#     JSD <- function(p, q) {
#       m <- 0.5 * (p + q)
#       return(0.5 * KL.divergence(p, m) + 0.5 * KL.divergence(q, m))
#     }
    
#     # Step 9: Compute the Jensen-Shannon divergence
#     js_divergence <- JSD(pdf_p, pdf_q)
#     return(js_divergence)
#   }

#   compute_crps <- function(x) {
#     cdf <- pnorm(x)  # Standard normal CDF
#     pdf <- dnorm(x)  # Standard normal PDF
#     crps <- x * (2 * cdf - 1) + 2  * pdf - 1/sqrt(pi)
#     return(crps)
#   }

#   # Compute mean CRPS for a vector of errors
#   compute_mean_crps <- function(errors,q) {
#     crps_values <- sapply(errors, compute_crps)  # Apply CRPS formula
#     mean_crps <- mean(q*crps_values)  # Take the mean of all CRPS values
#     return(mean_crps)
#   }

#     compute_crps <- function(x) {
#       cdf <- pnorm(x)  # Standard normal CDF
#       pdf <- dnorm(x)  # Standard normal PDF
#       crps <- x * (2 * cdf - 1) + 2  * pdf - 1/sqrt(pi)
#       return(crps)
#     }

#     compute_mean_crps <- function(errors,q) {
#       crps_values <- sapply(errors, compute_crps)  # Apply CRPS formula
#       mean_crps <- mean(q*crps_values)  # Take the mean of all CRPS values
#       return(mean_crps)
#     }

#     compute_kl_200_1 <- function(x) {
#       return(compute_kl_divergences(x, gridsize = c(200), 1))
#     }

#     compute_kl_200_2 <- function(x) {
#       return(compute_kl_divergences(x, gridsize = c(200), 2))
#     }

#     compute_kl_divergences <- function(p_sample, gridsize = c(200), i) {
#     # Step 1: Perform Kernel Density Estimation (KDE) on the sample
#     kde_p <- kde(p_sample, gridsize = gridsize)
    
#     # Extract density estimate and grid points
#     pdf_p <- kde_p$estimate  # Estimated density of p_sample
#     eval_points <- kde_p$eval.points  # Grid points
    
#     if (is.null(pdf_p) || is.null(eval_points)) {
#       stop("KDE estimation failed. Check input dimensions or grid size.")
#     }
    
#     # Step 2: Define the standard normal distribution (N(0,1))
#     mean_q <- 0  # Mean of standard normal
#     sd_q <- 1    # Standard deviation of standard normal
    
#     # Evaluate the standard normal PDF on the same grid as kde_p
#     pdf_q <- dnorm(eval_points, mean = mean_q, sd = sd_q)
    
#     # Step 3: Normalize the estimated densities
#     pdf_p <- pdf_p / sum(pdf_p)  # Normalize estimated density
#     pdf_q <- pdf_q / sum(pdf_q)  # Normalize normal distribution
    
#     # Step 4: KL Divergence function
#     KL.divergence <- function(p, q) {
#       epsilon <- 1e-16  # Small value to prevent division by zero or log of zero
#       p <- p + epsilon
#       q <- q + epsilon
#       return(sum(p * log(p / q)))
#     }
    
#     # Compute KL divergences
#     if(i == 1){
#       KL <- KL.divergence(pdf_p, pdf_q)  # KL(p_sample || N(0,1))
#     }else{
#       KL <- KL.divergence(pdf_q, pdf_p)  # KL(N(0,1) || p_sample)
#     }

#     return(KL)
#   }

#     scores <- rep(0,4)
    
#     # scores[1] <- mean(apply(e_t_matrix, 2, compute_kl_divergence))
#     # scores[2] <- mean(apply(e_t_matrix, 2, estimate_kl_divergence))
    
#     scores[1] <- mean(apply(e_t_matrix, 2, compute_kl_200_1))
#     scores[2] <- mean(apply(e_t_matrix, 2, compute_kl_200_2))
#     scores[3] <- mean(qt_matrix*rowSums(apply(e_t_matrix, 2, compute_crps))/TT)
#     scores[4] <- mean(apply(e_t_matrix, 2, compute_jsd_200))
    
#     s <- scores[iii]

#     if(is.na(s)){
#       s <- 0
#     }
    
#     return(s)
     
#   }
  
  

# # Example usage:
# # sample_data <- rnorm(1000)  # Generate a sample from a normal distribution
# # kl_results <- compute_kl_divergences(sample_data)
# # print(kl_results)

#   # cat("Optimization has started for seed:", seed, "\n")
#   # ga_result <- ga(
#   # type = "real-valued",
#   # fitness = function(d) -f_obj(d),  # Minimize loss by negating
#   # lower = c(1-1e-6, 0.98)   ,
#   # upper = c(1-1e-60, 1-1e-60) ,
#   # popSize = 20,
#   # maxiter = 1000,
#   # run = 10,
#   # parallel = FALSE  # Disable internal parallel processing in GA
#   # )

#   # optimal_w <- ga_result@solution 
#   # minimum_loss <- -ga_result@fitnessValue
#   # cat("Optimization has finished for seed:", seed, "\n")
#   # cat("Optim:", optimal_w, "\n")


#   # Parallel evaluation of `f_obj`
#   x_values <- seq(0.988, 0.9999, length.out = 200)
#   assign("x_values", x_values, envir = .GlobalEnv)
#   num_cores <- detectCores() - 1
#   cl <- makeCluster(num_cores)
#   clusterExport(cl, varlist = c("set_matrices", "make_df_mat", "make_df_mat_k", 
#                                 "update_theta_cpp_ndlm_exactV", 'compute_kl_divergence','estimate_kl_divergence',
#                                 'estimate_kl_divergence_univariate','estimate_differential_entropy_kde_univariate',
#                                 'estimate_kl_divergence_univariate_normal_to_p', 'estimate_expectation_term_univariate',
#                                 'estimate_log_density_kde_univariate'))
#   clusterEvalQ(cl, {
#     library(parallel)
#     library(mvtnorm)
#     library(Rcpp)
#     library(RcppArmadillo)
#     library(RcppEigen)
#     library(ks)
#     library(MASS)
#     library(FNN)
#     Rcpp::sourceCpp('/data/muscat_data/jaguir26/project1_ucsc_phd/kalman_NDLM.cpp')
#   })

#   # assign("compute_kl_divergences", compute_kl_divergences, envir = .GlobalEnv)

#   # Assign f_obj to global environment
#   assign("f_obj", f_obj, envir = .GlobalEnv)
#   assign("transfer_rate", transfer_rate, envir = .GlobalEnv)

#   f_obj_1 <- function(x) -f_obj(x, 1)
#   f_obj_2 <- function(x) -f_obj(x, 2)
#   f_obj_3 <- function(x) f_obj(x, 3)
#   f_obj_4 <- function(x) -f_obj(x, 4)

#   # Ensure these functions exist in the global environment before parallelizing
#   assign("f_obj_1", f_obj_1, envir = .GlobalEnv)
#   assign("f_obj_2", f_obj_2, envir = .GlobalEnv)
#   assign("f_obj_3", f_obj_3, envir = .GlobalEnv)
#   assign("f_obj_4", f_obj_4, envir = .GlobalEnv)

#   # y_values1 <- parSapply(cl, x_values, f_obj_1)
#   # y_values1 <- y_values1/sd(y_values1)
#   # # stopCluster(cl)
#   # # output_file <- paste0("f_obj_plot_parallel_", seed, "_KLD1.png")
#   # png(filename = output_file, width = 2000, height = 1400, res = 300)
#   # plot(x_values, y_values1, type = "l", col = "darkblue", lwd = 2,
#   #      xlab = "x", ylab = "f_obj_4(x)",
#   #      main = paste("Plot of f_obj_4(x) (Seed:", seed, ")"))
#   # grid(col = "gray", lty = "dotted")
#   # points(x_values, y_values1, col = "red", pch = 19, cex = 0.5)
#   # abline(v=transfer_rate,col = "purple", lwd = 2)
#   # dev.off()
#   # cat("Plot saved to", output_file, "for seed:", seed, "\n")

#   # y_values2 <- parSapply(cl, x_values, f_obj_2)
#   # y_values2 <- y_values2/sd(y_values2)
#   # # stopCluster(cl)
#   # # output_file <- paste0("f_obj_plot_parallel_", seed, "_KLD2.png")
#   # png(filename = output_file, width = 2000, height = 1400, res = 300)
#   # plot(x_values, y_values2, type = "l", col = "darkblue", lwd = 2,
#   #      xlab = "x", ylab = "f_obj_4(x)",
#   #      main = paste("Plot of f_obj_4(x) (Seed:", seed, ")"))
#   # grid(col = "gray", lty = "dotted")
#   # points(x_values, y_values2, col = "red", pch = 19, cex = 0.5)
#   # abline(v=transfer_rate,col = "purple", lwd = 2)
#   # dev.off()
#   # cat("Plot saved to", output_file, "for seed:", seed, "\n")

#   # y_values3 <- parSapply(cl, x_values, f_obj_3)
#   # y_values3 <- y_values/sd(y_values3)
#   # # stopCluster(cl)
#   # # output_file <- paste0("f_obj_plot_parallel_", seed, "_CRPS.png")
#   # png(filename = output_file, width = 2000, height = 1400, res = 300)
#   # plot(x_values, y_values3, type = "l", col = "darkblue", lwd = 2,
#   #      xlab = "x", ylab = "f_obj_4(x)",
#   #      main = paste("Plot of f_obj_4(x) (Seed:", seed, ")"))
#   # grid(col = "gray", lty = "dotted")
#   # points(x_values, y_values3, col = "red", pch = 19, cex = 0.5)
#   # abline(v=transfer_rate,col = "purple", lwd = 2)
#   # dev.off()
#   # cat("Plot saved to", output_file, "for seed:", seed, "\n")

#   # y_values4 <- parSapply(cl, x_values, f_obj_4)
#   # y_values4 <- y_values/sd(y_values4)
#   # # stopCluster(cl)
#   # # output_file <- paste0("f_obj_plot_parallel_", seed, "_JSD.png")

#   # stopCluster(cl)
#   # output_file <- paste0("f_obj_plot_parallel_", seed, "_JSD.png")
#   # png(filename = output_file, width = 2000, height = 1400, res = 300)
#   # plot(x_values, y_values4, type = "l", col = "darkblue", lwd = 2,
#   #      xlab = "x", ylab = "f_obj_4(x)",
#   #      main = paste("Plot of f_obj_4(x) (Seed:", seed, ")"))
#   # grid(col = "gray", lty = "dotted")
#   # points(x_values, y_values4, col = "red", pch = 19, cex = 0.5)
#   # abline(v=transfer_rate,col = "purple", lwd = 2)
#   # dev.off()
#   # cat("Plot saved to", output_file, "for seed:", seed, "\n")


# library(viridis)

# f_obj_list <- list(f_obj_1, f_obj_2, f_obj_3, f_obj_4)

# num_cores <- min(4, detectCores() - 1) 
# cl <- makeCluster(num_cores)

# # Export functions and objects to the cluster
# clusterExport(cl, varlist = c("set_matrices", "make_df_mat", "make_df_mat_k",
#                               "update_theta_cpp_ndlm_exactV", "f_obj", 
#                               "f_obj_1", "f_obj_2", "f_obj_3", "f_obj_4", 
#                               "x_values", "transfer_rate","set_matrices", "make_df_mat", "make_df_mat_k", 
#                                 "update_theta_cpp_ndlm_exactV", 'compute_kl_divergence','estimate_kl_divergence',
#                                 'estimate_kl_divergence_univariate','estimate_differential_entropy_kde_univariate',
#                                 'estimate_kl_divergence_univariate_normal_to_p', 'estimate_expectation_term_univariate',
#                                 'estimate_log_density_kde_univariate'))

# # Ensure required libraries and compiled code are loaded on each node
# clusterEvalQ(cl, {
#     library(parallel)
#     library(mvtnorm)
#     library(Rcpp)
#     library(RcppArmadillo)
#     library(RcppEigen)
#     library(ks)
#     library(MASS)
#     library(FNN)
#     Rcpp::sourceCpp('/data/muscat_data/jaguir26/project1_ucsc_phd/kalman_NDLM.cpp')
# })

# y_values_list <- parLapply(cl, list(f_obj_1, f_obj_2, f_obj_3, f_obj_4), function(f) {
#   y_values <- sapply(x_values, f)
#   return(y_values / sd(y_values))  # Normalize
# })

# stopCluster(cl)  # Ensure we close the cluster properly

# y_values1 <- y_values_list[[1]]
# y_values2 <- y_values_list[[2]]
# y_values3 <- y_values_list[[3]]
# y_values4 <- y_values_list[[4]]

# # Extract computed values
# normalize <- function(y) {
#   (y - min(y)) / (max(y) - min(y))
# }
# y_values1 <- normalize(y_values1)
# y_values2 <- normalize(y_values2)
# y_values3 <- normalize(y_values3)
# y_values4 <- normalize(y_values4)

# y_max <- max(c(y_values1,y_values2,y_values3,y_values4))
# y_min <- min(c(y_values1,y_values2,y_values3,y_values4))

# # Professional color palette
# colors <- viridis(4, option = "D")

# output_file <- paste0("f_obj_plot_parallel_ALL_", seed, ".png")
# png(filename = output_file, width = 2000, height = 1400, res = 300)
# plot(x_values, y_values1, type = "l", col = colors[1], lwd = 2,
#      xlab = "x", ylab = expression(f[italic(i)](x)),
#      main = bquote("Plots of" ~ Scores ~ "(Seed:" ~ .(seed) ~ ")"),
#      ylim = c(y_min, y_max))
# lines(x_values, y_values2, col = colors[2], lwd = 2)
# lines(x_values, y_values3, col = colors[3], lwd = 2)
# lines(x_values, y_values4, col = colors[4], lwd = 2)

# abline(v=x_values[y_values1==min(y_values1)], col = colors[1], lwd = 2.5)
# abline(v=x_values[y_values2==min(y_values2)], col = colors[2], lwd = 2.5)
# abline(v=x_values[y_values3==min(y_values3)], col = colors[3], lwd = 2.5)
# abline(v=x_values[y_values4==min(y_values4)], col = colors[4], lwd = 2.5)

# grid(col = "gray", lty = "dotted")
# points(x_values, y_values1, col = colors[1], pch = 19, cex = 0.5)
# points(x_values, y_values2, col = colors[2], pch = 19, cex = 0.5)
# points(x_values, y_values3, col = colors[3], pch = 19, cex = 0.5)
# points(x_values, y_values4, col = colors[4], pch = 19, cex = 0.5)
# abline(v = transfer_rate, col = "red", lwd = 2, lty = 2)
# legend("topleft", legend = c(expression(KL[1]), expression(KL[2]), 
#                               expression(CRPS), expression(JSD)),
#        col = colors, lty = 1, lwd = 2, cex = 1.2, box.lty = 0)
# dev.off()
# cat("Plot saved to", output_file, "for seed:", seed, "\n")
# }

# # seeds <- c(111, 300, 666, 888)
# seeds <- c(111, 300, 666, 3000916, 888891678)

# lapply(seeds, run_simulation)

In [ ]:
# Optimization has finished for seed: 111 
# Optim: 0.9999995 0.9996394 

# Optimization has finished for seed: 300 
# Optim: 0.9999993 0.9989743 

# Optimization has finished for seed: 666 
# Optim: 0.9999994 0.9998615 

# Optimization has finished for seed: 888 
# Optim: 0.9999992 0.9849209 

In [ ]:
# f_obj(optimal_w)
# f_obj(delta)
# optimal_w

## Check tis function works as intended!

## modify estimate_kl_divergence?
## Imp! Not compute the score of the expected but the expected score.

## Ability to recover lambda? 

# Ex AL

In [ ]:
# Load necessary library
library(ggplot2)

seed <- 111

# Define parameters
TT <- 1200
B  <- 5000
T  <- TT + B - 1
trend_degree <- 1
harmonics    <- c(1)
period       <- 12
transfer_rate  <- 0.995
observation_sd <- 1
evolution_sd   <- 1.0e-70

# Set seed for reproducibility
set.seed(seed)

# Generate covariates
covariates <- exp(matrix(rnorm(T * 1, sd = 4), ncol = 1))  # 1 random covariate

# Generate matrices
dlm_matrices <- set_matrices(
trend_degree = trend_degree,
harmonics = harmonics, 
period = period,
covariates = covariates,
covariate_mode = "transfer",
transfer_rate = transfer_rate,
T = T
)

# Initial state
initial_state <- c(-1000, 500, 100000, 0, 5, 2)/100000*observation_sd

# Simulate the model
simulation_results <- simulate_dlm(
G = dlm_matrices$G,
F = dlm_matrices$F,
T = T,
initial_state = initial_state,
observation_sd = observation_sd,
evolution_sd = evolution_sd
)

# Extract results
y <- simulation_results$y[B:T]
theta <- simulation_results$theta[B:T, ]
theta0 <- simulation_results$theta[B-1, ]

# Save observation plot
# png(filename = paste0("simulated_observations_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
plot(y, type = "l", col = "black", lwd = 0.5, xlab = "Time", ylab = "Observation", 
   main = paste("Simulated Observations (Seed:", seed, ")"))
points(y, pch = 19, col = "black", cex = 0.5)
# dev.off()

# Save state components plot
# png(filename = paste0("simulated_states_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
matplot(theta, type = "l", lty = 1, lwd = 1.5, xlab = "Time", ylab = "State Components", 
      main = paste("Simulated State Components (Seed:", seed, ")"))
legend("topright", legend = paste0("State ", 1:ncol(theta)), col = 1:ncol(theta), lty = 1, cex = 0.8)
# dev.off()

# Set up model parameters
p = 4
ppx = 2
J = 0

nu0 <- rep(4, 1)  # Degrees of freedom (>2 for well-defined mean)
S0 <- nu0 * rep(observation_sd, 1)  # Scale (high values imply weak prior knowledge)
m0 <- theta0
C0 <- diag(rep(1/ (p + ppx), ncol(theta)))  # Large variance for vague prior
ones <- matrix(1, ncol = (ncol(theta)), nrow = (ncol(theta)))

# Create degree of freedom matrices
df = 1 - 1e-10
dim.df = c(6)
k <- 2
ex_df_mat <- make_df_mat(df, dim.df, p + ppx + J)
ex_df_mat_k <- make_df_mat_k(df, dim.df, p + ppx + J, k)
#########################################################################
GG <- (dlm_matrices$G[,,B:T])
FF <- (dlm_matrices$F[,,B:T])
FF <- array(FF, dim = c(dim(FF)[1], 1, dim(FF)[2]))
model$GG <- GG 
model$FF <- FF
#########################################################################
y <- matrix(y, nrow =1)
Rcpp::sourceCpp('/data/muscat_data/jaguir26/project1_ucsc_phd/kalman.cpp')
print('done')

In [ ]:
#   # Load Kalman filter function
#   Rcpp::sourceCpp('/data/muscat_data/jaguir26/project1_ucsc_phd/kalman_NDLM.cpp')

#   # Fit model
#   fit_results <- update_theta_cpp_ndlm_exactV(
#     GG = dlm_matrices$G[,,B:T, drop = FALSE],
#     m0 = m0,
#     C0 = C0,
#     FF = dlm_matrices$F[,,B:T, drop = FALSE],
#     y = matrix(y, nrow = 1),
#     ex_df_mat = ex_df_mat,
#     ex_df_mat_k = ex_df_mat_k,
#     Ones = ones,
#     p = 4,
#     J = 0,
#     ppx = 1 + 1,
#     TT = TT,
#     k = k,
#     dM = 0,
#     nu0 = nu0,
#     S0 = S0
#   )
  
#   update.theta_exact <- fit_results
  
  # # Compute expectations
  # FF_t <- aperm(dlm_matrices$F[,,B:T, drop = FALSE], c(2, 1, 3))
  
#   compute_exp <- function(slice_index) {
#     FF_t[,,slice_index] %*% update.theta_exact$sm[,slice_index]
#   }
  
#   result_list <- lapply(1:ncol(update.theta_exact$sm), compute_exp)
#   result_array <- array(unlist(result_list), dim = c(J + 1, 1, ncol(update.theta_exact$sm)))
#   result_array <- aperm(result_array, c(1, 3, 2))[,,1]
#   exps <- result_array
  
#   # Compute variances
#   compute_var <- function(t) {
#     FF_t_slice <- FF_t[,,t]
#     sC_slice <- update.theta_exact$sC[,,t]
#     ((FF_t_slice) %*% sC_slice) %*% (FF_t_slice)
#   }
  
#   result_list_1 <- lapply(1:ncol(update.theta_exact$sm), compute_var)
#   vars_1 <- simplify2array(result_list_1)
  
#   S_TT <- c(update.theta_exact$S_t)
#   n_TT <- c(update.theta_exact$nu_t)
  
#   # Save prediction plot with filled credible intervals
#   # png(filename = paste0("prediction_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
#   plot(y, type = "l", col = "black", lwd = 0.5, xlab = "Time", ylab = "Observation", 
#        main = paste("Prediction (Seed:", seed, ")"))
#   points(y, pch = 19, col = "black", cex = 0.5)
  
#   ci_upper <- exps + sqrt(S_TT / n_TT) * qnorm(0.975)
#   ci_lower <- exps + sqrt(S_TT / n_TT) * qnorm(0.025)
  
#   polygon(c(seq_along(y), rev(seq_along(y))), c(ci_upper, rev(ci_lower)), 
#           col = adjustcolor("lightblue", alpha.f = 0.5), border = NA)
  
#   lines(exps, col = 'darkorange', lwd = 0.8)
#   # dev.off()

#   # Save state estimates plot
#   plot_state <- function(index) {
#     mean_estimate <- update.theta_exact$sm[index,]
#     ci_upper <- mean_estimate + sqrt(update.theta_exact$sC[index, index,]) * qnorm(0.975)
#     ci_lower <- mean_estimate + sqrt(update.theta_exact$sC[index, index,]) * qnorm(0.025)
    
#     time_points <- seq_along(mean_estimate)
    
#     # png(filename = paste0("state_", index, "_seed_", seed, ".png"), width = 2000, height = 1400, res = 300)
#     plot(time_points, mean_estimate, type = "l", col = 'darkblue', ylim = range(theta[, index]), lwd = 1.5,
#          main = paste("State Variable", index, "(Seed:", seed, ")"), ylab = "Estimate", xlab = "Time")
#     polygon(c(time_points, rev(time_points)), c(ci_upper, rev(ci_lower)), 
#             col = adjustcolor("lightblue", alpha.f = 0.5), border = NA)
#     lines(time_points, mean_estimate, col = 'darkblue', lwd = 1.5)
#     points(time_points, theta[, index], col = 'red', pch = 19, cex = 0.5)
#     lines(time_points, theta[, index], col = 'red', lwd = 0.5)
#     # dev.off()
#   }

#   lapply(c(1, 2, 3, 5, 6), plot_state)
  

In [ ]:
ps <- c(0.05, 0.5,0.95)

In [ ]:
# # for(p0 in ps){

# for(i in 1:length(ps) ){
    
#     p0 <- as.numeric(ps[i])

#     Y <- y
#     TT <- dim(Y)[2]
#     TT_sub <- TT
#     J <- dim(Y)[1] - 1
#     timestamps <- plot_data[, 'time']
#     idx <- 1:TT
#     y <- Y[,idx]
#     if (is.null(nrow(y))) {
#       JJJ <- 1
#       y <- array(y, c(JJJ, length(y)))
#     } else {
#       JJJ <- nrow(Y)
#       y <- array(y, c(JJJ, ncol(y)))
#     }
#     gam.init <- array(rep(0, JJJ), c(JJJ, 1))
#     sig.init <- array(rep(1, JJJ), c(JJJ, 1))
#     PriorSigma <- array(NA_real_, c(JJJ, 2))
#     PriorGamma <- array(NA_real_, c(JJJ, 3))
#     L = L.fn(p0)
#     U = U.fn(p0)
#     ###########################################################################################
#     ########### For every j
#     for (j in 1:(J+1)) {
#       if (!is.na(gam.init[j,])) {
#         if (gam.init[j,] < L | gam.init[j,] > U) {
#           stop(sprintf("gam.init must be between %s and %s for %s quantile", 
#                         round(L, 3), round(U, 3), p0))
#         }
#       } 
#     }
#     ###########################################################################################
#     ########### For every j
#     for (j in 1:(J+1)) {
#       if (is.na(PriorSigma[j,1]) || is.na(PriorSigma[j,2])) {
#         m_sigma = 1
#         v_sigma = 1
#         PriorSigma[j,1] = (m_sigma^2)/(v_sigma) + 2 
#         PriorSigma[j,2] = (m_sigma^3)/(v_sigma) + m_sigma 
#       }
#     }
#     ###########################################################################################
#     ########### For every j
#     for (j in 1:(J+1)) {
#       if (is.na(PriorGamma[j,1]) || is.na(PriorGamma[j,2]) || is.na(PriorGamma[j,3])) {
#         PriorGamma[j,1]  = 0
#         PriorGamma[j,2]  = 1
#         PriorGamma[j,3] = 1
#       }
#     }
#     ###########################################################################################
#     ########### For every j
#     gam0 = gam.init 
#     sig0 = sig.init 
#     ###########################################################################################
#     ########### For every j 
#     E1 <- array(NA_real_, c(J+1,1))
#     E1[,] <- 0.1
#     E2 <- array(NA_real_, c(J+1,1))
#     E2[,] <- 0.1
#     new.gamsig.out = list(E.gam = gam0,
#                           V.gam = E1, 
#                           E.sigma = sig0, 
#                           V.sig = E2,
#                           E.inv.sigma = 1/sig0, 
#                           E.c2.invb.absgam2.sigma = sig0 * (C.fn(p0, gam0)^2) * (abs(gam0)^2)/B.fn(p0, gam0), 
#                           E.c.invb.absgam = C.fn(p0, gam0) * abs(gam0)/B.fn(p0, gam0),  
#                           E.c.a.invb.absgam = C.fn(p0, gam0) * A.fn(p0, gam0) * abs(gam0)/B.fn(p0, gam0), 
#                           E.a2.invb.inv.sigma = (A.fn(p0,gam0)^2)/(B.fn(p0, gam0) * sig0), 
#                           E.invb.inv.sigma = 1/(sig0 * B.fn(p0, gam0)), 
#                           E.a.invb.inv.sigma = A.fn(p0, gam0)/(B.fn(p0, gam0) * sig0),
#                           E.log.sig.b = log( sig0*B.fn(p0, gam0) ),
#                           E.log.sig = log(sig0),
#                           E.prior.sig.gam = array(0, c(J+1,1)),
#                           entrop = array(0, c(J+1,1))  )
#     ###########################################################################################
#     ########### For every j
#     E1 <- array(NA_real_, c(J+1,TT_sub))
#     E1[,] <- truncnorm::etruncnorm(a = 0, b = Inf,  mean = 0.1, sd = 0.0)
#     E2 <- array(NA_real_, c(J+1,TT_sub))
#     E2[,] <- E1[,]^2 
#     new.sts.out = list(E.sts = E1, 
#                         E.sts2 = E2,
#                         tot.entrop = array(0, c(J+1,1)) )
#     ###########################################################################################
#     ########### For every j
#     E1 <- array(NA_real_, c(J+1,TT_sub))
#     E1[,] <- 1/sig0
#     E2 <- array(NA_real_, c(J+1,TT_sub))
#     E2[,] <- sig0
#     new.uts.out = list(E.uts = E1, 
#                         E.inv.uts = E2,
#                         E.log.uts = array(0, c(J+1,1)),
#                         tot.entrop = array(0, c(J+1,1)) )
#     ###########################################################################################
#     ########### For every j
#     init.dlm = dlm_df(colMeans(y), model_simp, df_simp, dim.df_simp, 
#                       s.priors = list(l0 = 1, S0 = mean(sig0)), 
#                       just.lik = FALSE)
#     FF_t <- aperm(model_simp$FF, c(2, 1, 3))
#     multiply_matrices <- function(slice_index) {
#       t(FF_t[1,,slice_index]) %*% init.dlm$m[slice_index,]
#     }
#     result_list <- lapply(1:TT_sub, multiply_matrices)
#     result_array <- array(unlist(result_list), dim = c(TT_sub,1))
#     exps0 = c(result_array) + stats::qnorm(p0, 0, sqrt(init.dlm$s[TT_sub]))
#     exps0 = t(replicate(J+1, exps0))
#     new.theta.out = list(exps = exps0, 
#                           exps2 = (exps0)^2)
#     ###########################################################################################
#     iter = 0
#     conv.count = 0
#     new.max = Inf
#     ###########################################################################################
#     ########### For every j
#     seq.gamma = new.gamsig.out$E.gam
#     seq.sigma = new.gamsig.out$E.sigma
#     ###########################################################################################
#     update_sts<-function(y, exps,inv.uts,c2.invb.absgam2.sigma,c.invb.absgam,c.a.invb.absgam){
#       s.sig2<-1/(1+c2.invb.absgam2.sigma*inv.uts); s.sig = sqrt(s.sig2)
#       s.mu<-s.sig2*(c.invb.absgam*(y-exps)*inv.uts-c.a.invb.absgam)
#       #
#       E.sts = truncnorm::etruncnorm(a=rep(0,TT_sub),b=rep(Inf,TT_sub),mean=s.mu,sd=s.sig)
#       V.sts = truncnorm::vtruncnorm(a=rep(0,TT_sub),b=rep(Inf,TT_sub),mean=s.mu,sd=s.sig)
#       E.sts2 = s.mu^2 + s.sig2 + s.mu*s.sig*exp(stats::dnorm(-s.mu/s.sig,log = TRUE)-stats::pnorm(s.mu/s.sig,log.p = TRUE))
#       return(list(sts.sig2=s.sig2,sts.mu=s.mu,
#                   E.sts=E.sts,E.sts2=E.sts2,
#                   tot.entrop = sum(0.5*log2(2*pi*exp(1)*s.sig2) - 1 )))
#     }

#     Kprime <- function(x){
#     sqrt(pi/2/x) * expint_E1(2*x) * exp(x)
#     }

#     gig_entrop <- function(a,b){
#     nu <- 0.5
#     s.ab <- sqrt(a*b)
#     K1 <- besselK(s.ab, nu)
#     K2 <- besselK(s.ab, nu+1)
#     K3 <- besselK(s.ab, nu-1)
#     y <- 0.5*log(b/a) + log(2*K1) - (nu-1)*Kprime(s.ab)/K1 + s.ab/2/K1*(K2 + K3)
#     return(y)
#     }
#     ###########################################################################################
#     update_uts<-function(y, exps,exps2,sts,sts2,inv.sigma,a2.invb.inv.sigma,invb.inv.sigma,c.invb.absgam,c2.invb.absgam2.sigma){
#       u.lambda = 0.5
#       u.psi = (a2.invb.inv.sigma + 2*inv.sigma)
#       u.chi = invb.inv.sigma*(y^2-2*y*exps+exps2) - 2*c.invb.absgam*sts*(y-exps) + c2.invb.absgam2.sigma*sts2
#       u.chi[u.chi<=0] = 1e-12
#       #
#       E.uts = sapply(u.chi,function(x){sqrt(x/u.psi)*HyperbolicDist::besselRatio(sqrt(x*u.psi),u.lambda,1,Inf)})
#       E.inv.uts = sapply(u.chi,function(x){sqrt(u.psi/x)*HyperbolicDist::besselRatio(sqrt(x*u.psi),u.lambda,1,Inf)-2*u.lambda/x})

#     nu <- 0.5
#     s.ab <- sqrt(u.psi*u.chi)
#     K1 <- besselK(s.ab, nu)

#       return(list(uts.lambda=u.lambda,
#                   uts.psi=u.psi,uts.chi=u.chi,
#                   E.uts=E.uts,E.inv.uts=E.inv.uts,
#                   E.log.uts=sum(Kprime(s.ab)/K1-0.5*log(u.psi/u.chi)),
#                   tot.entrop=sum(gig_entrop(u.psi,u.chi))))
#     }
#     ###########################################################################################
#     ########################
#     PriorGammaDens <- function(gamma, prior) {
#       crch::dtt(gamma, 
#                 location = prior[1], 
#                 scale = prior[2],   
#                 df = prior[3], 
#                 left = L, right = U, 
#                 log = FALSE)
#     }
#     LL <- L+0.001
#     UU <- U-0.001
#     update_gamma_sigma<-function(y, nn, prior_g, prior_s, gamma,var.gam,sigma,var.sig,exps,exps2,sts,sts2,uts,inv.uts, s_init, g_init){
#       #############################################################################################################################################

#       dq_transf <- function(theta_s,theta_g){
#           sig <- exp(theta_s)
#           gam <- LL+(-LL+UU)*exp(-exp(theta_g))
#               a = A.fn(p0,gam); b = B.fn(p0,gam); c = C.fn(p0,gam); p.fn(p0,gam)

#           yy <- log(PriorGammaDens(gam, prior_g)) - (prior_s[1] + 1) * log(sig) - prior_s[2]/sig
#           yy <- yy - (1.5*nn)*log(sig) - (0.5*nn)*log(b)-sum(uts)/sig -
#                   0.5*sum( inv.uts*(y^2-2*y*exps+exps2)/sig
#                           + (exps-y)*2*(inv.uts*c*abs(gam)*sts + a/sig)
#                           + sig*inv.uts*(c^2)*(abs(gam)^2)*sts2
#                           + 2*c*abs(gam)*sts*a
#                           + (uts*a^2)/sig )/b
#           yy <- yy + theta_s + theta_g - exp(theta_g)                   
#           return(yy)
#       }

#       theta_s_init <- log(s_init)
#       theta_g_init <- log(log((-L+U)/(-L+g_init)))
#       initial_values <- c(theta_s_init, theta_g_init)

#       # Optimization step
#       optim_results <- optim(par = initial_values, 
#                           fn = function(x) -dq_transf(x[1], x[2]), # Maximizing by minimizing the negative
#                           method = "L-BFGS-B", # This method allows box constraints
#                           lower = c(-Inf, -Inf), # Transform bounds for gam to theta_g space if needed
#                           upper = c(Inf, Inf),
#                           hessian = TRUE)
#       # Evaluate the Hessian at the optimal value
#       hessian_at_optimal <- -optim_results$hessian # SINCE WE MIN -f, not MAX f
#       # Take the inverse of the Hessian
#       inverse_hessian <- solve(hessian_at_optimal)

#       LD_mu <- optim_results$par
#       LD_S <- -inverse_hessian 

#       Expected_f <- function(f, theta_s, theta_g){
#           x <- hessian(func = f, x = LD_mu)%*%LD_S
#           e <- f(LD_mu) + 0.5*sum(diag(x))
#         return(e)
#       }

#       f.exp.theta_g <- function(theta){
#         sig = exp(theta[1]); gam = LL+(-LL+UU)*exp(-exp(theta[2]));
#         a = A.fn(p0,gam); b = B.fn(p0,gam); c = C.fn(p0,gam);
#         yy <- exp(theta[2])
#         return(yy)
#       }

#       f.log.sig.b <- function(theta){
#         sig = exp(theta[1]); gam = LL+(-LL+UU)*exp(-exp(theta[2]));
#         a = A.fn(p0,gam); b = B.fn(p0,gam); c = C.fn(p0,gam);
#         yy <- log(sig*b)
#         return(yy)
#       }

#       f.log.sig <- function(theta){
#         sig = exp(theta[1]); gam = LL+(-LL+UU)*exp(-exp(theta[2]));
#         a = A.fn(p0,gam); b = B.fn(p0,gam); c = C.fn(p0,gam);
#         yy <- log(sig)
#         return(yy)
#       }

#       f.prior.sig.gam <- function(theta){
#         sig = exp(theta[1]); gam = LL+(-LL+UU)*exp(-exp(theta[2]));
#         a = A.fn(p0,gam); b = B.fn(p0,gam); c = C.fn(p0,gam);
#         yy <- crch::dtt(gam, location = prior_g[1], scale = prior_g[2], df = prior_g[3], left = L, right = U, log = TRUE)
#         yy <- yy + nimble::dinvgamma(sig, shape = prior_s[1], scale =  prior_s[2], log = TRUE)
#         return(yy)
#       }


#       f.c2.s.abs.g2.inv.b <- function(theta){
#         sig = exp(theta[1]); gam = LL+(-LL+UU)*exp(-exp(theta[2]));
#         a = A.fn(p0,gam); b = B.fn(p0,gam); c = C.fn(p0,gam);
#         yy <- c^2*sig*abs(gam)^2/b
#         return(yy)
#       }

#       f.inv.sig <- function(theta){
#         sig = exp(theta[1])
#         yy <- 1/sig
#         return(yy)
#       }

#       f.c.abs.g.inv.b <- function(theta){
#         gam = LL+(-LL+UU)*exp(-exp(theta[2]))
#         b = B.fn(p0,gam); c = C.fn(p0,gam);
#         yy <- c*abs(gam)/b
#         return(yy)
#       }

#       f.c.abs.g.a.inv.b <- function(theta){
#         sig = exp(theta[1]); gam = LL+(-LL+UU)*exp(-exp(theta[2]));
#         a = A.fn(p0,gam); b = B.fn(p0,gam); c = C.fn(p0,gam);
#         yy <- c*abs(gam)*a/b
#         return(yy)
#       }

#       f.inv.s.inv.b <- function(theta){
#         sig = exp(theta[1]); gam = LL+(-LL+UU)*exp(-exp(theta[2]));
#         a = A.fn(p0,gam); b = B.fn(p0,gam); c = C.fn(p0,gam);
#         yy <- 1/sig/b
#         return(yy)
#       }

#       f.a.inv.s.inv.b <- function(theta){
#         sig = exp(theta[1]); gam = LL+(-LL+UU)*exp(-exp(theta[2]));
#         a = A.fn(p0,gam); b = B.fn(p0,gam); c = C.fn(p0,gam);
#         yy <- a/sig/b
#         return(yy)
#       }

#       f.a2.inv.s.inv.b <- function(theta){
#         sig = exp(theta[1]); gam = LL+(-LL+UU)*exp(-exp(theta[2]));
#         a = A.fn(p0,gam); b = B.fn(p0,gam); c = C.fn(p0,gam);
#         yy <- a^2/sig/b
#         return(yy)
#       }

#       f.sig <- function(theta){
#         sig = exp(theta[1]); 
#         yy <- sig
#         return(yy)
#       }

#       f.gam <- function(theta){
#         gam = LL+(-LL+UU)*exp(-exp(theta[2]));
#         yy <- gam
#         return(yy)
#       }

#       #############################################################################################################################################


#       E.sig = Expected_f(f.sig, LD_mu[1], LD_mu[2]);
#       E.gam = Expected_f(f.gam, LD_mu[1], LD_mu[2]);


#       E.inv.sigma = Expected_f(f.inv.sig, LD_mu[1], LD_mu[2])
#       E.c2.invb.absgam2.sigma = Expected_f(f.c2.s.abs.g2.inv.b, LD_mu[1], LD_mu[2])
#       E.c.invb.absgam = Expected_f(f.c.abs.g.inv.b, LD_mu[1], LD_mu[2])
#       E.c.a.invb.absgam = Expected_f(f.c.abs.g.a.inv.b, LD_mu[1], LD_mu[2])
#       E.a2.invb.inv.sigma = Expected_f(f.a2.inv.s.inv.b, LD_mu[1], LD_mu[2])
#       E.invb.inv.sigma = Expected_f(f.inv.s.inv.b, LD_mu[1], LD_mu[2])
#       E.a.invb.inv.sigma = Expected_f(f.a.inv.s.inv.b, LD_mu[1], LD_mu[2])
#       E.log.sig.b = Expected_f(f.log.sig.b, LD_mu[1], LD_mu[2])
#       E.log.sig = Expected_f(f.log.sig, LD_mu[1], LD_mu[2])
#       E.prior.sig.gam = Expected_f(f.prior.sig.gam, LD_mu[1], LD_mu[2])
#       E.exp.theta_g =  Expected_f(f.exp.theta_g, LD_mu[1], LD_mu[2])

#       entrop <- log(2*pi*exp(1)) + 0.5*determinant(as.matrix(LD_S), logarithm = TRUE)$modulus[1]-(log(-LL+UU)+sum(LD_mu)-E.exp.theta_g)

#       return(list(E.sigma=E.sig,E.inv.sigma=E.inv.sigma,E.gam=E.gam,
#                   E.c2.invb.absgam2.sigma = E.c2.invb.absgam2.sigma, E.c.invb.absgam = E.c.invb.absgam,
#                   E.c.a.invb.absgam = E.c.a.invb.absgam, E.a2.invb.inv.sigma = E.a2.invb.inv.sigma,
#                   E.invb.inv.sigma = E.invb.inv.sigma, E.a.invb.inv.sigma = E.a.invb.inv.sigma,
#                   Hess.LD = LD_S,
#                   E.log.sig.b=E.log.sig.b, 
#                   E.log.sig = E.log.sig, 
#                   E.prior.sig.gam= E.prior.sig.gam,
#                   E.theta = LD_mu,
#                   entrop = entrop))
#     }
#     Ones <- matrix(1, dim(model$GG)[1], dim(model$GG)[1])
#     ex.df.mat <- as.matrix(ex_df_mat)
#     ex.df.mat.k <- as.matrix(ex_df_mat_k)
#     crit_ELBO <- 0
#     ELBO <- 0
#     seq.elbo = ELBO
#     iter = 0
#     FLAG = TRUE
#     crit_ELBO <- 0
#     ELBO <- 0
#     seq.elbo = ELBO
#     iter = 0
#     FLAG = TRUE
#     tol1 <- 1e-1
#     tol2 <- 1e-3
#     conv.check <- 0
#     max_iter <- 2000
#     ########################
#       tictoc::tic("run time")

#       ########################
#       while (FLAG & iter < max_iter) {
#         cur.uts.out = new.uts.out
#         cur.sts.out = new.sts.out
#         cur.gamsig.out = new.gamsig.out
#         cur.theta.out = new.theta.out
#         FFF <- (new.gamsig.out$E.c.invb.absgam[,] * new.sts.out$E.sts + new.gamsig.out$E.a.invb.inv.sigma[,]/new.uts.out$E.inv.uts) / new.gamsig.out$E.invb.inv.sigma[,] 
#         QQQ <- 1/(new.gamsig.out$E.invb.inv.sigma[,] * new.uts.out$E.inv.uts)
#         if(J>0){
#         QQQ <- array(apply(QQQ, 2, function(col) diag(col)), dim = c(J+1, J+1, TT))
#         }else{
#          QQQ <- array(QQQ, dim = c(J+1, J+1, TT))
#         }

#         if ((crit_ELBO+conv.check) < tol1) {
#         update.theta <- update_theta_cpp(GG, m0, C0, FFF, QQQ, FF, y, ex.df.mat, ex.df.mat.k, Ones, p, J, ppx, TT, k, dM)
#         FF_t <- aperm(FF, c(2, 1, 3))
#         multiply_matrices <- function(slice_index) {
#           FF_t[,,slice_index] %*% update.theta$sm[,slice_index]
#         }
#         result_list <- lapply(1:ncol(update.theta$sm), multiply_matrices)
#         result_array <- array(unlist(result_list), dim = c(J+1, 1, ncol(update.theta$sm)))
#         result_array <- aperm(result_array, c(1, 3, 2))[,,1]
#         exps <- result_array
#         compute_product_1 <- function(t) {
#           FF_t_slice <- FF_t[,,t]
#           sC_slice <- update.theta$sC[,,t]
#           FF_slice <- FF[,,t]
#           result_slice <- t(FF_slice)%*%sC_slice%*%(FF_slice )
#           return(result_slice)
#         }
#         result_list_1 <- lapply(1:dim(FF)[3], compute_product_1)
#         vars_1 <- simplify2array(result_list_1)
#        if(J>0){
#         vars <- (apply(vars_1, 3, function(x) diag(x)))
#         exps2 = exps^2 + vars
#         }else{
#         exps2 = exps^2 + vars_1
#         vars_1 <- array( vars_1, c(1,TT) )  
#         exps2 <- array( exps2, c(1,TT) )  
#         exps <- array( exps, c(1,TT) )    
#         }
#           new.theta.out  <- update.theta 
#           new.theta.out$exps  <- exps
#           new.theta.out$exps2  <- exps2
#           new.theta.out$vars  <- vars

#           theta_update <- TRUE
#         }else{
#           theta_update <- FALSE
#         }
#         for (j in 1:(J+1)) {   
#           sts.dummy <- update_sts(y[j,],
#                                   new.theta.out$exps[j,], 
#                                   cur.uts.out$E.inv.uts[j,], 
#                                   cur.gamsig.out$E.c2.invb.absgam2.sigma[j,], 
#                                   cur.gamsig.out$E.c.invb.absgam[j,], 
#                                   cur.gamsig.out$E.c.a.invb.absgam[j,])
#           new.sts.out$E.sts[j,] <- sts.dummy$E.sts
#           new.sts.out$E.sts2[j,] <- sts.dummy$E.sts2
#           new.sts.out$tot.entrop[j,] <-  sts.dummy$tot.entrop
#           ########################
#           uts.dummy <- update_uts(y[j,],
#                                   new.theta.out$exps[j,], 
#                                   new.theta.out$exps2[j,], 
#                                   new.sts.out$E.sts[j,], 
#                                   new.sts.out$E.sts2[j,], 
#                                   cur.gamsig.out$E.inv.sigma[j,], 
#                                   cur.gamsig.out$E.a2.invb.inv.sigma[j,], 
#                                   cur.gamsig.out$E.invb.inv.sigma[j,], 
#                                   cur.gamsig.out$E.c.invb.absgam[j,], 
#                                   cur.gamsig.out$E.c2.invb.absgam2.sigma[j,]) 
#           new.uts.out$E.uts[j,] <- uts.dummy$E.uts
#           new.uts.out$E.inv.uts[j,] <- uts.dummy$E.inv.uts
#           new.uts.out$E.log.uts[j,] <- uts.dummy$E.log.uts
#           new.uts.out$tot.entrop[j,] <- uts.dummy$tot.entrop
#           ########################
#          gamsig.dummy <- update_gamma_sigma(y[j,], TT,
#                                              PriorGamma[j,],
#                                              PriorSigma[j,],
#                                              cur.gamsig.out$E.gam[j,], 
#                                              cur.gamsig.out$V.gam[j,], 
#                                              cur.gamsig.out$E.sigma[j,], 
#                                              cur.gamsig.out$V.sigma[j,], 
#                                              new.theta.out$exps[j,], 
#                                              new.theta.out$exps2[j,], 
#                                              new.sts.out$E.sts[j,], 
#                                              new.sts.out$E.sts2[j,], 
#                                              new.uts.out$E.uts[j,], 
#                                              new.uts.out$E.inv.uts[j,],
#                                              cur.gamsig.out$E.sigma[j,], 
#                                              cur.gamsig.out$E.gam[j,])    
#           new.gamsig.out$E.gam[j,] <- gamsig.dummy$E.gam
#           new.gamsig.out$E.sigma[j,] <- gamsig.dummy$E.sigma
#           new.gamsig.out$E.inv.sigma[j,] <- gamsig.dummy$E.inv.sigma
#           new.gamsig.out$E.c2.invb.absgam2.sigma[j,] <- gamsig.dummy$E.c2.invb.absgam2.sigma
#           new.gamsig.out$E.c.invb.absgam[j,] <- gamsig.dummy$E.c.invb.absgam
#           new.gamsig.out$E.c.a.invb.absgam[j,] <- gamsig.dummy$E.c.a.invb.absgam
#           new.gamsig.out$E.a2.invb.inv.sigma[j,] <- gamsig.dummy$E.a2.invb.inv.sigma
#           new.gamsig.out$E.invb.inv.sigma[j,] <- gamsig.dummy$E.invb.inv.sigma
#           new.gamsig.out$E.a.invb.inv.sigma[j,] <- gamsig.dummy$E.a.invb.inv.sigma
#           new.gamsig.out$E.log.sig.b[j,] <- gamsig.dummy$E.log.sig.b
#           new.gamsig.out$E.log.sig[j,] <- gamsig.dummy$E.log.sig
#           new.gamsig.out$E.prior.sig.gam[j,] <- gamsig.dummy$E.prior.sig.gam
#           new.gamsig.out$entrop[j,] <- gamsig.dummy$entrop
#         }
#         ##########
#         old.gam = seq.gamma[,dim(seq.gamma)[2]]
#         new.gam = new.gamsig.out$E.gam
#         seq.gamma = cbind(seq.gamma, new.gam)

#         old.sig = seq.sigma[,dim(seq.sigma)[2]]
#         new.sig = new.gamsig.out$E.sigma
#         seq.sigma = cbind(seq.sigma, new.sig)

#         conv.check <- sum(old.gam-new.gam)^2 + sum(old.sig-new.sig)^2
#         ##########
#         # ELBO
#         ##########
#         elbo <- 0
#         elbo <- elbo -TT/2*sum(new.gamsig.out$E.log.sig.b[,])
#         elbo <- elbo -0.5*sum(new.uts.out$E.log.uts[,])
#         elbo <- elbo -TT*(J+1)/2*log(pi)
#         elbo <- elbo -0.5*sum((new.gamsig.out$E.invb.inv.sigma[,]*new.uts.out$E.inv.uts[,])*(y[,]^2-2*y[,]*cur.theta.out$exps[,]+new.theta.out$exps2[,]))
#         elbo <- elbo +sum((y[,]-new.theta.out$exps[,])*(new.gamsig.out$E.c.invb.absgam[,]*new.sts.out$E.sts*new.uts.out$E.inv.uts[,]+new.gamsig.out$E.a.invb.inv.sigma[,]))-0.5*sum(new.sts.out$E.sts2[,]*new.uts.out$E.inv.uts[,]*new.gamsig.out$E.c2.invb.absgam2.sigma[,])
#         elbo <- elbo -sum(new.gamsig.out$E.c.a.invb.absgam[,]*new.sts.out$E.sts[,])-0.5*sum(new.gamsig.out$E.a2.invb.inv.sigma[,]*new.uts.out$E.uts[,])
#         elbo <- elbo -TT*sum(new.gamsig.out$E.log.sig[,])-sum(new.gamsig.out$E.inv.sigma[,]*new.uts.out$E.uts[,])-0.5*sum(new.sts.out$E.sts2[,])+sum(new.gamsig.out$E.prior.sig.gam[,])
#         elbo <- elbo +sum(new.uts.out$tot.entrop[,])+sum(new.sts.out$E.tot.entrop[,])+sum(new.gamsig.out$E.sig.gam.entrop[,])
#         elbo <- elbo + new.theta.out$elbo.part
#     ######################
#         elbo <- elbo/TT/(J+1)
#         crit_ELBO <- abs(ELBO-elbo)
#         ELBO <- elbo
#         seq.elbo =  cbind(seq.elbo, ELBO) 

#         # print(c(elbo, crit_ELBO, conv.check))
#         # print(c(new.gamsig.out$E.gam[1,],new.gamsig.out$E.sigma[1,]))
#         # flush.console()
#         iter = iter + 1

#         if(theta_update){
#           if ((crit_ELBO+conv.check) < tol2) {
#             FLAG = FALSE
#           }
#         }

#       }
#     ########################
#     run.time = tictoc::toc(quiet = TRUE)
#     ########################
#     if (verbose) {
#       cat(sprintf("VB converged: %s iterations, %s seconds", 
#                   iter, round(run.time$toc - run.time$tic, 3)), "\n")
#     }

#     a <- 50
#     plot(a:length(seq.elbo[1,]),seq.elbo[1,a:length(seq.elbo[1,])], lty = 4)
#     lines(a:length(seq.elbo[1,]),seq.elbo[1,a:length(seq.elbo[1,])])
#     plot(a:length(seq.elbo[1,]),seq.sigma[1,a:length(seq.elbo[1,])], lty = 4)
#     lines(a:length(seq.elbo[1,]),seq.sigma[1,a:length(seq.elbo[1,])])
#     plot(a:length(seq.elbo[1,]),seq.gamma[1,a:length(seq.elbo[1,])], lty = 4)
#     lines(a:length(seq.elbo[1,]),seq.gamma[1,a:length(seq.elbo[1,])])
#     #########################################################################
#     Rcpp::sourceCpp('/data/muscat_data/jaguir26/project1_ucsc_phd/kalman.cpp')
#     print('done')
#     flush.console()
#     #########################################################################
#     update.theta_exact <- update.theta
#     FF_t <- aperm(dlm_matrices$F[,,B:T, drop = FALSE], c(2, 1, 3))
#     compute_exp <- function(slice_index) {
#     FF_t[,,slice_index] %*% update.theta_exact$sm[,slice_index]
#     }
#     result_list <- lapply(1:ncol(update.theta_exact$sm), compute_exp)
#     result_array <- array(unlist(result_list), dim = c(J + 1, 1, ncol(update.theta_exact$sm)))
#     result_array <- aperm(result_array, c(1, 3, 2))[,,1]
#     exps <- result_array
#     compute_var <- function(t) {
#     FF_t_slice <- FF_t[,,t]
#     sC_slice <- update.theta_exact$sC[,,t]
#     ((FF_t_slice) %*% sC_slice) %*% (FF_t_slice)
#     }
#     result_list_1 <- lapply(1:ncol(update.theta_exact$sm), compute_var)
#     vars_1 <- simplify2array(result_list_1)
#     #########################################################################
#     plot(y[1,], type = "l", col = "black", lwd = 0.5, xlab = "Time", ylab = "Observation", 
#        main = paste("Prediction (Seed:", seed, ")"))
#     points(y[1,], pch = 19, col = "black", cex = 0.5)
#     ci_upper <- exps + new.gamsig.out$E.sigma[1,] * qnorm(0.975)
#     ci_lower <- exps + new.gamsig.out$E.sigma[1,] * qnorm(0.025)
#     polygon(c(seq_along(y), rev(seq_along(y))), c(ci_upper, rev(ci_lower)), 
#           col = adjustcolor("lightblue", alpha.f = 0.5), border = NA)

#     lines(exps, col = 'darkorange', lwd = 0.8)
#     plot_state <- function(index) {
#     mean_estimate <- update.theta_exact$sm[index,]
#     ci_upper <- mean_estimate + (new.gamsig.out$E.sigma[1,]) * qnorm(0.975)
#     ci_lower <- mean_estimate + (new.gamsig.out$E.sigma[1,]) * qnorm(0.025)
#     time_points <- seq_along(mean_estimate)
#     plot(time_points, mean_estimate, type = "l", col = 'darkblue', ylim = range(theta[, index]), lwd = 1.5,
#          main = paste("State Variable", index, "(Seed:", seed, ")"), ylab = "Estimate", xlab = "Time")
#     polygon(c(time_points, rev(time_points)), c(ci_upper, rev(ci_lower)), 
#             col = adjustcolor("lightblue", alpha.f = 0.5), border = NA)
#     lines(time_points, mean_estimate, col = 'darkblue', lwd = 1.5)
#     points(time_points, theta[, index], col = 'red', pch = 19, cex = 0.5)
#     lines(time_points, theta[, index], col = 'red', lwd = 0.5)
#     }
#     #########################################################################
#     n.samp <- 5000
#     samp.gamma = array(NA_real_, c(J+1, n.samp))
#     samp.sigma = array(NA_real_, c(J+1, n.samp))
#     samp.uts = array(NA_real_, c(J+1, TT, n.samp))
#     samp.sts = array(NA_real_, c(J+1, TT, n.samp))
#     for (j in 1:(J+1) ) { 
#         ########################      
#           sts.dummy <- update_sts(y[j,],
#                                   cur.theta.out$exps[j,], 
#                                   cur.uts.out$E.inv.uts[j,], 
#                                   cur.gamsig.out$E.c2.invb.absgam2.sigma[j,], 
#                                   cur.gamsig.out$E.c.invb.absgam[j,], 
#                                   cur.gamsig.out$E.c.a.invb.absgam[j,])
#           new.sts.out$E.sts[j,] <- sts.dummy$E.sts
#           new.sts.out$E.sts2[j,] <- sts.dummy$E.sts2
#           ########################
#           uts.dummy <- update_uts(y[j,],
#                                   cur.theta.out$exps[j,], 
#                                   cur.theta.out$exps2[j,], 
#                                   new.sts.out$E.sts[j,], 
#                                   new.sts.out$E.sts2[j,], 
#                                   cur.gamsig.out$E.inv.sigma[j,], 
#                                   cur.gamsig.out$E.a2.invb.inv.sigma[j,], 
#                                   cur.gamsig.out$E.invb.inv.sigma[j,], 
#                                   cur.gamsig.out$E.c.invb.absgam[j,], 
#                                   cur.gamsig.out$E.c2.invb.absgam2.sigma[j,]) 
#           new.uts.out$E.uts[j,] <- uts.dummy$E.uts
#           new.uts.out$E.inv.uts[j,] <- uts.dummy$E.inv.uts
#           ########################
#          gamsig.dummy <- update_gamma_sigma(y[j,], TT, 
#                                              PriorGamma[j,],
#                                              PriorSigma[j,],
#                                              cur.gamsig.out$E.gam[j,], 
#                                              cur.gamsig.out$V.gam[j,], 
#                                              cur.gamsig.out$E.sigma[j,], 
#                                              cur.gamsig.out$V.sigma[j,], 
#                                              cur.theta.out$exps[j,], 
#                                              cur.theta.out$exps2[j,], 
#                                              new.sts.out$E.sts[j,], 
#                                              new.sts.out$E.sts2[j,], 
#                                              new.uts.out$E.uts[j,], 
#                                              new.uts.out$E.inv.uts[j,],
#                                              cur.gamsig.out$E.sigma[j,], 
#                                              cur.gamsig.out$E.gam[j,])    
#           new.gamsig.out$E.gam[j,] <- gamsig.dummy$E.gam
#           new.gamsig.out$E.sigma[j,] <- gamsig.dummy$E.sigma
#           new.gamsig.out$E.inv.sigma[j,] <- gamsig.dummy$E.inv.sigma
#           new.gamsig.out$E.c2.invb.absgam2.sigma[j,] <- gamsig.dummy$E.c2.invb.absgam2.sigma
#           new.gamsig.out$E.c.invb.absgam[j,] <- gamsig.dummy$E.c.invb.absgam
#           new.gamsig.out$E.c.a.invb.absgam[j,] <- gamsig.dummy$E.c.a.invb.absgam
#           new.gamsig.out$E.a2.invb.inv.sigma[j,] <- gamsig.dummy$E.a2.invb.inv.sigma
#           new.gamsig.out$E.invb.inv.sigma[j,] <- gamsig.dummy$E.invb.inv.sigma
#           new.gamsig.out$E.a.invb.inv.sigma[j,] <- gamsig.dummy$E.a.invb.inv.sigma
#       theta_s <- gamsig.dummy$E.theta[1]
#       theta_g <- gamsig.dummy$E.theta[2]
#       samp.LD <- rmvnorm(n = n.samp, mean = c(theta_s, theta_g), sigma = gamsig.dummy$Hess.LD)
#       samp.gamma[j,] = LL+(-LL+UU)*exp(-exp(samp.LD[,2]))
#       samp.sigma[j,] = exp(samp.LD[,1]) 
#       # Generalized Inverse Gausian Sampling
#       samp.uts[j,,] = t(sample_gig_devroye_vector(n.samp, uts.dummy$uts.lambda, uts.dummy$uts.psi, uts.dummy$uts.chi))
#       # Truncated normal
#       samp.sts[j,,] = t(sample_truncnorm(n.samp, TT, sts.dummy$sts.mu, sts.dummy$sts.sig2) )
#     }   
#     FFF <- (new.gamsig.out$E.c.invb.absgam[,] * new.sts.out$E.sts + new.gamsig.out$E.a.invb.inv.sigma[,]/new.uts.out$E.inv.uts) / new.gamsig.out$E.invb.inv.sigma[,] 
#     QQQ <- 1/(new.gamsig.out$E.invb.inv.sigma[,] * new.uts.out$E.inv.uts)

#     if(J>0){
#     QQQ <- array(apply(QQQ, 2, function(col) diag(col)), dim = c(J+1, J+1, TT))
#     }else{
#       QQQ <- array(QQQ, dim = c(J+1, J+1, TT))
#     }
#     update.theta <- update_theta_cpp(GG, m0, C0, FFF, QQQ, FF, y, ex.df.mat, ex.df.mat.k, Ones, p, J, ppx, TT, k, dM)    
#     FF_t <- aperm(FF, c(2, 1, 3))

#     multiply_matrices <- function(slice_index) {
#       FF_t[,,slice_index] %*% update.theta$sm[,slice_index]
#     }
#     result_list <- lapply(1:ncol(update.theta$sm), multiply_matrices)
#     result_array <- array(unlist(result_list), dim = c(J+1, 1, ncol(update.theta$sm)))
#     result_array <- aperm(result_array, c(1, 3, 2))[,,1]
#     exps <- result_array
#     compute_product_1 <- function(t) {
#       FF_t_slice <- FF_t[,,t]
#       sC_slice <- update.theta$sC[,,t]
#       FF_slice <- FF[,,t]
#       result_slice <- t(FF_slice)%*%sC_slice%*%(FF_slice )
#       return(result_slice)
#     }
#     result_list_1 <- lapply(1:dim(FF)[3], compute_product_1)
#     vars_1 <- simplify2array(result_list_1)
#     if(J>0){
#     vars <- (apply(vars_1, 3, function(x) diag(x)))
#     exps2 = exps^2 + vars
#     }else{
#     exps2 = exps^2 + vars_1
#     vars_1 <- array( vars_1, c(1,TT) )  
#     exps2 <- array( exps2, c(1,TT) )  
#     exps <- array( exps, c(1,TT) )    
#     }
#     new.theta.out  <- update.theta 
#     new.theta.out$exps  <- exps
#     new.theta.out$exps2  <- exps2
#     new.theta.out$vars  <- vars
#     result <- generate_samples(n.samp, TT, p+ppx, J, FF, new.theta.out$sC, new.theta.out$sm, samp.sigma, p0, samp.gamma, samp.sts)
#     #########################################################################
#     palette <- c("#4DB6AC", "#FF7043", "#8E44AD")
#     hist_samp_sigma <- hist(
#       samp.sigma, breaks = 100, freq = FALSE, 
#       col = adjustcolor(palette[1], alpha.f = 0.7), border = "white",
#       main = "Distribution of samp.sigma", xlab = "samp.sigma", ylab = "Density"
#     )
#     lines(density(samp.sigma), col = palette[3], lwd = 2)
#     rug(samp.sigma, col = adjustcolor(palette[2], alpha.f = 0.8))
#     abline(v = quantile(samp.sigma, probs = c(0.025, 0.975)), col = palette[2], lwd = 2, lty = 2)
#     hist_samp_gamma <- hist(
#       samp.gamma, breaks = 100, freq = FALSE, 
#       col = adjustcolor(palette[1], alpha.f = 0.7), border = "white",
#       main = "Distribution of samp.gamma", xlab = "samp.gamma", ylab = "Density"
#     )
#     lines(density(samp.gamma), col = palette[3], lwd = 2)
#     rug(samp.gamma, col = adjustcolor(palette[2], alpha.f = 0.8))
#     abline(v = quantile(samp.gamma, probs = c(0.025, 0.975)), col = palette[2], lwd = 2, lty = 2)
#     #########################################################################
#     quantiles <- c(0.025, p0, 0.975)
#     y_sim <- result$samp_post_pred[1,,]
#     y_sim_quantiles <- apply(y_sim, 1, function(x) quantile(x, probs = quantiles))
#     y_sim_quantiles <- t(y_sim_quantiles)
#     theta_sim_quantiles_list <- lapply(1:(p+ppx+J), function(c) {
#       theta_sim <- result$samp_theta[c,,]  # Extract the samples for component c
#       theta_sim_quantiles <- apply(theta_sim, 1, function(x) quantile(x, probs = quantiles))
#       t(theta_sim_quantiles)  # Transpose to get time points as rows and quantiles as columns
#     })
#     theta_sim_quantiles_array <- array(
#       unlist(theta_sim_quantiles_list),
#       dim = c((p+ppx+J), dim(result$samp_theta)[2], length(quantiles))
#     )
#     #########################################################################
#     plot(
#       y[1,], type = "l", col = "black", lwd = 0.5, 
#       xlab = "Time", ylab = "Observation", 
#       main = paste("Prediction (Seed:", seed, ")"),
#       ylim = range(c(y[1,],y_sim_quantiles[,1], y_sim_quantiles[,3])) 
#     )

#     quantiles <- c(0.025, p0, 0.975)

#     y_sim <- result$samp_post_pred[1,,]
#     y_sim_quantiles <- apply(y_sim, 1, function(x) quantile(x, probs = quantiles))
#     y_sim_quantiles <- t(y_sim_quantiles)

#     theta_sim_quantiles_list <- lapply(1:(p+ppx+J), function(c) {
#       theta_sim <- result$samp_theta[c,,]  
#       theta_sim_quantiles <- apply(theta_sim, 1, function(x) quantile(x, probs = quantiles))
#       t(theta_sim_quantiles)  
#     })

#     theta_sim_quantiles_array <- array(
#       unlist(theta_sim_quantiles_list),
#       dim = c((p+ppx+J), dim(result$samp_theta)[2], length(quantiles))
#     ) 

#     points(y[1,], pch = 19, col = "black", cex = 0.5)
#     polygon(
#       x = c(1:nrow(y_sim_quantiles), rev(1:nrow(y_sim_quantiles))), 
#       y = c(y_sim_quantiles[,3], rev(y_sim_quantiles[,1])), 
#       col = adjustcolor("lightblue", alpha.f = 0.5), border = NA
#     )
#     lines(y_sim_quantiles[,2], col = 'darkorange', lwd = 1.5)
#     lines(exps[1,], col = 'pink', lwd = 1.5)
#     lines(y_sim_quantiles[,1], col = 'lightblue', lwd = 1, lty = 2)  # Lower bound
#     lines(y_sim_quantiles[,3], col = 'lightblue', lwd = 1, lty = 2)  # Upper bound
#     #########################################################################
#     y_det <- rowSums(theta[,-4])
#     # plot.ts(y[1,],col='gray', lwd = 0.5)
#     # points(y[1,],col='gray', cex = 0.5)
#     # lines(y_det+observation_sd*qnorm(0.95),col='darkblue')
#     # lines(y_det+observation_sd*qnorm(0.5),col='forestgreen')
#     # lines(y_det+observation_sd*qnorm(0.05),col='darkred')
#     #########################################################################
#     idx <- (TT-100):TT
#     plot(
#       y[1,idx], type = "l", col = "black", lwd = 0.5, 
#       xlab = "Time", ylab = "Observation", 
#       main = paste("Prediction (Seed:", seed, ")"),
#       ylim = range(c(y[1,idx], y_sim_quantiles[idx,1], y_sim_quantiles[idx,3])) 
#     )
#     points(y[1,idx], pch = 19, col = "black", cex = 0.5)
#     polygon(
#       x = c(1:nrow(y_sim_quantiles[idx,]), rev(1:nrow(y_sim_quantiles[idx,]))), 
#       y = c(y_sim_quantiles[idx,3], rev(y_sim_quantiles[idx,1])), 
#       col = adjustcolor("lightblue", alpha.f = 0.5), border = NA
#     )
#     lines(y_sim_quantiles[idx,2], col = 'darkorange', lwd = 1.5)
#     lines(exps[1,idx], col = 'pink', lwd = 1.5)
#     lines(y_sim_quantiles[idx,1], col = 'lightblue', lwd = 1, lty = 2)  # Lower bound
#     lines(y_sim_quantiles[idx,3], col = 'lightblue', lwd = 1, lty = 2)  # Upper bound

#     lines(y_det[idx]+observation_sd*qnorm(0.95),col='darkblue')
#     lines(y_det[idx]+observation_sd*qnorm(0.5),col='forestgreen')
#     lines(y_det[idx]+observation_sd*qnorm(0.05),col='darkred')

#     # samples <- get((paste0("samples_", gsub("\\.", ".", sprintf("%.2f", p0)))))
#     samples <- result
#     exps_sims   <- array( NA_real_, c(n.samp,TT) )
#     for(i in 1:n.samp){
#         update.theta_sim <- samples$samp_theta[,,i] 
#         FF_t <- aperm(FF, c(2, 1, 3))
#         multiply_matrices <- function(slice_index) {
#           FF_t[,,slice_index] %*% update.theta_sim[,slice_index]
#         }
#         result_list   <- lapply(1:ncol(update.theta_sim), multiply_matrices)
#         result_array  <- array(unlist(result_list), dim = c(J+1, 1, ncol(update.theta_sim)))
#         result_array  <- aperm(result_array, c(1, 3, 2))[,,1]
#         exps_sims[i,] <- result_array
#     }
#     quantiles_result <- apply(exps_sims, 2, function(column) quantile(column, probs = c(0.025, 0.975)))

#         suffix <- paste0("_", sprintf("%.2f", p0))
#     assign(paste0("y_sim", suffix), y_sim)
#     assign(paste0("theta_sim_quantiles_list", suffix), theta_sim_quantiles_list)
#     assign(paste0("y_sim_quantiles", suffix), y_sim_quantiles)
#     assign(paste0("theta_sim_quantiles_array", suffix), theta_sim_quantiles_array)
#     assign(paste0("samples", suffix), result)                       
#     assign(paste0("exps_sims", suffix), exps_sims)
#     assign(paste0("quantiles_result", suffix), quantiles_result) 
            
#     save(list = c(
#     paste0("y_sim", suffix),
#     paste0("theta_sim_quantiles_list", suffix),
#     paste0("y_sim_quantiles", suffix),
#     paste0("theta_sim_quantiles_array", suffix),
#     paste0("samples", suffix), 
#     paste0("exps_sims", suffix),
#     paste0("quantiles_result", suffix)     
#     ), 
#     file = paste0("iteration_p0_", gsub("\\.", "", sprintf("%.2f", p0)), ".RData"))

#     cat("\n--- SAVED R OJECTS ---\n")
  
#     # Trigger health check at the start
#     monitor_health()
#     gc()   

# }

In [ ]:
# ls()

In [ ]:
# load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_005.RData')                                                                                                                            
# load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_020.RData')                                                                                                                            
# load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_035.RData')                                                                                                                            
# load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_050.RData')                                        
# load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_065.RData') 
# load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_080.RData')  
# load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_095.RData')  

In [ ]:
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_005.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_010.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_015.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_020.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_025.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_030.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_035.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_040.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_045.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_050.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_055.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_060.RData')                                        
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_065.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_070.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_075.RData') 
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_080.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_085.RData')                                                                                                                            
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_090.RData')  
load('/data/muscat_data/jaguir26/project1_ucsc_phd/iteration_p0_095.RData')  

In [ ]:
# ls()

In [ ]:
y_det <- rowSums(theta[,-4])
idx <- (TT-50):TT

# p0_values <- c(0.05,0.2,0.35,0.5,0.65,0.80, 0.95)
# colors <- c("pink",'gray','gray', "lightgreen",'gray','gray', "lightblue")  # Adjust colors for visual distinction

p0_values <- c(0.05,0.5, 0.95)
colors <- c("pink", "lightgreen", "lightblue")  # Adjust colors for visual distinction


plot(
  y[1,idx], type = "l", col = "darkgray", lwd = 0.5, 
  xlab = "Time", ylab = "Observation", 
  main = "Prediction for Different p0 Values",
  # ylim = range(c(y[1,idx], y_sim_0.05[idx,1], y_sim_0.05[idx,3],
  #                y_sim_0.50[idx,1], y_sim_0.50[idx,3],
  #                y_sim_0.95[idx,1], y_sim_0.95[idx,3]))
  ylim = range(c(y[1,idx]))
)

points(y[1,idx], pch = 19, col = "darkgray", cex = 0.8)

for (i in seq_along(p0_values)) {
  p0 <- p0_values[i]
  suffix <- paste0("_", sprintf("%.2f", p0))
  y_sim_quantiles <- get(paste0("y_sim_quantiles", suffix))
  # quantiles_exps <- get((paste0("quantiles_result_", gsub("\\.", ".", sprintf("%.2f", p0)))))
  exps_sims  <- get((paste0("exps_sims_", gsub("\\.", ".", sprintf("%.2f", p0)))))
  quantiles_exps <- apply(exps_sims, 2, function(column) quantile(column, probs = c(0.025, 0.975)))
  quant_m <- colMeans(exps_sims)
  # polygon(
  #   x = c(1:nrow(y_sim_quantiles[idx,]), rev(1:nrow(y_sim_quantiles[idx,]))), 
  #   y = c(y_sim_quantiles[idx,3], rev(y_sim_quantiles[idx,1])), 
  #   col = adjustcolor(colors[i], alpha.f = 0.3), border = NA
  # )
    
  polygon(
    x = c(1:ncol(quantiles_exps[,idx]), rev(1:ncol(quantiles_exps[,idx]))), 
    y = c(quantiles_exps[2,idx], rev(quantiles_exps[1,idx])), 
    col = adjustcolor(colors[i], alpha.f = 0.3), border = NA
  )
  lines(quant_m[idx], col = "darkorange", lwd = 1, lty = 3)
}

lines(y_det[idx]+observation_sd*qnorm(0.95),col='darkblue',lty = 2, lwd = 1)
# lines(y_det[idx]+observation_sd*qnorm(0.80),col='purple',lty = 2, lwd = 1)
# lines(y_det[idx]+observation_sd*qnorm(0.65),col='purple',lty = 2, lwd = 1)
lines(y_det[idx]+observation_sd*qnorm(0.5),col='forestgreen',lty = 2, lwd = 1)
# lines(y_det[idx]+observation_sd*qnorm(0.35),col='purple',lty = 2, lwd = 1)
# lines(y_det[idx]+observation_sd*qnorm(0.20),col='purple',lty = 2, lwd = 1)
lines(y_det[idx]+observation_sd*qnorm(0.05),col='darkred',lty = 2, lwd = 1)

# legend(
#   "topright", legend = paste0("p0 = ", p0_values), 
#   col = colors, lty = 1, lwd = 1.5, bty = "n", 
#   fill = adjustcolor(colors, alpha.f = 0.3)
# )


In [ ]:
# idx <- (TT-TT+1):TT
idx <- (TT-50):TT

# p0_values <- c(0.05,0.2,0.35,0.5,0.65,0.80, 0.95)
p0_values <- c(0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95)
colors <- rep(c("lightblue"),length(p0_values))  # Adjust colors for visual distinction
colors2 <- rep(c("black"),length(p0_values)) 

# colors <- c("pink",'gray','gray', "lightgreen",'gray','gray', "lightblue")  # Adjust colors for visual distinction
# colors2 <- c("darkred",'purple','purple', "forestgreen",'purple','purple', "darkblue")

# colors <- c("pink", "lightgreen", "lightblue")  
# colors2 <- c("darkred", "forestgreen", "darkblue")



for (i in seq_along(p0_values)) {
  plot(
      y[1,idx], type = "l", col = "darkgray", lwd = 0.5, 
      xlab = "Time", ylab = "Observation", 
      main = paste0("Inference for: ", sprintf("%.2f", p0), 'th quantile.'),
      # ylim = range(c(y[1,idx], y_sim_0.05[idx,1], y_sim_0.05[idx,3],
      #                y_sim_0.50[idx,1], y_sim_0.50[idx,3],
      #                y_sim_0.95[idx,1], y_sim_0.95[idx,3]))
      ylim = range(c(y[1,idx]))
    )

  points(y[1,idx], pch = 19, col = "darkgray", cex = 0.8)

  p0 <- p0_values[i]
  suffix <- paste0("_", sprintf("%.2f", p0))
  y_sim_quantiles <- get(paste0("y_sim_quantiles", suffix))
  # quantiles_exps <- get((paste0("quantiles_result_", gsub("\\.", ".", sprintf("%.2f", p0)))))
  exps_sims  <- get((paste0("exps_sims_", gsub("\\.", ".", sprintf("%.2f", p0)))))
  quantiles_exps <- apply(exps_sims, 2, function(column) quantile(column, probs = c(0.025, 0.975)))
  quant_m <- colMeans(exps_sims)
  # polygon(
  #   x = c(1:nrow(y_sim_quantiles[idx,]), rev(1:nrow(y_sim_quantiles[idx,]))), 
  #   y = c(y_sim_quantiles[idx,3], rev(y_sim_quantiles[idx,1])), 
  #   col = adjustcolor(colors[i], alpha.f = 0.3), border = NA
  # )
    
  polygon(
    x = c(1:ncol(quantiles_exps[,idx]), rev(1:ncol(quantiles_exps[,idx]))), 
    y = c(quantiles_exps[2,idx], rev(quantiles_exps[1,idx])), 
    col = adjustcolor(colors[i], alpha.f = 0.6), border = NA
  )
  lines(quant_m[idx], col = "darkred", lwd = 1.5, lty = 3)
    
  lines(y_det[idx]+observation_sd*qnorm(p0),col=colors2[i],lty = 2, lwd = 1)
  # points(y_det[idx]+observation_sd*qnorm(p0),col=colors2[i])
}



# legend(
#   "topright", legend = paste0("p0 = ", p0_values), 
#   col = colors, lty = 1, lwd = 1.5, bty = "n", 
#   fill = adjustcolor(colors, alpha.f = 0.3)
# )


In [ ]:
names(new.theta.out_0.25)
(new.theta.out_0.25$sm0)
(new.theta.out_0.25$sC0)

In [ ]:
# idx <- (TT-TT+1):TT
idx <- (TT-100):TT

# p0_values <- c(0.05,0.2,0.35,0.5,0.65,0.80, 0.95)
p0_values <- c(0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95)
colors <- rep(c("lightblue"),length(p0_values))  # Adjust colors for visual distinction
colors2 <- rep(c("black"),length(p0_values)) 

for (i in seq_along(p0_values)) {
    p0 <- p0_values[i]
  plot(
      y[1,idx], type = "l", col = "darkgray", lwd = 0.5, 
      xlab = "Time", ylab = "Observation", 
      main = paste0("Prediction under: exAL(", sprintf("%.2f", p0), ')  - Model'),
      # ylim = range(c(y[1,idx]))
        ylim = range(c(y[1,idx], y_sim_0.05[idx,1], y_sim_0.05[idx,3],
                 y_sim_0.50[idx,1], y_sim_0.50[idx,3],
                 y_sim_0.95[idx,1], y_sim_0.95[idx,3]))
    )

  points(y[1,idx], pch = 19, col = "darkgray", cex = 0.8)

  
  suffix <- paste0("_", sprintf("%.2f", p0))
  y_sim_quantiles <- get(paste0("y_sim_quantiles", suffix))
  # quantiles_exps <- get((paste0("quantiles_result_", gsub("\\.", ".", sprintf("%.2f", p0)))))
  exps_sims  <- get((paste0("exps_sims_", gsub("\\.", ".", sprintf("%.2f", p0)))))
  quantiles_exps <- apply(exps_sims, 2, function(column) quantile(column, probs = c(0.025, 0.975)))
  quant_m <- colMeans(exps_sims)
  polygon(
    x = c(1:nrow(y_sim_quantiles[idx,]), rev(1:nrow(y_sim_quantiles[idx,]))), 
    y = c(y_sim_quantiles[idx,3], rev(y_sim_quantiles[idx,1])), 
    col = adjustcolor(colors[i], alpha.f = 0.3), border = NA
  )
    
#   polygon(
#     x = c(1:ncol(quantiles_exps[,idx]), rev(1:ncol(quantiles_exps[,idx]))), 
#     y = c(quantiles_exps[2,idx], rev(quantiles_exps[1,idx])), 
#     col = adjustcolor(colors[i], alpha.f = 0.6), border = NA
#   )

#   lines(quant_m[idx], col = "darkred", lwd = 1.5, lty = 3)
     lines(y_sim_quantiles[idx,2], col = "darkred", lwd = 1.5, lty = 3)
    
  lines(y_det[idx]+observation_sd*qnorm(p0),col=colors2[i],lty = 2, lwd = 1)
  # points(y_det[idx]+observation_sd*qnorm(p0),col=colors2[i])
}



# legend(
#   "topright", legend = paste0("p0 = ", p0_values), 
#   col = colors, lty = 1, lwd = 1.5, bty = "n", 
#   fill = adjustcolor(colors, alpha.f = 0.3)
# )


In [ ]:
# p0 <- p0_values[1]
# samples <- get((paste0("samples_", gsub("\\.", ".", sprintf("%.2f", p0)))))
# dim(samples$samp_post_pred[1,,])
# samples2 <- get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", p0)))))

In [ ]:
# paste0("iteration_p0_0",  100*0.05, ".RData")
# p0_values <- c(0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 
#                0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95)

# file_names <- sprintf("iteration_p0_%03d.RData", as.integer(0.05 * 100))

# print(file_names)
#       file_name <- sprintf("iteration_p0_%03d.RData", as.integer(p0 * 100))
#       paste0("/data/muscat_data/jaguir26/project1_ucsc_phd/", file_name)  
# "iteration_p0_010.RData"

In [ ]:
# dim(samples2)
# dim(samples)
# dim(get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", 0.5))))))

In [ ]:
n.samp <- 10000

In [ ]:
# saveRDS(y_reps_sim, file = "y_reps_sim.rds")
# y_reps_sim <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/y_reps_sim.rds")
# dim(y_reps_sim)

In [ ]:
# p0_values <- c(0.05,0.2,0.35,0.5,0.65,0.80, 0.95)
p0_values <- c(0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95)

# y_reps_sim <- array( NA_real_, c(length(p0_values),n.samp,TT) )
y_reps_sim <- array( NA_real_, c(length(p0_values),TT,n.samp) )

for (i in 1:length(p0_values)) {
    p0 <- p0_values[i]
    # samples <- get((paste0("samples_", gsub("\\.", ".", sprintf("%.2f", p0)))))
    samples <- get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", p0)))))
    y_reps_sim[i,,] <- samples
}
saveRDS(y_reps_sim, file = "y_reps_sim.rds")

In [ ]:
saveRDS(y, file = "y.rds")
y <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/y.rds")

In [ ]:
dim(y)

In [ ]:
# Ensure required libraries are installed and loaded
if (!requireNamespace("GA", quietly = TRUE)) {
  install.packages("GA")
}
if (!requireNamespace("parallel", quietly = TRUE)) {
  install.packages("parallel")
}

library(GA)
library(parallel)

In [ ]:
# compute_crps_generic <- function(w, t) {
#   set.seed(111)
    
#   w        <- w / sum(w)
#   sims_t   <- t(y_reps_sim[,,t])
#   y        <- y[1, t]
#   n        <- nrow(sims_t)
#   selected <- sims_t[cbind(1:n, apply( matrix(runif(n), nrow = n), 1, function(x) which.max(cumsum(w) > x)))]
  
#   crps_quantile_representation <- function(y, sample) {
#     sorted_sample <- sort(sample)
#     tau_values <- seq(1 / length(sample), 1, length.out = length(sample))
#     crps_value <- sum(ifelse(
#                               y >= sorted_sample,
#                               tau_values * (y - sorted_sample), 
#                               (1 - tau_values) * (sorted_sample - y)
#                             ))
#     return(2 * crps_value / length(sample))
#   }
  
#   crps_quantile_representation(y, selected)
# }

# optimize_weights_generic <- function(t) {
#   crps_func <- function(w, t) {
#     compute_crps_generic(w, t)
#   }
  
#   ga_result <- ga(
#     type = "real-valued",
#     fitness = function(w) -crps_func(w, t),  # Minimize loss by negating
#     lower = rep(0, 7),                       # Lower bounds
#     upper = rep(10000, 7),                   # Upper bounds
#     popSize = 40,                            # Population size
#     maxiter = 1000,                          # Maximum iterations
#     run = 100,                               # Convergence tolerance
#     parallel = FALSE                         # Disable GA's internal parallelization
#   )
  
#   best_solution <- ga_result@solution[1, ]
#   optimal_w <- best_solution / sum(best_solution)
#   minimum_loss <- -ga_result@fitnessValue

#   list(
#     Time_Step = t,
#     Optimal_Weights = optimal_w,
#     Minimum_Loss = minimum_loss
#   )
# }

# time_steps <- list(
#   regular = 1:TT
# )

# results_regular <- mclapply(
#   time_steps$regular,
#   optimize_weights_generic,
#   mc.cores = parallel::detectCores() - 1
# )

# all_results <- list(
#   Regular = results_regular
# )

# saveRDS(all_results, "/data/muscat_data/jaguir26/project1_ucsc_phd/SIM_optimization_results.rds")
# write.csv(do.call(rbind, results_regular), "/data/muscat_data/jaguir26/project1_ucsc_phd/SIM_results_regular.csv")
# print("Results saved successfully:")
# print(all_results)



In [ ]:
library(tibble)
library(stringr)
library(purrr)

lines <- readLines("/data/muscat_data/jaguir26/project1_ucsc_phd/SIM_results_regular.csv")
data_rows <- lines[-1]  # Exclude header
parse_row <- function(row) {
  row <- str_remove_all(row, '"')  # Remove quotes
  parts <- strsplit(row, ",", fixed = TRUE)[[1]]
  
  time_step <- as.numeric(parts[2])
  optimal_weights <- paste(parts[3:(length(parts) - 1)], collapse = ",")
  minimum_loss <- as.numeric(parts[length(parts)])
  
  weights <- str_match_all(optimal_weights, "x\\d+ = ([0-9\\.]+)")[[1]][, 2]
  weights <- as.numeric(weights)
  
  tibble(
    Time_Step = time_step,
    Weight_1 = weights[1],
    Weight_2 = weights[2],
    Weight_3 = weights[3],
    Weight_4 = weights[4],
    Weight_5 = weights[5],
    Weight_6 = weights[6],
    Weight_7 = weights[7],
    Weight_8 = weights[8],
    Weight_9 = weights[9],
    Weight_10 = weights[10],
    Weight_11 = weights[11],
    Weight_12 = weights[12],
    Weight_13 = weights[13],
    Weight_14 = weights[14],
    Weight_15 = weights[15],
    Weight_16 = weights[16],
    Weight_17 = weights[17],
    Weight_18 = weights[18],
    Weight_19 = weights[19],
    Minimum_Loss = minimum_loss
  )
}

parsed_data <- map_dfr(data_rows, parse_row)
print(parsed_data)
write_csv(parsed_data, "/data/muscat_data/jaguir26/project1_ucsc_phd/SIM_parsed_results.csv")


In [ ]:
parsed_regular_path <- "/data/muscat_data/jaguir26/project1_ucsc_phd/SIM_parsed_results.csv"

# Read the parsed regular results
parsed_regular <- read_csv(parsed_regular_path,show_col_types = FALSE)
# Normalize weights (to ensure each set of weights sums to 1)
weights_matrix <- as.matrix(parsed_regular[, 2:20])
weights_matrix <- weights_matrix / rowSums(weights_matrix)

# Define the number of samples and time steps
num_samples <- dim(y_reps_sim)[3]
num_time_steps <- dim(y_reps_sim)[2]

# y_reps_sim <- array( NA_real_, c(length(p0_values),TT,n.samp) )

# Efficient sampling using apply across the time dimension
synth_regular <- sapply(1:num_time_steps, function(t) {
  weights <- weights_matrix[t, ]
  apply(y_reps_sim[,t,], 2, function(y_column) {
    sample(y_column, size = 1, prob = weights)
  })
})

synth_regular <- t(synth_regular)


In [ ]:
dim(synth_regular)

In [ ]:
sim_timestamps <- timestamps[(length(timestamps)-TT+1):length(timestamps)]

In [ ]:
# Display the first few lines of each dataset
cat("First few rows of Parsed Regular Results:\n")
print(head(parsed_regular))

parsed_regular <- parsed_regular %>%
  mutate(Date = as.Date(sim_timestamps[Time_Step]))

write_csv(parsed_regular, "/data/muscat_data/jaguir26/project1_ucsc_phd/parsed_results_with_dates.csv")


In [ ]:
# parsed_regular

In [ ]:
# Load necessary libraries
library(ggplot2)
library(dplyr)
library(zoo)
library(patchwork)
library(tidyr)

# Reshape the regular data
weights_long_regular <- parsed_regular %>%
  dplyr::select(Date, starts_with("Weight")) %>%
  pivot_longer(
    cols = starts_with("Weight"),
    names_to = "Weight_Index",
    values_to = "Value"
  )

ggplot(weights_long_regular, aes(x = Date, y = Weight_Index, fill = Value)) +
  geom_tile() +
  scale_fill_gradient(low = "white", high = "darkred", limits = c(0, 1)) +
  labs(
    title = "Heatmap of Weights Over Time (Regular)",
    x = "Date",
    y = " ",
    fill = "Weight Value"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1),
    panel.grid = element_blank(),
    aspect.ratio = 0.2
  )

parsed_regular <- parsed_regular %>%
  arrange(Date) %>%
  mutate(Rolling_Mean = rollmean(Minimum_Loss, k = 7, fill = NA, align = "right"))

regular_loss_plot <- ggplot(parsed_regular, aes(x = Date, y = Minimum_Loss)) +
  geom_line(color = "#2c7bb6") +
  # ylim(0, 10) +
  labs(
    title = "Opt. CRPS ",
    x = "Date",
    y = "Minimum Loss"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 30, hjust = 1),
    plot.title = element_text(hjust = 0.5)
  )

regular_rolling_plot <- ggplot(parsed_regular, aes(x = Date, y = Rolling_Mean)) +
  geom_line(color = "#d7191c") +
  # ylim(0, 10) +
  labs(
    title = "7-Day Mean Opt. CRPS ",
    x = "Date",
    y = "Rolling Mean"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 30, hjust = 1),
    plot.title = element_text(hjust = 0.5)
  )


# filename <- "CRPS_opt.png"
# png(filename = paste0(output_dir, filename), width = 6000, height = 4000, res = 600)
# Combine plots using patchwork
(regular_loss_plot ) / (regular_rolling_plot )
# dev.off()

In [ ]:
# parsed_regular

- Compare with true components 
- Compare with true quantilea
- Check selection of transfer

In [ ]:
# ls()

In [ ]:
# # Ensure required libraries are installed and loaded
# if (!requireNamespace("GA", quietly = TRUE)) {
#   install.packages("GA")
# }
# if (!requireNamespace("parallel", quietly = TRUE)) {
#   install.packages("parallel")
# }

# library(GA)
# library(parallel)


# y_reps_sim <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/y_reps_sim.rds")
# y <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/y.rds")
# TT <- dim(y)[2]

# dim(y)
# dim(y_reps_sim)

# compute_crps_generic <- function(w, t) {
#   set.seed(111)
#   w        <- w / sum(w)
#   sims_t   <- t(y_reps_sim[,,t])
#   y        <- y[1, t]
#   n        <- nrow(sims_t)
#   selected <- sims_t[cbind(1:n, apply( matrix(runif(n), nrow = n), 1, function(x) which.max(cumsum(w) > x)))]
  
#   crps_quantile_representation <- function(y, sample) {
#     sorted_sample <- library(GA)
# library(parallel)


# y_reps_sim <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/y_reps_sim.rds")
# y <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/y.rds")
# TT <- dim(y)[2]

# dim(y)
# dim(y_reps_sim)

# compute_crps_generic <- function(w, t) {
#   set.seed(111)
#   w        <- w / sum(w)
#   sims_t   <- t(y_reps_sim[,,t])
#   y        <- y[1, t]
#   n        <- nrow(sims_t)
#   selected <- sims_t[cbind(1:n, apply( matrix(runif(n), nrow = n), 1, function(x) which.max(cumsum(w) > x)))]
  
#   crps_quantile_representation <- function(y, sample) {
#     sorted_sample <- 7uyyyyyyyyyyyyyyyyyy5tsort(sample)
#     tau_values <- seq(1 / length(sample), 1, length.out = length(sample))
#     crps_value <- sum(ifelse(
#                               y >= sorted_sample,
#                               tau_values * (y - sorted_sample), 
#                               (1 - tau_values) * (sorted_sample - y)
#                             ))
#     return(2 * crps_value / length(sample))
#   }
  
#   crps_quantile_representation(y, selected)
# }

# # optimize_weights_generic <- function(t) {
# #   crps_func <- function(w, t) {
# #     compute_crps_generic(w, t)
# #   }
  
# #   ga_result <- ga(
# #     type = "real-valued",
# #     fitness = function(w) -crps_func(w, t),  # Minimize loss by negating
# #     lower = rep(0, 7),                       # Lower bounds
# #     upper = rep(10000, 7),                   # Upper bounds
# #     popSize = 100,                            # Population size
# #     maxiter = 1000,                          # Maximum iterations
# #     run = 500,                               # Convergence tolerance
# #     parallel = FALSE                         # Disable GA's internal parallelization
# #   )
  
# #   best_solution <- ga_result@solution[1, ]
# #   optimal_w <- best_solution / sum(best_solution)
# #   minimum_loss <- -ga_result@fitnessValue

# #   list(
# #     Time_Step = t,
# #     Optimal_Weights = optimal_w,
# #     Minimum_Loss = minimum_loss
# #   )
# # }

# # time_steps <- list(
# #   regular = 1:TT
# # )

# # results_regular <- mclapply(
# #   time_steps$regular,
# #   optimize_weights_generic,
# #   mc.cores = parallel::detectCores() - 1
# # )

# # all_results <- list(
# #   Regular = results_regular
# # )

# # saveRDS(all_results, "/data/muscat_data/jaguir26/project1_ucsc_phd/SIM_optimization_results.rds")
# # write.csv(do.call(rbind, results_regular), "/data/muscat_data/jaguir26/project1_ucsc_phd/SIM_results_regular.csv")
# # print("Results saved successfully:")
# # print(all_results)



In [ ]:
# compute_crps_generic(rep(1,7),66)

In [ ]:
# dim(y_reps_sim)
# # p0_values <- c(0.05,0.2,0.35,0.5,0.65,0.80, 0.95)
# p0_values <- c(0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95)
# w <- rep(0.5, length(p0_values)-1)

## Alternative Synth

In [ ]:
# # set.seed(123) # For reproducibility

# # generate_samples <- function(p0_values, w, y_reps_sim) {
# #   n_quantiles <- length(p0_values)          # Number of quantiles
# #   sample_size <- dim(y_reps_sim)[2]         # Sample size
# #   n_timestamps <- dim(y_reps_sim)[3]        # Number of timestamps
  
# #   midpoints <- (p0_values[-1] + p0_values[-length(p0_values)]) / 2
# #   u_samples <- matrix(runif(sample_size * n_timestamps), nrow = sample_size, ncol = n_timestamps)
# #   intervals <- apply(u_samples, 2, function(u) cut(u, breaks = c(0, midpoints, 1), labels = FALSE, include.lowest = TRUE))
# #   output_array <- t(sapply(seq_len(n_timestamps), function(t) {
# #     sapply(seq_len(sample_size), function(i) {
# #       quantile_level <- intervals[i, t]  # Find the quantile level for this sample
# #       u <- u_samples[i, t]  # Uniform value for this sample
# #       samples        <- get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", p0_values[quantile_level])))))
# #       sample_values  <- samples[t, ]
# #       # sample_values <- y_reps_sim[quantile_level, , t]
# #       target_value <- quantile(sample_values, u)
# #       # closest_value <- sample_values[which.min(abs(sample_values - target_value))]
# #       closest_value <- target_value
# #       return(closest_value)
# #     })
# #   })) 
# #   return(output_array)
# # }

In [ ]:
# # result <- generate_samples(p0_values, w, y_reps_sim)
# # print(dim(result))  

# library(parallel)

# set.seed(123) # For reproducibility

# generate_samples_parallel <- function(p0_values, w, y_reps_sim) {
#   n_quantiles <- length(p0_values)          # Number of quantiles
#   sample_size <- dim(y_reps_sim)[2]         # Sample size
#   n_timestamps <- dim(y_reps_sim)[3]        # Number of timestamps
  
#   midpoints <- (p0_values[-1] + p0_values[-length(p0_values)]) / 2
#   u_samples <- matrix(runif(sample_size * n_timestamps), nrow = sample_size, ncol = n_timestamps)
#   intervals <- apply(u_samples, 2, function(u) cut(u, breaks = c(0, midpoints, 1), labels = FALSE, include.lowest = TRUE))
  
#   # Number of cores for parallelization
#   num_cores <- detectCores() - 1
#   cl <- makeCluster(num_cores)
  
#   # Explicitly list and export all 19 `y_sim_*` matrices
#   clusterExport(cl, varlist = c(
#     "y_sim_0.05", "y_sim_0.10", "y_sim_0.15", "y_sim_0.20",
#     "y_sim_0.25", "y_sim_0.30", "y_sim_0.35", "y_sim_0.40",
#     "y_sim_0.45", "y_sim_0.50", "y_sim_0.55", "y_sim_0.60",
#     "y_sim_0.65", "y_sim_0.70", "y_sim_0.75", "y_sim_0.80",
#     "y_sim_0.85", "y_sim_0.90", "y_sim_0.95"
#   ), envir = environment())
  
#   # Export other required variables
#   clusterExport(cl, varlist = c("p0_values", "u_samples", "intervals"), envir = environment())
#   clusterEvalQ(cl, {
#     library(stats) # For quantile function
#   })
  
#   # Define the function to process each timestamp in parallel
#   process_timestamp <- function(t) {
#     sapply(seq_len(sample_size), function(i) {
#       quantile_level <- intervals[i, t]  # Find the quantile level for this sample
#       u <- u_samples[i, t]  # Uniform value for this sample
#       samples <- get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", p0_values[quantile_level])))))
#       sample_values <- samples[t, ]
#       target_value <- quantile(sample_values, u)
#       closest_value <- target_value # Find the value closest to the target quantile
#       return(closest_value)
#     })
#   }
  
#   # Parallelize over timestamps
#   output_list <- parLapply(cl, seq_len(n_timestamps), process_timestamp)
  
#   # Stop the cluster
#   stopCluster(cl)
  
#   # Combine results into an array
#   output_array <- do.call(cbind, output_list)
#   return(output_array)
# }

# # Example usage
# # Ensure these matrices are defined in your environment before running
# # Example: y_sim_0.05 <- array(rnorm(5000 * 1200), c(5000, 1200))

# result <- generate_samples_parallel(p0_values, w, y_reps_sim)
# print(dim(result))  # Should print 5000 x 1200

In [ ]:
# p0_values <- c(0.05,0.2,0.35,0.5,0.65,0.80, 0.95)
p0_values <- c(0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95)
y_reps_sim_synth <- array( NA_real_, c(length(p0_values),TT,n.samp) )
for (i in 1:length(p0_values)) {
    p0 <- p0_values[i]
    # samples <- get((paste0("samples_", gsub("\\.", ".", sprintf("%.2f", p0)))))
    samples <- get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", p0)))))
    y_reps_sim_synth[i,,] <- samples
}

In [ ]:
# dim(get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", 0.5))))))
# dim(y_reps_sim_synth)

In [ ]:
# n_quantiles <- length(p0_values)          # Number of quantiles
# sample_size <- dim(y_reps_sim_synth)[2]         # Sample size
# n_timestamps <- dim(y_reps_sim_synth)[3]        # Number of timestamps

# u_samples <- matrix(runif(sample_size * n_timestamps), nrow = sample_size, ncol = n_timestamps)

# t <- 666

# output_vec <- numeric(sample_size)  
# i <- 6666
  
# u <- u_samples[i, t] 
# u <- 0.099999
# print(u)

# # Case 1: u < tau_1
# if (u < p0_values[1]) {
# sample_values <- y_reps_sim_synth[1, , t]
# output_vec[i] <- quantile(sample_values, u)
# output_vec[i]
# }

# # Case 2: u > tau_K
# if (u > p0_values[n_quantiles]) {
# sample_values <- y_reps_sim_synth[n_quantiles, , t]
# output_vec[i] <- quantile(sample_values, u)
# output_vec[i]
# }

In [ ]:
# k <- max(which(p0_values <= u))  

# sample_values_k <- y_reps_sim_synth[k, , t]
# sample_values_k1 <- y_reps_sim_synth[k + 1, , t]

# q_k <- quantile(sample_values_k, u)
# q_k1 <- quantile(sample_values_k1, u)

# w <- (u - p0_values[k]) / (p0_values[k + 1] - p0_values[k])
# output_vec[i] <- sample(c(q_k,q_k1),1, prob = c(1-w,w))

# # output_vec[i] <- q_k + w * (q_k1 - q_k)

# # k
# # q_k1
# # q_k
# # w
# # output_vec[i] 

In [ ]:
library(parallel)

set.seed(123) 

generate_samples_parallel <- function(p0_values, y_reps_sim_synth) {
  n_quantiles <- length(p0_values)          
  sample_size <- dim(y_reps_sim_synth)[3]         
  n_timestamps <- dim(y_reps_sim_synth)[2]        
  u_samples <- matrix(runif(sample_size * n_timestamps), nrow = sample_size, ncol = n_timestamps)
  num_cores <- detectCores() - 1
  cl <- makeCluster(num_cores)
  clusterExport(cl, varlist = c("p0_values", "y_reps_sim_synth", "u_samples"), envir = environment())
  clusterEvalQ(cl, {
    library(stats) 
 })
  
  process_timestamp <- function(t) {
    output_vec <- numeric(sample_size)  
    for (i in seq_len(sample_size)) {
      u <- u_samples[i, t]  
        
      # Case 1: u < tau_1
      if (u < p0_values[1]) {
        sample_values <- y_reps_sim_synth[1, t, ]
        output_vec[i] <- quantile(sample_values, u)
        next
      }
      
      # Case 2: u > tau_K`
      if (u > p0_values[n_quantiles]) {
        sample_values <- y_reps_sim_synth[n_quantiles, t, ]
        output_vec[i] <- quantile(sample_values, u)
        next
      }
      
      k <- max(which(p0_values <= u))  
      
      sample_values_k <- y_reps_sim_synth[k, t, ]
      sample_values_k1 <- y_reps_sim_synth[k + 1, t, ]
      
      q_k <- quantile(sample_values_k, u)
      q_k1 <- quantile(sample_values_k1, u)
      
      w <- (u - p0_values[k]) / (p0_values[k + 1] - p0_values[k])
      # output_vec[i] <- q_k + w * (q_k1 - q_k)
      output_vec[i] <- sample(c(q_k,q_k1),1, prob = c(1-w,w))
    }
    
    return(output_vec)
  }
  
  output_list <- parLapply(cl, seq_len(n_timestamps), process_timestamp)
  
  stopCluster(cl)
  
  output_array <- do.call(cbind, output_list)
  return(output_array)
}

p0_values <- c(0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 
               0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95)
# y_reps_sim_synth <- array(rnorm(19 * 5000 * 1200), c(19, 5000, 1200)) # Example array
result <- generate_samples_parallel(p0_values, y_reps_sim_synth)
print(dim(result))  


In [ ]:
# dim(y_reps_sim_synth)

In [ ]:
# n_quantiles  <- length(p0_values)          
# sample_size  <- dim(y_reps_sim)[2]         
# n_timestamps <- dim(y_reps_sim)[3]        

# midpoints <- (p0_values[-1] + p0_values[-length(p0_values)]) / 2
# print(midpoints)
# u_samples <- matrix(runif(sample_size * n_timestamps), nrow = sample_size, ncol = n_timestamps)
# head(u_samples)
# intervals <- apply(u_samples, 2, function(u) cut(u, breaks = c(0, midpoints, 1), labels = FALSE, include.lowest = TRUE))
# (intervals)                   

In [ ]:
# midpoints
# dim(y_reps_sim)

# s <- 4111
# t <- 1000
# # u <- u_samples[s, t]
# # u
# # quantile_level <- intervals[s, t] 
# # quantile_level

# u              <-  0.9
# quantile_level <- cut(u, breaks = c(0, midpoints, 1), labels = FALSE, include.lowest = TRUE)
# quantile_level
# samples        <- get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", p0_values[quantile_level])))))
# hist(samples, breaks=100)
# abline(v=quantile(samples[t, ], u), col = 'red')
# abline(v=(y_det[t]+observation_sd*qnorm(u) ), col = 'blue')


In [ ]:
# set.seed(123) # For reproducibility

# # s <- 4111
# # u
# # quantile_level <- intervals[s, t] 
# # u <- u_samples[s, t]

# test <- array(NA_real_, dim = c(n.samp) )

# t <- 1150
# for(s in 1:n.samp){
# u              <- runif(1)
# quantile_level <- cut(u, breaks = c(0, midpoints, 1), labels = FALSE, include.lowest = TRUE)
# samples        <- get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", p0_values[quantile_level])))))
# sample_values  <- samples[t, ]
# target_value   <- quantile(sample_values, u)
# test[s] <- target_value
# }
# hist(test, breaks=100)

# qs <-  c(quantiles_result_0.05[,t],
#         quantiles_result_0.10[,t],
#         quantiles_result_0.15[,t],
#         quantiles_result_0.20[,t],
#         quantiles_result_0.25[,t],
#         quantiles_result_0.30[,t],
#         quantiles_result_0.35[,t],
#         quantiles_result_0.40[,t],
#         quantiles_result_0.45[,t],
#         quantiles_result_0.50[,t],
#         quantiles_result_0.55[,t],
#         quantiles_result_0.60[,t],
#         quantiles_result_0.65[,t],
#         quantiles_result_0.70[,t],
#         quantiles_result_0.75[,t],
#         quantiles_result_0.80[,t],
#         quantiles_result_0.85[,t],
#         quantiles_result_0.90[,t],
#         quantiles_result_0.95[,t])

# abline(v=c(qs), col = rep('orange',2*length(p0_values) ) )

# abline(v=y[t], col = 'red', lwd = 2)
# abline(v=as.matrix(quantile(test, c(0.025,0.5,0.975))), col = rep('blue',3) , lwd = 2)

In [ ]:
# dim(result)

In [ ]:
# saveRDS(result, file = "alt_synth.rds")
result <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/alt_synth.rds")

In [ ]:
par(mfrow = c(1, 1))

# idx <- (TT-TT+1):TT
idx <- (TT-1200+1):(TT-0)

plot.ts(y[1,idx], col = 'gray', ylim =c(20,50),
       ylab = "",
       xlab = "",       
       main = "Synthesis II, via Convex Combination")
points(y[1,idx], col = 'gray', cex = 0.5)
quantiles <- c(0.025,0.975)
synth_q <- (apply(result, 2, function(x) quantile(x, probs = quantiles)))
lines(synth_q[1,idx], col = 'lightblue', lwd=1.5)
lines(synth_q[2,idx], col = 'lightblue', lwd=1.5)
          
polygon(x = c(1:ncol(synth_q[,idx]), rev(1:ncol(synth_q[,idx]))), 
        y = c(synth_q[2,idx], rev(synth_q[1,idx])), 
        col = adjustcolor('lightblue', alpha.f = 0.3), border = NA)

quantiles <-  p0_values
synth_q <- (apply(result, 2, function(x) quantile(x, probs = quantiles)))
for (i in 1:length(p0_values)) {
    p0 <- p0_values[i]
    lines(synth_q[i,idx], col = 'darkblue', lwd=0.5)
}
                  
lines(colMeans(result[,idx]), col = 'darkorange', lwd=1)

plot.ts(y[1,idx], col = 'gray', ylim =c(20,50),
       ylab = "",
       xlab = "",       
       main = "Truth")
points(y[1,idx], col = 'gray', cex = 0.5)
quantiles <- c(0.025,0.975)
synth_q <- (apply(result, 2, function(x) quantile(x, probs = quantiles)))
lines((y_det[idx]+observation_sd*qnorm(quantiles[2])), col = 'darkred', lwd=1.5)
lines((y_det[idx]+observation_sd*qnorm(quantiles[1])), col = 'darkred', lwd=1.5)
          
polygon(x = c(1:length(idx), rev(1:length(idx)) ), 
        y = c( (y_det[idx]+observation_sd*qnorm(quantiles[2])),  rev(y_det[idx]+observation_sd*qnorm(quantiles[1])) ), 
        col = adjustcolor('pink', alpha.f = 0.3), border = NA)

                   
                  
for (i in 1:length(p0_values)) {
    p0 <- p0_values[i]
    lines(y_det[idx]+observation_sd*qnorm(p0),col='red',lty = 2, lwd = 0.5)
}

par(mfrow = c(1, 1))

In [ ]:
result <- t(synth_regular) 

In [ ]:
par(mfrow = c(1, 1))

# idx <- (TT-TT+1):TT
idx <- (TT-1200+1):(TT-0)

plot.ts(y[1,idx], col = 'gray', ylim =c(20,50),
       ylab = "",
       xlab = "",       
       main = "Synthesis I, via CRPS Mixture")
points(y[1,idx], col = 'gray', cex = 0.5)
quantiles <- c(0.025,0.975)
synth_q <- (apply(result, 2, function(x) quantile(x, probs = quantiles)))
lines(synth_q[1,idx], col = 'lightblue', lwd=1.5)
lines(synth_q[2,idx], col = 'lightblue', lwd=1.5)
          
polygon(x = c(1:ncol(synth_q[,idx]), rev(1:ncol(synth_q[,idx]))), 
        y = c(synth_q[2,idx], rev(synth_q[1,idx])), 
        col = adjustcolor('lightblue', alpha.f = 0.3), border = NA)

quantiles <-  p0_values
synth_q <- (apply(result, 2, function(x) quantile(x, probs = quantiles)))
for (i in 1:length(p0_values)) {
    p0 <- p0_values[i]
    lines(synth_q[i,idx], col = 'darkblue', lwd=0.5)
}
                  
lines(colMeans(result[,idx]), col = 'darkorange', lwd=1)


In [ ]:
library(ggplot2)

# Define plot parameters
idx <- (TT-50+1):(TT-0)  # Time index
quantiles <- c(0.025, 0.975)  # Confidence intervals
synth_q <- apply(result, 2, function(x) quantile(x, probs = quantiles))
p0_quantiles <- apply(result, 2, function(x) quantile(x, probs = p0_values))

# --- Plot 1: Synthesis via Convex Combination ---
png("synthesis_convex_combination3.png", width = 2000, height = 1400, res = 300)

plot.ts(y[1, idx], col = 'black', ylim = c(20, 50),
        ylab = "Value",
        xlab = "Time",
        main = "Synthesis via Convex Combination", 
        lwd = 1.2)

points(y[1, idx], col = 'gray', cex = 0.5)

# Add confidence interval bands
polygon(x = c(1:length(idx), rev(1:length(idx))), 
        y = c(synth_q[2, idx], rev(synth_q[1, idx])), 
        col = adjustcolor('lightblue', alpha.f = 0.3), border = NA)

# Add 2.5% and 97.5% quantiles
lines(synth_q[1, idx], col = 'blue', lwd = 1.5)
lines(synth_q[2, idx], col = 'blue', lwd = 1.5)

# Add quantiles from p0_values
for (i in 1:length(p0_values)) {
    lines(p0_quantiles[i, idx], col = 'darkblue', lwd = 0.5, lty = 2)
}

# Add mean
lines(colMeans(result[, idx]), col = 'darkorange', lwd = 1.5)

dev.off()

# --- Plot 2: Truth ---
png("truth_plot3.png", width = 2000, height = 1400, res = 300)

plot.ts(y[1, idx], col = 'gray', ylim = c(20, 50),
        ylab = "Value",
        xlab = "Time",
        main = "Truth", 
        lwd = 1.2)

points(y[1, idx], col = 'gray', cex = 0.5)

# Add confidence interval bands
polygon(x = c(1:length(idx), rev(1:length(idx))), 
        y = c(y_det[idx] + observation_sd * qnorm(quantiles[2]),  
              rev(y_det[idx] + observation_sd * qnorm(quantiles[1]))), 
        col = adjustcolor('pink', alpha.f = 0.3), border = NA)

# Add 2.5% and 97.5% quantiles
lines(y_det[idx] + observation_sd * qnorm(quantiles[2]), col = 'darkred', lwd = 1.5)
lines(y_det[idx] + observation_sd * qnorm(quantiles[1]), col = 'darkred', lwd = 1.5)

# Add target quantiles
for (i in 1:length(p0_values)) {
    lines(y_det[idx] + observation_sd * qnorm(p0_values[i]), col = 'red', lty = 2, lwd = 0.5)
}

dev.off()


In [ ]:
dim(result)

In [ ]:
# y_reps_sim <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/y_reps_sim.rds")
# dim(y_reps_sim)
y_reps_sim_synth <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/y_reps_sim_synth.rds")

dim(y_reps_sim_synth)
# saveRDS(y_reps_sim_synth, file = "y_reps_sim_synth.rds")



In [ ]:
# set.seed(123) # For reproducibility

# n_quantiles <- length(p0_values)          # Number of quantiles
# sample_size <- dim(y_reps_sim_synth)[2]         # Sample size
# n_timestamps <- dim(y_reps_sim_synth)[3]        # Number of timestamps

# u_samples <- matrix(runif(sample_size * n_timestamps), nrow = sample_size, ncol = n_timestamps)

# t <- 1200
# output_vec <- numeric(sample_size)  # Initialize result for this timestamp

# i <- 666
# # u <- u_samples[i, t]  # Uniform value for this sample
# u <- 0.9501

# if (u < p0_values[1]) {
# sample_values <- y_reps_sim_synth[1, , t]
# output_vec[i] <- quantile(sample_values, u)
# # next
# }

# if (u > p0_values[n_quantiles]) {
# sample_values <- y_reps_sim_synth[n_quantiles, , t]
# output_vec[i] <- quantile(sample_values, u)
#     output_vec[i]
# # next
# }

# k <- max(which(p0_values <= u))  # Find the interval index

# sample_values_k <- y_reps_sim_synth[k, , t]
# sample_values_k1 <- y_reps_sim_synth[k + 1, , t]

# q_k <- quantile(sample_values_k, u)
# q_k1 <- quantile(sample_values_k1, u)

# w <- (u - p0_values[k]) / (p0_values[k + 1] - p0_values[k])
# output_vec[i] <- sample(c(q_k,q_k1),1, prob = c(1-w,w))

# output_vec[i]
# u
# w
# q_k
# q_k1


In [ ]:
# dim(y_reps_sim_synth)

In [ ]:
# y_sim_p0 <- get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", 0.95)))))
# dim(y_sim_p0)
# dim(result)
# dim(y_reps_sim_synth)
# quantile(y_sim_p0[1200,],0.95)
# quantile(result[,1200],0.95)
# quantile(y_reps_sim_synth[n_quantiles, , 1200],0.95)


In [ ]:
# quantile(y_sim_p0[1200,],0.95)

In [ ]:
# quantile(y_sim_p0[1200,],0.95)
# dim(y_sim_p0)
# quantile(y_reps_sim_synth[n_quantiles, , 1200],0.95)
# dim(y_reps_sim_synth)

In [ ]:
 # p0_values[n_quantiles]

In [ ]:
# samples <- ( get((paste0("y_sim_", gsub("\\.", ".", sprintf("%.2f", 0.95))))) )

In [ ]:
# dim(samples)
# dim(y_reps_sim_synth[length(p0_values),,])

In [ ]:
# y_reps_sim <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/y_reps_sim.rds")
# y <- readRDS("/data/muscat_data/jaguir26/project1_ucsc_phd/y.rds")

# TT <- dim(y)[2]
# compute_crps_generic <- function(w, t) {
#   set.seed(111)
#   w        <- w / sum(w)
#   sims_t   <- t(y_reps_sim[,t,])
#   y        <- y[1, t]
#   n        <- nrow(sims_t)
#   selected <- sims_t[cbind(1:n, apply( matrix(runif(n), nrow = n), 1, function(x) which.max(cumsum(w) > x)))]
                                      
#   crps_quantile_representation <- function(y, sample) {
#     sorted_sample <- sort(sample)
#     tau_values <- seq(1 / length(sample), 1, length.out = length(sample))
#     crps_value <- sum(ifelse(
#                               y >= sorted_sample,
#                               tau_values * (y - sorted_sample), 
#                               (1 - tau_values) * (sorted_sample - y)
#                             ))
#     return(2 * crps_value / length(sample))
#   }
#   crps_quantile_representation(y, selected)
# }

In [ ]:
# compute_crps_generic(rep(1, 7), 1200 )